# NEXUS AI v5.7.1 — Klein-Four Compiler and Locked Goal Interface

**Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828**

Complete self-contained, single-initialization-cell notebook. All v2–v5.7 tests remain present; v5.7.1 adds a two-argument compiler-side Klein-four goal tool to prevent the v5.7 Gemma call from inventing `placement_class=any`.

Run the first code cell. The read-only demonstration is second; the optional LM Studio cell is last. The first-vacuum depths used for newly registered witnesses are previously supplied and native-reverified—not calculated from initial boundary.


In [1]:
"""NEXUS local AI / compiler bridge — compiler-side BIND resolver / lazy frame v4.
Dean A. Kulik, QuHarmonics Research Group, ORCID 0009-0003-3128-8828.

Python 3.10+, standard library only. Runs as a script or in one Jupyter cell.
LM Studio: http://localhost:1235/v1 (override with LM_STUDIO_URL).

Conceptual state: (w,E,p) with E[r]=D[p XOR r].
Lazy field: E[r]=V[r XOR s], backing dictionary V and internal frame s.
F: x=V[s]^w; V'[s]=V[s]^P(x); s'=s^x; p'=p^x; w'=P(x).
H_h: p'=p^h, w'=w, (V',s')=(V,s).
Both forward and transport perform O(1) native field operations.
F H_h = H_h F, H_h^(-1)=H_h, H_a H_b=H_(a^b).
Old absolute snapshots and v1/v2/v3 hash-linked ledger records remain readable.
BIND derives one-step routes from exact readout constraints without traversing ring addresses.
BIND is read-only and never silently fills a missing target field.
Full ledger before/after snapshots still require O(active cells) serialization.

The local language model requests tools; the compiler computes and verifies operations.
No unrestricted shell or filesystem tool is exposed to the model.
"""
from __future__ import annotations
import hashlib
import json
import os
import random
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.error import URLError, HTTPError

BASE = os.getenv('LM_STUDIO_URL', 'http://localhost:1235/v1').rstrip('/')
if BASE.endswith('/v1'):
    BASE = BASE[:-3]
TOKEN = os.getenv('LM_API_TOKEN', '')
WIDTH, SEED_P, SEED_W = 4, 0, 1
ROOT = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
LEDGER = ROOT / 'NEXUS_local_AI_events.jsonl'  # Reuse & verify the original history.
GENESIS = '0' * 64
BIND_PREDICTION = 'NEXUS BIND v4 compiler-side prediction, 2026-11-02.\nBefore linking the next independent test readout:\n1. For any native width 2..16, any complete valid destination has one F-predecessor given by the exact inverse of F; a forward replay reconstructs the destination bit-for-bit.\n2. For any live state and target WHERE, the unique XOR transport h = p XOR target_where yields the same (w,E) and returns by applying h again.\n3. Given a complete one-step endpoint, the three declared provider contracts H, F, F^-1 are evaluated algebraically; zero providers -> UNBOUND, one -> BOUND, multiple -> AMBIGUOUS. Missing field -> UNLINKED and never assumed empty.\n4. BIND neither changes the live machine nor appends to the event ledger, and old ledger v1, v2, v3 events remain replayable.\nWAY HOME: BOUND-H returns by H_h; BOUND F-predecessor returns by F; BOUND-F returns by F^-1; BOUND-F^-1 returns by F; each is checked on the complete (p,w,D) state.\n'
BIND_PREDICTION_SHA256 = '65a5dae5b1d5543799c1700d8970149a1021c9c7618ab66ee27d58ea9e8bbda0'


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'), ensure_ascii=False)


def chain_hash(previous, payload):
    return hashlib.sha256((previous + '\n' + canonical(payload)).encode('utf-8')).hexdigest()


def partner(x: int, n: int) -> int:
    """Native P: reverse nonpalindromes, complement palindromes."""
    mask = (1 << n) - 1
    if not (2 <= n <= 16 and 0 <= x <= mask):
        raise ValueError('n must be 2..16 and x must fit n bits')
    rev = int(f'{x:0{n}b}'[::-1], 2)
    return rev if rev != x else x ^ mask


class Machine:
    """Native conceptual state (w, E, p); lazy E[r]=V[r XOR s].

    V is the sparse backing field, s the internal frame, p the absolute reader WHERE.
    Both forward/backward and transport touch O(1) native field cells.
    Exporting a complete absolute/relative field costs O(k) for k occupied cells.
    """
    def __init__(self, n=WIDTH, p=SEED_P, w=SEED_W, field=None,
                 *, relative_field=None):
        if not (2 <= n <= 16):
            raise ValueError('n must be 2..16 for this prototype')
        mask = (1 << n) - 1
        if type(p) is not int or type(w) is not int or not (0 <= p <= mask and 0 <= w <= mask):
            raise ValueError('p and w must be n-bit words')
        if field is not None and relative_field is not None:
            raise ValueError('supply absolute field or relative_field, not both')
        # Initial frame is zero: the backing keys begin in reader-relative coordinates.
        if relative_field is not None:
            backing = {int(k): int(v) for k, v in relative_field.items() if int(v)}
        else:
            backing = {int(k) ^ p: int(v) for k, v in (field or {}).items() if int(v)}
        if any(not (0 <= k <= mask and 0 <= v <= mask) for k,v in backing.items()):
            raise ValueError('field keys and values must be n-bit words')
        self.n, self.p, self.w = n, p, w
        self.V, self.s = backing, 0
        self.last_hash = GENESIS

    def copy(self):
        other = Machine(self.n, self.p, self.w, relative_field={})
        other.V, other.s = self.V.copy(), self.s
        other.last_hash = self.last_hash
        return other

    def adopt(self, other):
        self.p, self.w = other.p, other.w
        self.V, self.s = other.V.copy(), other.s
        self.last_hash = other.last_hash

    def snapshot(self):
        """External v2-compatible state (p,w,D), for tool and ledger compatibility."""
        absolute = {self.p ^ self.s ^ k: v for k,v in self.V.items()}
        return {'n': self.n, 'p': self.p, 'w': self.w,
                'field': {str(k): absolute[k] for k in sorted(absolute)}}

    def relative_snapshot(self):
        relative = {k ^ self.s: v for k,v in self.V.items()}
        return {str(k): relative[k] for k in sorted(relative)}

    @staticmethod
    def _set(field, at, value):
        if value:
            field[at] = value
        else:
            field.pop(at, None)

    def forward(self):
        """One-cell native step, O(1) independent of field occupancy."""
        oldp, oldframe = self.p, self.s
        x = self.V.get(oldframe, 0) ^ self.w
        w2 = partner(x, self.n)
        self._set(self.V, oldframe, self.V.get(oldframe, 0) ^ w2)
        self.s, self.p, self.w = oldframe ^ x, oldp ^ x, w2
        return {'direction': 'forward', 'x': x, 'written_at': oldp}

    def backward(self):
        """One-cell exact inverse, independent of the number of occupied cells."""
        x = partner(self.w, self.n)
        oldframe, oldp = self.s ^ x, self.p ^ x
        self._set(self.V, oldframe, self.V.get(oldframe, 0) ^ self.w)
        oldw = x ^ self.V.get(oldframe, 0)
        self.s, self.p, self.w = oldframe, oldp, oldw
        return {'direction': 'reverse', 'x': x, 'restored_at': oldp}

    def transport(self, h: int):
        """O(1) pure placement; both V and s remain literally unchanged."""
        if type(h) is not int or not 0 <= h < (1 << self.n):
            raise ValueError(f'transport h must be an integer 0..{(1 << self.n)-1}')
        prev = self.p
        self.p ^= h
        # Keep exact v2 route metadata to preserve historical event-replay checks.
        return {'direction': 'transport', 'h': h, 'from_word': prev,
                'to_word': self.p, 'relocated_field_cells': len(self.V)}

def verify():
    """Executed deterministic checks: involution, random F/H commuting squares, routes."""
    checks = 0
    for n in range(2, 11):
        for x in range(1 << n):
            assert partner(partner(x, n), n) == x
            assert partner(x, n) != x
            checks += 1
    rng = random.Random(181)
    for n in range(2, 17):
        mask = (1 << n) - 1
        for _ in range(320):
            D = {}
            for __ in range(1 + rng.randrange(12)):
                k = rng.randrange(mask + 1)
                v = rng.randrange(mask + 1)
                if v:
                    D[k] = v
            m = Machine(n, rng.randrange(mask+1), rng.randrange(mask+1), D)
            h, a, b = (rng.randrange(mask+1) for __ in range(3))
            # H is its own inverse; transport group is XOR.
            t = m.copy(); original_V = t.V; original_s = t.s
            t.transport(h)
            assert t.V is original_V and t.s == original_s
            t.transport(h)
            assert t.V is original_V and t.s == original_s
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.transport(a); t.transport(b)
            u = m.copy(); u.transport(a ^ b)
            assert t.snapshot() == u.snapshot()
            # The commuting square F o H == H o F.
            left = m.copy(); left.transport(h); left.forward()
            right = m.copy(); right.forward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('FH=HF', n)
            # Also commute for inverse steps.
            left = m.copy(); left.transport(h); left.backward()
            right = m.copy(); right.backward(); right.transport(h)
            assert left.snapshot() == right.snapshot(), ('F^-1H=HF^-1', n)
            # Roundtrip on arbitrary populated field, not just vacuum seed.
            t = m.copy(); t.forward(); t.backward()
            assert t.snapshot() == m.snapshot()
            t = m.copy(); t.backward(); t.forward()
            assert t.snapshot() == m.snapshot()
            checks += 6
    for n in (4, 8, 16):
        m = Machine(n=n, p=0, w=1)
        initial = m.snapshot()
        for _ in range(1000):
            m.forward()
        for _ in range(1000):
            m.backward()
        assert m.snapshot() == initial
    print(f'RUNS: {checks:,} deterministic conditions; P, arbitrary-field roundtrip, '
          'transport involution/group law, F H = H F, inverse commutation, '
          'and 1000-step return at n=4,8,16.')


def apply_mutation(machine, kind, h=None):
    if kind == 'step':
        return machine.forward()
    if kind == 'reverse':
        return machine.backward()
    if kind == 'transport':
        return machine.transport(h)
    raise ValueError('unknown operation')


def replay():
    m = Machine()
    if not LEDGER.exists():
        return m
    count, legacy = 0, 0
    with LEDGER.open('r', encoding='utf-8') as f:
        for count, line in enumerate(f, 1):
            event = json.loads(line)
            if event['before'] != m.snapshot():
                raise RuntimeError(f'Ledger discontinuity before line {count}')
            kind = event['operation']
            # Supports old unhashed events without rewriting or discarding them.
            if 'event_hash' in event:
                if event.get('prev_hash') != m.last_hash:
                    raise RuntimeError(f'Ledger hash link invalid at line {count}')
                payload = {k: v for k, v in event.items()
                           if k not in ('prev_hash', 'event_hash')}
                digest = chain_hash(m.last_hash, payload)
                if digest != event['event_hash']:
                    raise RuntimeError(f'Ledger event hash invalid at line {count}')
            else:
                legacy += 1
                payload = event
                digest = chain_hash(m.last_hash, payload)
            meta = apply_mutation(m, kind, event.get('route', {}).get('h'))
            if event['after'] != m.snapshot() or (event.get('route') and meta != event['route']):
                raise RuntimeError(f'Ledger event mismatch after line {count}')
            m.last_hash = digest
    print(f'Replayed {count} event(s), including {legacy} legacy event(s). '
          f'Last chain digest: {m.last_hash}')
    return m


def operation(m: Machine, kind: str, h=None):
    if kind == 'state':
        return {'status': 'RUNS', 'state': m.snapshot(), 'relative_field': m.relative_snapshot(), 'ledger_head': m.last_hash}
    if kind not in ('step', 'reverse', 'transport'):
        return {'error': 'Only state, step, reverse, transport are allowed'}
    before = m.snapshot()
    trial = m.copy()
    try:
        meta = apply_mutation(trial, kind, h)
    except ValueError as e:
        return {'error': str(e), 'status': 'OPEN'}
    after = trial.snapshot()
    # Independent compiler return; never ask the model to certify it.
    check = trial.copy()
    if kind == 'step':
        check.backward()
    elif kind == 'reverse':
        check.forward()
    else:
        check.transport(h)
    assert check.snapshot() == before, f'{kind}: way home failed'
    payload = {'version': 3, 'operation': kind, 'before': before,
               'after': after, 'route': meta}
    digest = chain_hash(m.last_hash, payload)
    record = {**payload, 'prev_hash': m.last_hash, 'event_hash': digest}
    # Write before committing the in-memory state, so a disk failure cannot create
    # an unrecorded accepted transition. Existing ledger lines are never rewritten.
    with LEDGER.open('a', encoding='utf-8') as f:
        f.write(canonical(record) + '\n')
        f.flush()
        os.fsync(f.fileno())
    trial.last_hash = digest
    m.adopt(trial)
    return {'status': 'RUNS', **record, 'return_verified': True,
            'relative_field': m.relative_snapshot(),
            'native_field_key_rewrites': 0 if kind == 'transport' else 1}


def commute(m: Machine, h: int):
    """Run two commuting paths on copies; leave the live state and ledger unchanged."""
    if type(h) is not int or not 0 <= h < (1 << m.n):
        return {'status': 'OPEN', 'error': 'invalid n-bit XOR offset'}
    starting = m.snapshot()
    left = m.copy(); left.transport(h); left.forward()
    right = m.copy(); right.forward(); right.transport(h)
    assert left.snapshot() == right.snapshot(), 'commuting square failed'
    lback = left.copy(); lback.backward(); lback.transport(h)
    rback = right.copy(); rback.transport(h); rback.backward()
    assert lback.snapshot() == starting == rback.snapshot(), 'commuting return failed'
    return {'status': 'RUNS', 'identity': 'F(H_h(S)) == H_h(F(S))',
            'h': h, 'starting': starting, 'left': left.snapshot(),
            'right': right.snapshot(), 'both_returns_verified': True,
            'live_machine_unchanged': m.snapshot() == starting,
            'ledger_head': m.last_hash, 'ledger_head_unchanged': True}




def _target_machine(value, n):
    """Turn a COMPLETE external readout into a machine; never invent missing fields.

    Returns (Machine, None) or (None, structured diagnostic).
    """
    if not isinstance(value, dict):
        return None, {'status': 'OPEN', 'binding': 'UNLINKED',
                      'missing_link': 'complete target state (n,p,w,field)'}
    required = {'n', 'p', 'w', 'field'}
    missing = sorted(required - set(value))
    if missing:
        return None, {'status': 'OPEN', 'binding': 'UNLINKED',
                      'missing_fields': missing,
                      'missing_link': 'full target readout; no omitted fields are set to zero'}
    if any(type(value[key]) is not int for key in ('n','p','w')) or value['n'] != n:
        return None, {'status':'OPEN', 'binding':'UNBOUND',
                      'missing_link':'target n,p,w must be integers; width must match the active machine'}
    if not isinstance(value['field'], dict):
        return None, {'status':'OPEN', 'binding':'UNLINKED',
                      'missing_link':'field must be an explicit object, including {} if empty'}
    f = {}
    mask = (1 << n) - 1
    for key, val in value['field'].items():
        if type(val) is not int or type(key) not in (str,int):
            return None, {'status':'OPEN','binding':'UNBOUND',
                          'missing_link':'field entries must have integral address/value'}
        if isinstance(key, str) and (not key.isdecimal() or str(int(key)) != key):
            return None, {'status':'OPEN','binding':'UNBOUND',
                          'missing_link': 'field addresses must be canonical nonnegative decimal integers'}
        k = int(key)
        if not (0 <= k <= mask and 0 <= val <= mask):
            return None, {'status':'OPEN','binding':'UNBOUND',
                          'missing_link':'field address/value exceeds n-bit domain'}
        if val: f[k] = val
    try:
        return Machine(n=n, p=value['p'], w=value['w'], field=f), None
    except ValueError as exc:
        return None, {'status':'OPEN','binding':'UNBOUND','missing_link':str(exc)}


def bind(m: Machine, *, target_where=None, predecessor_of_current=False,
         predecessor_of=None, target_state=None):
    """Derive a route from the supplied HOLE rather than search ring contents.

    Contract alternatives (exactly one):
      target_where: relocate current WHAT to an absolute p using H_(p XOR target).
      predecessor_of_current=True: invert F on the current complete destination.
      predecessor_of: invert F on a supplied COMPLETE destination state.
      target_state: one-step destination; resolve ownership among H, F, F^-1.

    Distinct operators that land at the same target are reported AMBIGUOUS;
    absent or partial target fields are UNLINKED, never defaulted to zero.
    Every BOUND route includes its inverse way home. Read-only.
    """
    initial = m.snapshot(); head = m.last_hash
    goals = sum((target_where is not None, predecessor_of_current is True,
                 predecessor_of is not None, target_state is not None))
    if goals != 1 or type(predecessor_of_current) is not bool:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'supply exactly one: target_where, predecessor_of_current=true, predecessor_of, target_state'}
    proof = {'live_machine_unchanged': True, 'ledger_head_unchanged': True,
             'ledger_head':head}
    if target_where is not None:
        if type(target_where) is not int or not 0 <= target_where < 1 << m.n:
            return {'status':'OPEN','binding':'UNBOUND',
                    'missing_link':f'WHERE must be an n-bit integer 0..{(1<<m.n)-1}', **proof}
        h = m.p ^ target_where             # direct address: no candidate enumeration
        dest = m.copy(); dest.transport(h)
        home = dest.copy(); home.transport(h)
        assert home.snapshot() == initial and dest.w == m.w
        assert dest.relative_snapshot() == m.relative_snapshot()
        result = {'status':'RUNS','binding':'BOUND','relation':'same WHAT at required WHERE',
                  'source':initial,'destination':dest.snapshot(),
                  'route':{'operation':'transport','h':h,'cost':'O(1) native field operations'},
                  'way_home':{'operation':'transport','h':h},
                  'return_verified': True, **proof}
    elif predecessor_of_current or predecessor_of is not None:
        if predecessor_of_current:
            destination = m.copy()
        else:
            destination, error = _target_machine(predecessor_of, m.n)
            if error:
                return {**error, **proof}
        source = destination.copy(); inverse = source.backward()
        check = source.copy(); check.forward()
        assert check.snapshot() == destination.snapshot()
        result = {'status':'RUNS','binding':'BOUND','relation':'unique F-predecessor of complete destination',
                  'source':source.snapshot(),'destination':destination.snapshot(),
                  'route':{'operation':'reverse','x':inverse['x'],
                           'readout':'x=P_n(w_destination)', 'cost':'O(1) native field operations'},
                  'way_home':{'operation':'step'},'return_verified':True,**proof}
    else:
        destination, error = _target_machine(target_state, m.n)
        if error:
            return {**error,**proof}
        wanted = destination.snapshot()
        providers = []
        # Exactly three declared providers; no generated candidate words or paths.
        # H: the constraint is equality of relative computation (w,E).
        if m.w == destination.w and m.relative_snapshot() == destination.relative_snapshot():
            h = m.p ^ destination.p
            providers.append({'operation':'transport','h':h,
                              'way_home':{'operation':'transport','h':h}})
        # F: derive the required predecessor FROM the destination and compare.
        preceding = destination.copy(); preceding.backward()
        if preceding.snapshot() == initial:
            providers.append({'operation':'step','way_home':{'operation':'reverse'}})
        # F^-1: derive the required successor FROM the destination and compare.
        following = destination.copy(); following.forward()
        if following.snapshot() == initial:
            providers.append({'operation':'reverse','way_home':{'operation':'step'}})
        if len(providers) == 1:
            p = providers[0]
            result = {'status':'RUNS','binding':'BOUND',
                      'relation':'one-step provider selected by exact endpoint constraints',
                      'source':initial,'destination':wanted,'route':p,
                      'way_home':p['way_home'],'return_verified':True,**proof}
        elif len(providers) > 1:
            result = {'status':'OPEN','binding':'AMBIGUOUS',
                      'source':initial,'destination':wanted,'providers':providers,
                      'missing_link':'operator identity; an independent readout must select the provider',
                      **proof}
        else:
            result = {'status':'OPEN','binding':'UNBOUND',
                      'source':initial,'destination':wanted,'providers':[],
                      'missing_link':'no one-step H, F, or F^-1 provider owns this endpoint; a composition or another lawful operator is not yet supplied',
                      **proof}
    assert m.snapshot() == initial and m.last_hash == head, 'BIND mutated live state'
    return result


def verify_bind():
    """Exact finite tests; all are independent of LM Studio and existing user ledger."""
    checks = 0
    rng = random.Random(20261102)
    for n in range(2,17):
        mask = (1<<n)-1
        for _ in range(180):
            D={}
            for __ in range(1+rng.randrange(8)):
                k=rng.randrange(mask+1); v=rng.randrange(mask+1)
                if v: D[k]=v
            m=Machine(n, rng.randrange(mask+1), rng.randrange(mask+1), D)
            initial=m.snapshot(); h=rng.randrange(mask+1)
            a=bind(m,target_where=m.p ^ h)
            assert a['binding']=='BOUND' and a['destination']['p']==(m.p^h)
            assert a['route']['h']==h and a['return_verified']
            dest=m.copy(); dest.forward()
            b=bind(m, target_state=dest.snapshot())
            # Endpoint may have two valid one-step providers on degenerate states.
            assert any(p['operation']=='step' for p in
                       ([b['route']] if b['binding']=='BOUND' else b.get('providers',[])))
            c=bind(m,predecessor_of=dest.snapshot())
            assert c['binding']=='BOUND' and c['source']==initial
            d=bind(m,predecessor_of_current=True)
            expected=m.copy(); expected.backward()
            assert d['source']==expected.snapshot()
            assert m.snapshot()==initial
            checks+=4
    missing=bind(Machine(),target_state={'n':4,'p':1,'w':8})
    assert missing['binding']=='UNLINKED' and missing['missing_fields']==['field']
    # With u at both XOR-opposite cells, F and H_u have the same endpoint;
    # the backwards provider also agrees. A genuine multi-owner HOLE.
    probe=Machine(n=4,p=0,w=0,field={0:15,15:15})
    endpoint=probe.copy(); endpoint.forward()
    result=bind(probe,target_state=endpoint.snapshot())
    assert result['binding']=='AMBIGUOUS',result
    assert bind(Machine(),target_where=100)['binding']=='UNBOUND'
    checks+=3
    print(f'RUNS: {checks:,} BIND checks (widths 2..16, populated fields, exact inverse, ambiguous/unlinked cases).')


def request_json(method, endpoint, payload=None, timeout=180):
    headers = {'Content-Type': 'application/json'}
    if TOKEN:
        headers['Authorization'] = f'Bearer {TOKEN}'
    data = None if payload is None else json.dumps(payload).encode('utf-8')
    req = Request(BASE + endpoint, headers=headers, data=data, method=method)
    with urlopen(req, timeout=timeout) as res:
        return json.loads(res.read().decode('utf-8'))


def find_model():
    data = request_json('GET', '/api/v1/models')
    loaded = [(model['display_name'], inst['id'])
              for model in data.get('models', []) if model.get('type') == 'llm'
              for inst in model.get('loaded_instances', [])]
    if not loaded:
        print('No loaded LLM. Load Gemma 4 E4B in LM Studio, then rerun.')
        return None
    for i, (label, ident) in enumerate(loaded, 1):
        print(f'  {i}. {label} [{ident}]')
    print('Using first loaded model:', loaded[0][0])
    return loaded[0][1]


# v5: independent coupled pair compiler with exact inverse and witnessed relative returns.
class PairMachine:
    """Two readers on one field, in the same n-bit domain.

    Internal E[r]=V[r XOR s] is relative to reader A.  Both readers
    read their OLD field locations before both writes commit, as in
    NEXUS_Pair_Self_Move.c; coincident writes XOR together.
    """
    def __init__(self, n=4, pA=0, wA=2, pB=0, wB=2, field=None):
        mask = (1 << n) - 1
        if not 2 <= n <= 16:
            raise ValueError('supported prototype width is 2..16')
        if any(type(v) is not int or not 0 <= v <= mask
               for v in (pA,wA,pB,wB)):
            raise ValueError('readers and carries must fit n bits')
        D = {int(k):int(v) for k,v in (field or {}).items() if int(v)}
        if any(not (0 <= k <= mask and 0 <= v <= mask) for k,v in D.items()):
            raise ValueError('field keys and values must fit n bits')
        self.n, self.pA, self.wA, self.pB, self.wB = n,pA,wA,pB,wB
        self.V = {k ^ pA: v for k,v in D.items()}
        self.s = 0
        self.last_hash = GENESIS

    def copy(self):
        other = PairMachine(self.n, self.pA, self.wA, self.pB, self.wB)
        other.V,other.s = self.V.copy(),self.s
        other.last_hash = self.last_hash
        return other

    @staticmethod
    def _xor_write(D, key, value):
        updated = D.get(key, 0) ^ value
        if updated:
            D[key] = updated
        else:
            D.pop(key, None)

    def forward(self):
        old_s, old_A, old_B = self.s, self.pA, self.pB
        delta = old_A ^ old_B
        key_A,key_B = old_s, old_s ^ delta
        xA = self.V.get(key_A,0) ^ self.wA
        xB = self.V.get(key_B,0) ^ self.wB
        yA,yB = partner(xA,self.n), partner(xB,self.n)
        self._xor_write(self.V,key_A,yA)
        self._xor_write(self.V,key_B,yB)
        self.s, self.pA,self.pB,self.wA,self.wB = (old_s ^ xA,old_A ^ xA,old_B ^ xB,yA,yB)
        return {'xA':xA,'xB':xB,'written_A':old_A,'written_B':old_B}

    def backward(self):
        xA,xB = partner(self.wA,self.n),partner(self.wB,self.n)
        old_s,old_A,old_B = self.s ^ xA,self.pA ^ xA,self.pB ^ xB
        key_A,key_B = old_s,old_s ^ old_A ^ old_B
        self._xor_write(self.V,key_A,self.wA)
        self._xor_write(self.V,key_B,self.wB)
        old_wA,old_wB = self.V.get(key_A,0) ^ xA, self.V.get(key_B,0) ^ xB
        self.s,self.pA,self.pB,self.wA,self.wB = old_s,old_A,old_B,old_wA,old_wB
        return {'xA':xA,'xB':xB,'restored_A':old_A,'restored_B':old_B}

    def transport(self,h):
        if type(h) is not int or not 0 <= h < (1<<self.n):
            raise ValueError('invalid XOR placement offset')
        self.pA ^= h
        self.pB ^= h
        return {'h':h,'internal_field_unchanged':True}

    def swap(self):
        delta = self.pA ^ self.pB
        self.pA,self.pB = self.pB,self.pA
        self.wA,self.wB = self.wB,self.wA
        self.s ^= delta
        return {'swapped':True,'field_unchanged':True}

    def relative_field(self):
        rel={k ^ self.s:v for k,v in self.V.items()}
        return {str(k):rel[k] for k in sorted(rel)}

    def snapshot(self):
        absolute={self.pA ^ self.s ^ k:v for k,v in self.V.items()}
        return {
            'n':self.n,'pA':self.pA,'wA':self.wA,
            'pB':self.pB,'wB':self.wB,
            'field':{str(k):absolute[k] for k in sorted(absolute)}
        }

    def internal(self):
        return {'delta':self.pA ^ self.pB,'wA':self.wA,'wB':self.wB,'relative_field':self.relative_field()}


def pair_return_certificate(pair, ticks=None, *, expected_h=None, max_steps=20000):
    """Check a SUPPLIED depth; never find t by searching an orbit.

    States BOUND/AMBIGUOUS/UNBOUND/UNLINKED; providers are label-preserved
    and reader-exchanged placements. Does not modify `pair`.
    """
    if ticks is None:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'return depth t is not supplied; derive it independently from the pair compiler'}
    if type(ticks) is not int or ticks <= 0 or ticks > max_steps:
        return {'status':'OPEN','binding':'UNBOUND','missing_link':f'ticks must be 1..{max_steps} for this bounded verification tool; no search performed'}
    mask=(1<<pair.n)-1
    if expected_h is not None and (type(expected_h) is not int or not 0 <= expected_h <= mask):
        return {'status':'OPEN','binding':'UNBOUND','missing_link':'expected_h must be a valid n-bit word'}
    initial=pair.snapshot()
    current=pair.copy()
    for _ in range(ticks):
        current.forward()
    end=current.snapshot()
    candidates=[]
    # Providers are constructed from the required end-reader position, not
    # by enumerating address offsets or scanning intermediate readouts.
    for label_mode in ('same_labels','swapped_labels'):
        provider=pair.copy()
        if label_mode=='swapped_labels':
            provider.swap()
        h=provider.pA ^ current.pA
        if expected_h is not None and h!=expected_h:
            continue
        provider.transport(h)
        if provider.snapshot()==end and h!=0:
            candidates.append({'reader_labels':label_mode,'h':h,
                               'operation': 'transport' if label_mode=='same_labels' else 'swap_then_transport',
                               'way_home':{'operation':'inverse_pair_steps','ticks':ticks}})
    # Verify the return path by inversely replaying the SUPPLIED number of steps.
    reverse=current.copy()
    for _ in range(ticks):
        reverse.backward()
    inverse_ok = reverse.snapshot()==initial
    assert inverse_ok, 'pair reverse failed on a supplied route'
    common={'ticks':ticks,'source':initial,'endpoint':end,'return_verified':inverse_ok,
            'live_machine_unchanged':pair.snapshot()==initial,
            'field_vacuum_at_endpoint':not end['field'],
            'route_policy':'supplied-depth verification; no candidate-depth search'}
    if len(candidates)==1:
        return {'status':'RUNS','binding':'BOUND','route':candidates[0],**common}
    if len(candidates)>1:
        return {'status':'OPEN','binding':'AMBIGUOUS','providers':candidates,
                'missing_link':'reader-label ownership; an independent distinguishing readout is required',**common}
    return {'status':'OPEN','binding':'UNBOUND','providers':[],
            'missing_link':'this supplied depth does not give a nonzero same- or swapped-label transport of the source',**common}


def verify_pair():
    rng=random.Random(20261102)
    checks=0
    for n in range(2,17):
        mask=(1<<n)-1
        for _ in range(240):
            D={}
            for __ in range(rng.randrange(10)):
                key=rng.randrange(mask+1); val=rng.randrange(mask+1)
                if val: D[key]=val
            P=PairMachine(n,rng.randrange(mask+1),rng.randrange(mask+1),
                            rng.randrange(mask+1),rng.randrange(mask+1),D)
            original=P.snapshot()
            # exact inverse with filled fields, including coincident readers
            a=P.copy(); a.forward(); a.backward()
            assert a.snapshot()==original
            b=P.copy(); b.backward(); b.forward()
            assert b.snapshot()==original
            h=rng.randrange(mask+1)
            a=P.copy(); a.transport(h); a.forward()
            b=P.copy(); b.forward(); b.transport(h)
            assert a.snapshot()==b.snapshot()
            a=P.copy(); a.swap(); a.forward()
            b=P.copy(); b.forward(); b.swap()
            assert a.snapshot()==b.snapshot()
            a=P.copy(); a.swap(); a.swap()
            assert a.snapshot()==original
            a=P.copy(); a.transport(h);a.transport(h)
            assert a.snapshot()==original
            # Verify invariant native frame pA^s under steps.
            a=P.copy(); frame=a.pA ^ a.s; a.forward()
            assert (a.pA ^ a.s)==frame
            checks+=7
    # Verify tiny and moderate source-C carrier witnesses, without searching.
    n=4;r=n//2;eR=1<<(r-1);eL=1<<r;u=(1<<n)-1
    def from_k(coord):return ((eR if coord&1 else 0)^(eL if coord&2 else 0)^(u if coord&4 else 0))
    assert [partner(from_k(i),n) for i in range(8)]==[from_k((i ^ 4) if (i&1)==((i>>1)&1) else (i ^ 3)) for i in range(8)]
    a=PairMachine(n,0,eR,0,eR)
    assert pair_return_certificate(a,2)['binding']=='AMBIGUOUS'
    b=PairMachine(n,0,eR,from_k(4),from_k(6))
    result=pair_return_certificate(b,806)
    assert result['binding'] in ('BOUND','AMBIGUOUS') and result['field_vacuum_at_endpoint'],result
    assert pair_return_certificate(b)['binding']=='UNLINKED'
    checks+=4
    print(f'RUNS: {checks:,} pair compiler checks across widths 2..16; two independent source-C supplied-depth certificates.')

PAIR_LEDGER = ROOT / 'NEXUS_local_AI_pair_events.jsonl'  # new; original single-reader ledger untouched
PAIR_GENESIS = '0' * 64


def seed_pair():
    # Exact 806-step witness from NEXUS_Pair_Self_Move.c:
    # A=(0,eR), B=(u,u+eL) in the eight-word carrier at n=4.
    n=4
    u=(1<<n)-1
    eR=1<<(n//2-1)
    eL=1<<(n//2)
    return PairMachine(n, 0,eR,u,u^eL)


def pair_replay(path=None):
    path=PAIR_LEDGER if path is None else Path(path)
    pair=seed_pair()
    pair.last_hash=PAIR_GENESIS
    if not path.exists():
        return pair
    count=0
    with path.open('r',encoding='utf-8') as f:
        for count,line in enumerate(f,1):
            event=json.loads(line)
            if event.get('before')!=pair.snapshot():
                raise RuntimeError(f'Pair ledger discontinuity at line {count}')
            if event.get('prev_hash')!=pair.last_hash:
                raise RuntimeError(f'Pair ledger parent hash mismatch at line {count}')
            payload={k:v for k,v in event.items() if k not in ('prev_hash','event_hash')}
            digest=chain_hash(pair.last_hash,payload)
            if event.get('event_hash')!=digest:
                raise RuntimeError(f'Pair ledger event hash mismatch at line {count}')
            op=event['operation']
            route = pair.forward() if op=='step' else pair.backward() if op=='reverse' else None
            if route is None or route!=event['route'] or pair.snapshot()!=event['after']:
                raise RuntimeError(f'Pair ledger re-execution mismatch at line {count}')
            pair.last_hash=digest
    print(f'RUNS: replayed {count} pair event(s); pair head {pair.last_hash}')
    return pair


def pair_operation(pair,kind,path=None):
    path=PAIR_LEDGER if path is None else Path(path)
    if kind=='state':
        return {'status':'RUNS','state':pair.snapshot(),
                'internal':pair.internal(),'ledger_head':pair.last_hash}
    if kind not in ('step','reverse'):
        return {'status':'OPEN','binding':'UNBOUND','missing_link':'pair mutation requires step or reverse'}
    before=pair.snapshot()
    trial=pair.copy()
    route=trial.forward() if kind=='step' else trial.backward()
    after=trial.snapshot()
    return_machine=trial.copy()
    if kind=='step':return_machine.backward()
    else:return_machine.forward()
    assert return_machine.snapshot()==before
    payload={'version':5,'operation':kind,'before':before,'after':after,'route':route}
    digest=chain_hash(pair.last_hash,payload)
    record={**payload,'prev_hash':pair.last_hash,'event_hash':digest}
    with path.open('a',encoding='utf-8') as f:
        f.write(canonical(record)+'\n')
        f.flush();os.fsync(f.fileno())
    pair.pA,pair.wA,pair.pB,pair.wB,pair.s=(trial.pA,trial.wA,trial.pB,trial.wB,trial.s)
    pair.V=trial.V
    pair.last_hash=digest
    return {'status':'RUNS',**record,'return_verified':True}


def pair_bind_return(pair,ticks=None,expected_h=None):
    before=pair.snapshot();head=pair.last_hash
    result=pair_return_certificate(pair,ticks,expected_h=expected_h)
    assert pair.snapshot()==before and pair.last_hash==head
    return {**result,'ledger_head':head,'ledger_head_unchanged':True}


PAIR_TOOLS=[
    {'type':'function','function':{'name':'nexus_pair_state',
        'description':'READ the independent two-reader coupled field and their relative coordinate; no mutation.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_pair_step',
        'description':'Commit exactly one simultaneous coupled-reader step; records a separate SHA-256 pair ledger event.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_pair_reverse',
        'description':'Commit one exact inverse step of the independent coupled-reader state.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_pair_bind_return',
        'description':'READ-ONLY verify a GIVEN two-reader return depth; determine same-label or swapped-label self-transport and return path. '
                      'DO NOT guess, invent, or scan for a period. Supply ticks=806 for the documented n4 A=(0,eR), B=(u,u+eL) seed.',
        'parameters':{'type':'object','properties':{
            'ticks':{'type':'integer','minimum':1,'maximum':20000,
                     'description':'Required return depth established independently; do not invent.'},
            'expected_h':{'type':'integer','minimum':0,'maximum':15,
                          'description':'Optional independently specified expected XOR displacement.'}},
            'required':['ticks'],'additionalProperties':False}}},
]


def pair_model_dispatch(name,args):
    if not isinstance(args,dict):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'expected object arguments'}
    if 'pair_machine' not in globals() or pair_machine is None:
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'pair engine not initialized'}
    contract={
        'nexus_pair_state':(set(),set()),
        'nexus_pair_step':(set(),set()),
        'nexus_pair_reverse':(set(),set()),
        'nexus_pair_bind_return':({'ticks','expected_h'},{'ticks'})
    }
    if name not in contract:
        return {'status':'OPEN','binding':'UNBOUND','missing_link':'not a pair tool'}
    allowed,required=contract[name]
    supplied=set(args)
    if not required.issubset(supplied) or not supplied.issubset(allowed):
        return {'status':'OPEN','binding':'UNLINKED','missing_fields':sorted(required-supplied),
                'unexpected_fields':sorted(supplied-allowed)}
    if name=='nexus_pair_bind_return':
        return pair_bind_return(pair_machine,ticks=args['ticks'],expected_h=args.get('expected_h'))
    return pair_operation(pair_machine,{
        'nexus_pair_state':'state','nexus_pair_step':'step','nexus_pair_reverse':'reverse'}[name])

# v4.1: explicit contracts, one task per tool, so an empty call is invalid.
# The internal bind() resolver is unchanged: no mathematical rule was edited.
STATE_SCHEMA = {
    'type': 'object',
    'properties': {
        'n': {'type': 'integer'},
        'p': {'type': 'integer'},
        'w': {'type': 'integer'},
        'field': {'type': 'object', 'additionalProperties': {'type': 'integer'}},
    },
    'required': ['n','p','w','field'],
    'additionalProperties': False,
}

TOOLS = [
    {'type':'function','function':{'name':'nexus_state',
       'description':'READ current exact state and ledger digest; no mutation.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_step',
       'description':'Execute one reversible forward reader-field step F.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_reverse',
       'description':'Execute exact inverse F^-1 of the current state.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_transport',
       'description':'Execute XOR relocation H_h on the LIVE machine; changes its position, writes a ledger event.',
       'parameters':{'type':'object','properties':{'h':{
           'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1}},
           'required':['h'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_commute',
       'description':'Read-only verify F H_h = H_h F for the current state, without committing a change.',
       'parameters':{'type':'object','properties':{'h':{
           'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1}},
           'required':['h'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind',
       'description':'READ-ONLY BIND: preserve the current WHAT and relocate its reader to a REQUIRED target WHERE. '
                     'Call with exactly {"target_where":8} if target WHERE is 8. '
                     'Derives h = current_p XOR target_where and the exact inverse route. '
                     'Do not send empty arguments or other goal fields.',
       'parameters':{'type':'object','properties':{
           'target_where': {'type':'integer','minimum':0,'maximum':(1<<WIDTH)-1,
                            'description':'REQUIRED absolute destination reader position, not XOR offset.'}},
           'required':['target_where'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind_predecessor',
       'description':'READ-ONLY BIND: derive the unique F-predecessor of the CURRENT complete state.',
       'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind_predecessor_of',
       'description':'READ-ONLY BIND: derive the unique F-predecessor of a supplied COMPLETE destination state. '
                     'All four fields n,p,w,field are REQUIRED; never assume empty omitted field.',
       'parameters':{'type':'object','properties':{'target_state':STATE_SCHEMA},
                     'required':['target_state'],'additionalProperties':False}}},
    {'type':'function','function':{'name':'nexus_bind_endpoint',
       'description':'READ-ONLY BIND: identify the one-step provider (transport, F, or F^-1) '
                     'for a supplied COMPLETE target state. May return BOUND, AMBIGUOUS, or UNBOUND.',
       'parameters':{'type':'object','properties':{'target_state':STATE_SCHEMA},
                     'required':['target_state'],'additionalProperties':False}}},
]

TOOLS.extend(PAIR_TOOLS)

SYSTEM = (
    'You interface with an independently executing NEXUS compiler. '
    'Use actual tools; never invent results. '
    'When the user requests preserving current WHAT at destination WHERE, '
    'call nexus_bind with required argument target_where equal to that absolute address. '
    'For current predecessor use nexus_bind_predecessor. '
    'For a supplied complete endpoint use nexus_bind_endpoint. '
    'Do not pass mutually exclusive goals together. '
    'Do not invent missing readout fields or choose an operator if AMBIGUOUS. '
    'BIND is read-only. Explain tool result and its way home, without claiming the live state changed. '
    'For the bound PAIR, use nexus_pair_bind_return with an independently supplied ticks; '
    'the compiler verifies but does not discover the period. Never invent tick counts.'
)

def dispatch_model_tool(machine, name, args):
    """Exact gateway. Invalid args are OPEN and never mutate the machine."""
    if not isinstance(args, dict):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'tool arguments must be a JSON object'}
    if name.startswith('nexus_pair_'):
        return pair_model_dispatch(name,args)
    contracts = {
        'nexus_state': (set(),set()),
        'nexus_step': (set(),set()),
        'nexus_reverse': (set(),set()),
        'nexus_transport': ({'h'},{'h'}),
        'nexus_commute': ({'h'},{'h'}),
        'nexus_bind': ({'target_where'},{'target_where'}),
        'nexus_bind_predecessor': (set(),set()),
        'nexus_bind_predecessor_of': ({'target_state'},{'target_state'}),
        'nexus_bind_endpoint': ({'target_state'},{'target_state'}),
    }
    if name not in contracts:
        return {'status':'OPEN','error':'unknown compiler tool '+str(name)}
    allowed, required = contracts[name]
    supplied = set(args)
    if not required.issubset(supplied) or not supplied.issubset(allowed):
        return {
            'status':'OPEN','binding':'UNLINKED',
            'missing_fields': sorted(required-supplied),
            'unexpected_fields': sorted(supplied-allowed),
            'missing_link':'invalid tool arguments for '+name+'; required fields '+str(sorted(required)),
            'received_args': args,
        }
    if name == 'nexus_bind':
        return bind(machine, target_where=args['target_where'])
    if name == 'nexus_bind_predecessor':
        return bind(machine, predecessor_of_current=True)
    if name == 'nexus_bind_predecessor_of':
        return bind(machine, predecessor_of=args['target_state'])
    if name == 'nexus_bind_endpoint':
        return bind(machine, target_state=args['target_state'])
    if name == 'nexus_commute':
        return commute(machine, args['h'])
    return operation(machine, {'nexus_state':'state','nexus_step':'step',
                              'nexus_reverse':'reverse','nexus_transport':'transport'}[name],
                     args.get('h'))


def ask_model(model_id, machine, user_text, history=None):
    # Retains the earlier calling convention. Print actual tool arguments to disambiguate failures.
    messages = history if history is not None else [{'role':'system','content':SYSTEM}]
    messages.append({'role':'user','content':user_text})
    for _ in range(16):
        response = request_json('POST', '/v1/chat/completions', {
            'model': model_id, 'messages': messages, 'tools': TOOLS,
            'tool_choice':'auto', 'temperature':0.1, 'max_tokens':1024,
            'stream':False,
        })
        msg = response['choices'][0]['message']
        calls = msg.get('tool_calls') or []
        if not calls:
            content = msg.get('content') or '(no text returned)'
            messages.append({'role':'assistant','content':content})
            print('\nMODEL:',content)
            return content
        messages.append({'role':'assistant','content':msg.get('content'), 'tool_calls':calls})
        for call in calls:
            fn = call.get('function',{}).get('name','')
            raw_args = call.get('function',{}).get('arguments') or '{}'
            print(f'  CALL {fn} raw arguments: {raw_args}')
            try:
                args = json.loads(raw_args)
                result = dispatch_model_tool(machine,fn,args)
            except (ValueError,TypeError,json.JSONDecodeError) as err:
                result = {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid tool JSON: '+str(err)}
            print(f'  TOOL {fn}:',canonical(result))
            messages.append({'role':'tool','tool_call_id':call['id'],'content':canonical(result)})
    print('Tool-round budget reached; exact machine state:',canonical(machine.snapshot()))

def main():
    global pair_machine
    verify()
    verify_bind()
    verify_pair()
    pair_machine=pair_replay()
    print('BIND prediction SHA-256:', BIND_PREDICTION_SHA256)
    machine = replay()
    print('Current state:', canonical(machine.snapshot()))
    print('Pair seed/current:', canonical(pair_machine.snapshot()))
    print('LM Studio:', BASE)
    try:
        model_id = find_model()
    except (URLError, HTTPError, TimeoutError, ValueError) as err:
        print('LM Studio not reachable:', err)
        model_id = None
    chat = [{'role':'system','content':SYSTEM}]
    print('Commands: :state, :step, :reverse, :transport <h>, :commute <h>, :bind <WHERE>, :back, :pair-state, :pair-step, :pair-reverse, :pair-return <ticks>, :clear, :quit')
    while True:
        try:
            line = input('nexus> ').strip()
        except (EOFError, KeyboardInterrupt):
            break
        if not line: continue
        if line == ':quit': break
        if line in (':pair-state',':pair-step',':pair-reverse'):
            print(json.dumps(pair_operation(pair_machine, line[len(':pair-'):]), indent=2))
            continue
        if line.startswith(':pair-return'):
            parts=line.split()
            try:
                ticks=int(parts[1]) if len(parts)==2 else None
                print(json.dumps(pair_bind_return(pair_machine,ticks=ticks),indent=2))
            except ValueError as err:
                print('Invalid ticks:',err)
            continue
        if line == ':clear':
            chat[:] = [{'role':'system','content':SYSTEM}]
            print('Conversation context reset; deterministic machine unchanged.')
        elif line in (':state',':step',':reverse'):
            print(json.dumps(operation(machine,line[1:]),indent=2))
        elif line == ':back':
            print(json.dumps(bind(machine,predecessor_of_current=True),indent=2))
        elif line.startswith(':bind '):
            try:
                print(json.dumps(bind(machine,target_where=int(line.split()[1],0)),indent=2))
            except (ValueError,IndexError) as err:
                print('Invalid WHERE:',err)
        elif line.startswith(':transport') or line.startswith(':commute'):
            parts = line.split()
            if len(parts) != 2:
                print('Usage: :transport 3 or :commute 3  (also accepts 0x3)')
                continue
            try:
                h = int(parts[1],0)
                print(json.dumps(commute(machine,h) if parts[0] == ':commute'
                                 else operation(machine,'transport',h),indent=2))
            except ValueError as err:
                print('Invalid transport:', err)
        elif not model_id:
            print('Model unavailable; compiler commands still work.')
        else:
            try:
                ask_model(model_id,machine,line,chat)
            except (URLError,HTTPError,TimeoutError,KeyError,ValueError) as err:
                print('LM Studio API error:',err)



# Jupyter startup: no input() loop. Both engines keep separate ledgers.
verify()
verify_bind()
verify_pair()
machine = replay()
pair_machine = pair_replay()
print("\nNEXUS PAIR v5 READY — two isolated compiler states")
print("Single reader:", canonical(machine.snapshot()))
print("Paired readers:", canonical(pair_machine.snapshot()))
print("NEXUS pair ledger:", str(PAIR_LEDGER))
print("LM Studio:", BASE)
print("To verify the supplied 806-step pair return, use: pair_bind_return(pair_machine,806)")


"""NEXUS v5.1 paired-return auto-BIND and exact switch-tape certificate.

Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828.

Run AFTER the first code cell of NEXUS_v5_notebook. Uses the existing
PairMachine, partner, PAIR_TOOLS, TOOLS, SYSTEM and pair_model_dispatch.

The full-state seed and witness remain with their original ledger owner.
A vacuum event is reached by the deterministic native step, not by testing
candidate depths. It is an observed route, not a sealed pre-data landing.
"""


def invert_periodic_switch_tape(q):
    """Recover a complete period-T carry tape from q=(I+S+S^2)w.

    Work over GF(2), with S shifting the stream forward one index.
    The zero-seed recurrence z[t+2]=z[t]+z[t+1]+q[t] gives the
    particular displacement (c0,c1) after T ticks.
    Homogeneous advance is the triad matrix M with M^3=I.

    At T mod 3 == 1: (w0,w1)=(c1,c0+c1).
    At T mod 3 == 2: (w0,w1)=(c0+c1,c0).
    At T mod 3 == 0: two word-valued initial conditions are free.
    """
    T = len(q)
    if T < 1:
        raise ValueError('switch tape cannot be empty')
    if T % 3 == 0:
        return {'status':'OPEN', 'binding':'UNBOUND', 'reason':
                'Q_T is singular: two word-valued triad anchors are unlinked',
                'period':T, 'kernel_word_dimension':2}
    c0 = c1 = 0
    for qt in q:
        c0,c1 = c1,c0 ^ c1 ^ qt
    if T % 3 == 1:
        w0,w1 = c1,c0 ^ c1
    else:
        w0,w1 = c0 ^ c1,c0
    w = [w0,w1]
    for qt in q:
        w.append(w[-2] ^ w[-1] ^ qt)
    if (w[T],w[T+1]) != (w0,w1):
        raise ArithmeticError('periodic carry endpoint mismatch')
    out=w[:T]
    if any((out[t]^out[(t+1)%T]^out[(t+2)%T])!=q[t] for t in range(T)):
        raise ArithmeticError('recovered carry tape disagrees with switches')
    return {'status':'STANDS','binding':'BOUND','period':T,
            'kernel_word_dimension':0,'carries':out}


def _xor_field_write(field, key, value):
    value = field.get(key,0) ^ value
    if value:field[key]=value
    else:field.pop(key,None)


def _switch_digest(q,n):
    import hashlib
    byte_count=(n+7)//8
    payload=b'NEXUS/v5.1/switch/GF2'+n.to_bytes(2,'big')+len(q).to_bytes(8,'big')
    payload += b''.join(int(x).to_bytes(byte_count,'big') for x in q)
    return hashlib.sha256(payload).hexdigest()


def pair_bind_first_vacuum(pair, *, max_ticks=20000, reconstruct_switches=True):
    """READ-ONLY event-driven BIND to first nonzero free return; reconstruct full orbit.

    No supplied depth. The event detector runs native F until the first D=0.
    Only two possible ownership labels (same/swapped) are checked at the
    endpoint. An H_h or swap-H_h landing of order two derives T=2*t.
    For 3∤T, each reader's switches independently invert into its entire
    carry stream. The two carry streams reconstruct the joint field and
    both WHERE paths from their initial positions and D=0, without a carry seed.
    """
    if type(max_ticks) is not int or not 1 <= max_ticks <= 200000:
        raise ValueError('max_ticks must be 1..200000')
    initial=pair.snapshot()
    head=pair.last_hash
    common={'source':initial,'ledger_head':head,'live_machine_unchanged':True,
            'method':'first-vacuum event along native F; no candidate-depth proposals'}
    if initial['field']:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'initial retained field must be empty for this seed-relative certificate',**common}
    forward=pair.copy()
    first=None
    for tick in range(1,max_ticks+1):
        forward.forward()
        if not forward.V:
            first=tick
            break
    if first is None:
        return {'status':'OPEN','binding':'UNBOUND',
                'missing_link':f'no vacuum event during {max_ticks} native steps; horizon is not a period claim',
                'ticks_executed':max_ticks,**common}
    endpoint=forward.snapshot()
    # From empty field and the exact single-carrier 3-bit charge, the relative
    # separation at a return is fixed. Verify rather than infer this at other widths.
    old_r=initial['pA']^initial['pB']
    new_r=endpoint['pA']^endpoint['pB']
    if old_r!=new_r:
        return {'status':'OPEN','binding':'UNBOUND',
                'missing_link':'relative separation changed; affine one-carrier law not applicable',
                'first_vacuum_tick':first,'endpoint':endpoint,
                'relative_before':old_r,'relative_after':new_r,**common}
    providers=[]
    for label in ('same_labels','swapped_labels'):
        owner=pair.copy()
        if label=='swapped_labels':owner.swap()
        h=owner.pA ^ forward.pA
        owner.transport(h)
        if h!=0 and owner.snapshot()==endpoint:
            providers.append({'reader_labels':label,'h':h,
                              'operation':'transport' if label=='same_labels' else 'swap_then_transport',
                              'way_home':{'operation':'inverse_pair_steps','ticks':first},
                              'endpoint_way_home':{'operation':'same_involutive_provider','h':h,'reader_labels':label}})
    result={'first_vacuum_tick':first,'endpoint':endpoint,
            'relative_separation':old_r,'providers':providers,**common}
    if len(providers)!=1:
        return {'status':'OPEN','binding':'AMBIGUOUS' if providers else 'UNBOUND',
                'missing_link':'label ownership' if providers else 'free endpoint has different carry ownership: not a translated complete state',
                **result}
    provider=providers[0]
    T=2*first
    result.update({'status':'RUNS','binding':'BOUND','route':provider,
                   'period':T,'period_derived_from':'equivariance and order-two free-return provider',
                   'reader_carries_at_first_vacuum':[endpoint['wA'],endpoint['wB']]})
    if not reconstruct_switches:
        return result
    if T%3==0:
        result['switch_reconstruction']={'status':'OPEN','binding':'UNBOUND',
            'period_mod_3':0,'kernel_word_dimension_per_reader':2,
            'missing_link':'each carry tape needs two independent word anchors'}
        return result
    # Record ONLY the carry tapes, not the full state history. The event proves
    # the full period from the order-two symmetry; check it independently.
    witness=pair.copy()
    W_A=[];W_B=[]
    for _ in range(T):
        W_A.append(witness.wA)
        W_B.append(witness.wB)
        witness.forward()
    if witness.snapshot()!=initial:
        raise AssertionError('derived period failed direct paired replay')
    qA=[W_A[t]^W_A[(t+1)%T]^W_A[(t+2)%T] for t in range(T)]
    qB=[W_B[t]^W_B[(t+1)%T]^W_B[(t+2)%T] for t in range(T)]
    inv_A=invert_periodic_switch_tape(qA)
    inv_B=invert_periodic_switch_tape(qB)
    if inv_A['binding']!='BOUND' or inv_B['binding']!='BOUND':
        raise AssertionError('expected nonsingular switch inversion')
    recA,recB=inv_A['carries'],inv_B['carries']
    # This is diagnostic comparison only; reconstruction below gets the
    # original initial WHEREs and vacuum, but NOT the original carry seeds.
    if recA!=W_A or recB!=W_B:
        raise AssertionError('switch-only carry reconstruction mismatch')
    del W_A,W_B
    pA,pB=initial['pA'],initial['pB']
    D={}
    verifier=pair.copy()
    first_rebuilt=None
    matched=0
    for tick in range(T):
        wa,wb=recA[tick],recB[tick]
        wna,wnb=recA[(tick+1)%T],recB[(tick+1)%T]
        xa,xb=partner(wna,pair.n),partner(wnb,pair.n)
        # Check simultaneous reads against the freshly reconstructed field.
        if (D.get(pA,0)^wa)!=xa or (D.get(pB,0)^wb)!=xb:
            raise AssertionError(f'switch tape violates joint read at tick {tick}')
        _xor_field_write(D,pA,wna)
        _xor_field_write(D,pB,wnb)
        pA^=xa; pB^=xb
        verifier.forward()
        expected=verifier.snapshot()
        if (pA,pB,wna,wnb,D)!=(expected['pA'],expected['pB'],
            expected['wA'],expected['wB'],
            {int(k):v for k,v in expected['field'].items()}):
            raise AssertionError(f'absolute reconstructed state disagrees at tick {tick+1}')
        matched+=1
        if tick+1==first:
            first_rebuilt={'n':pair.n,'pA':pA,'wA':wna,'pB':pB,'wB':wnb,
                           'field':{str(k):D[k] for k in sorted(D)}}
    if first_rebuilt!=endpoint or D or (pA,pB,recA[0],recB[0])!=(initial['pA'],initial['pB'],initial['wA'],initial['wB']):
        raise AssertionError('switch reconstructed end of full return failed')
    if pair.snapshot()!=initial or pair.last_hash!=head:
        raise AssertionError('read-only certificate modified live machine')
    result['switch_reconstruction']={
        'status':'RUNS','binding':'BOUND','period_mod_3':T%3,
        'kernel_word_dimension_per_reader':0,
        'qA_sha256':_switch_digest(qA,pair.n),
        'qB_sha256':_switch_digest(qB,pair.n),
        'qA_prefix':qA[:10],'qB_prefix':qB[:10],
        'switch_nonzero_A':sum(bool(v) for v in qA),
        'switch_nonzero_B':sum(bool(v) for v in qB),
        'recovered_initial_carries':[recA[0],recB[0]],
        'input_carry_anchors':0,
        'required_external_WHERES':2,
        'initial_field':'vacuum',
        'full_joint_states_matched':matched,
        'first_vacuum_reconstructed':first_rebuilt==endpoint,
        'full_return_reconstructed':True,
        'live_machine_unchanged':True,
        'way_home':'inverse joint F for each tick; endpoint also has its exact order-two provider'}
    return result


# Extend the existing live LM Studio tool gateway without altering old v5 tools.
# Keep the original v5 dispatcher stable if this Jupyter cell is re-executed.
if '_PAIR_DISPATCH_V5_ORIGINAL' not in globals():
    _PAIR_DISPATCH_V5_ORIGINAL = pair_model_dispatch

def pair_model_dispatch(name,args):
    if name=='nexus_pair_bind_first_vacuum':
        if not isinstance(args,dict) or args:
            return {'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'nexus_pair_bind_first_vacuum takes no arguments'}
        if 'pair_machine' not in globals() or pair_machine is None:
            return {'status':'OPEN','binding':'UNLINKED','missing_link':'pair engine is not initialized'}
        return pair_bind_first_vacuum(pair_machine)
    return _PAIR_DISPATCH_V5_ORIGINAL(name,args)


_PAIR_AUTOBIND_TOOL={
    'type':'function','function':{
        'name':'nexus_pair_bind_first_vacuum',
        'description':'READ-ONLY: derive the first empty-field event by exact simultaneous native execution. '
                      'Bind its end state to same-label or exchanged-label transport (when unique); '
                      'derive the period from the order-two symmetry, then reconstruct both readers and '
                      'their shared field from the two switch tapes without carry seeds. '
                      'No ticks parameter. Does not mutate the live pair or its ledger.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}}
if not any(x.get('function',{}).get('name')=='nexus_pair_bind_first_vacuum' for x in PAIR_TOOLS):
    PAIR_TOOLS.append(_PAIR_AUTOBIND_TOOL)
if not any(x.get('function',{}).get('name')=='nexus_pair_bind_first_vacuum' for x in TOOLS):
    TOOLS.append(_PAIR_AUTOBIND_TOOL)
_V51_SYSTEM_PROMPT = (' For a paired return with UNKNOWN depth, call nexus_pair_bind_first_vacuum '
           'with empty arguments. Never supply a guessed depth. The binder derives the '
           'first vacuum event, checks complete carry ownership, and attaches the '
           'switch-tape reconstruction and way home. If OPEN, report the missing link.')
if _V51_SYSTEM_PROMPT not in SYSTEM:
    SYSTEM += _V51_SYSTEM_PROMPT


"""NEXUS AI v5.2 — paired switch holonomy and boundary compiler.
Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828.

DROP-IN EXTENSION: run AFTER both v5.1 notebook engine and switch-inversion cells.
Standard library only; no mutation to live machines or their ledgers.

Compiler contract:
  1. On the native even-width eight-word carrier, P(x)=A(x) XOR u,
     where A is linear and involutive.
  2. Given a CYCLIC switch tape q=(I+S+S^2)w of known length T,
     the endpoint placement residue is
       h = A(XOR_t q[t]) XOR ((T & 1)*u).
  3. A same-label paired vacuum return requires equal reader residues,
     but that equality alone is not sufficient. Invert the switches,
     then check every simultaneous field read/write and final vacuum.
  4. If T is a multiple of 3, do not invent carry anchors. Check
     q residue-class XOR equality; otherwise name the missing anchors.

This module does NOT infer T or q from just the initial boundary.
The optional event reader executes to the first field vacuum, then
uses the independently derived switch checksum to certify WHERE.
"""
from functools import reduce
from operator import xor
import hashlib

V52_PREDICTION = """NEXUS v5.2 | 2026-11-02 | switch-boundary seal before data linkage.
WHAT: On an even-width eight-word native carrier K=span(u,eR,eL),
P(x)=A(x) xor u with A linear and A*A=I. For a carry-periodic
switch tape q=(I+S+S^2)w of length T, h=A(XOR q) xor [(T mod 2)u].
For a same-label vacuum return, the two reader switch XORs must match;
equality alone does NOT establish a vacuum or a valid trajectory.
For 3 not dividing T, invert q uniquely and verify every simultaneous
read/write to bind a complete translated return.
For 3 dividing T, a cyclic solution exists iff the XORs of q at
indices 0,1,2 modulo 3 agree; if so, two word-valued carry anchors
remain free per reader before coupled constraints are supplied.
Way home: BOUND endpoint returns via H_h; full history via inverse joint steps.
New execution readout: the seed (A=0,eR; B=u,u xor eL; D=0)
shall give first vacuum at 806, checksum-derived h=u xor eR xor eL,
and independent reconstructed joint states for even n=4..16.
No result is claimed for deriving 806 before executing to the event.
"""
V52_SEAL_SHA256 = hashlib.sha256(V52_PREDICTION.encode('utf-8')).hexdigest()


def _v52_carrier(n):
    if type(n) is not int or n < 4 or n > 16 or n % 2:
        raise ValueError('eight-word affine carrier supported at even widths 4..16')
    u = (1 << n) - 1
    eR, eL = 1 << (n//2-1), 1 << (n//2)
    K = frozenset((a*u) ^ (b*eR) ^ (c*eL)
                  for a in (0,1) for b in (0,1) for c in (0,1))
    return u, K, eR, eL


def _v52_A(x, n, K, u):
    if x not in K:
        raise ValueError('word is outside the declared affine carrier')
    return partner(x, n) ^ u


def _v52_xor(seq):
    return reduce(xor, seq, 0)


def _v52_first_bad(tapes):
    for label, q in tapes:
        if len(q) % 3 == 0:
            totals = [_v52_xor(q[j::3]) for j in range(3)]
            if len(set(totals)) != 1:
                return label, totals
    return None


def pair_switch_boundary(qA=None, qB=None, *, n=4, pA=0, pB=15):
    """Bind a same-label, empty-field translated return from FULL CYCLIC tapes.

    Inputs are SWITCH words, not the original carry tapes. Does not
    access PairMachine or any ledger. Does not presume unknown data=0;
    the contract explicitly fixes D0=0, and the caller supplies both WHEREs.
    """
    base = {'source':'switch_tapes','initial_field':'explicit_vacuum',
            'operation':'read_only_pair_switch_boundary',
            'return_type':'same_labels'}
    if qA is None or qB is None:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'both full cyclic switch tapes qA and qB are required', **base}
    if not isinstance(qA,(list,tuple)) or not isinstance(qB,(list,tuple)) or not qA or len(qA)!=len(qB):
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'qA and qB must be nonempty equal-length sequences',**base}
    try: u,K,_,_ = _v52_carrier(n)
    except ValueError as exc:
        return {'status':'OPEN','binding':'UNBOUND','missing_link':str(exc),**base}
    T=len(qA)
    if any(type(z) is not int or z not in K for z in (*qA,*qB)):
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'switch words must be linked as valid elements of the same native carrier',**base}
    if any(type(z) is not int or not 0<=z<=u for z in (pA,pB)) or (pA^pB) not in K:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'both starting WHEREs must be valid and in one carrier coset',**base}
    Q_A,Q_B=_v52_xor(qA),_v52_xor(qB)
    hA=_v52_A(Q_A,n,K,u) ^ (u if T&1 else 0)
    hB=_v52_A(Q_B,n,K,u) ^ (u if T&1 else 0)
    basics={'ticks':T,'switch_xor_A':Q_A,'switch_xor_B':Q_B,
            'h_from_A':hA,'h_from_B':hB,'start_where':[pA,pB],**base}
    if Q_A != Q_B:
        return {'status':'OPEN','binding':'UNBOUND',
                'missing_link':'unequal switch checksums imply unequal reader displacements; no common translation',**basics}
    if T%3==0:
        bad=_v52_first_bad([('A',qA),('B',qB)])
        if bad is not None:
            return {'status':'OPEN','binding':'UNBOUND',
                    'missing_link':'singular switch operator image condition fails',
                    'reader':bad[0],'mod3_xor_totals':bad[1],**basics}
        return {'status':'OPEN','binding':'AMBIGUOUS',
                'missing_link':'two word-valued carry anchors per reader; couple to independent boundary reads',
                'kernel_word_dimension_per_reader':2,**basics}
    inv_A=invert_periodic_switch_tape(list(qA))
    inv_B=invert_periodic_switch_tape(list(qB))
    if inv_A.get('binding')!='BOUND' or inv_B.get('binding')!='BOUND':
        return {'status':'OPEN','binding':'UNBOUND',
                'missing_link':'carry inversion did not bind',**basics}
    wa,wb=inv_A['carries'],inv_B['carries']
    DA={};a,b=pA,pB
    for t in range(T):
        xa=partner(wa[(t+1)%T],n)
        xb=partner(wb[(t+1)%T],n)
        # Both reads are simultaneous from OLD field, prior to either write.
        if (DA.get(a,0)^wa[t])!=xa or (DA.get(b,0)^wb[t])!=xb:
            return {'status':'OPEN','binding':'UNBOUND',
                    'missing_link':'cyclic carry solution violates simultaneous old-field reads',
                    'first_bad_tick':t,**basics}
        _xor_field_write(DA,a,wa[(t+1)%T])
        _xor_field_write(DA,b,wb[(t+1)%T])
        a ^= xa; b ^= xb
    if (a,b)!=(pA^hA,pB^hB):
        raise AssertionError('analytic switch checksum and reconstructed placement disagree')
    if DA:
        return {'status':'OPEN','binding':'UNBOUND',
                'missing_link':'read-consistent cyclic carries fail final field-vacuum closure',
                'remaining_field_cells':len(DA),**basics}
    return {'status':'RUNS','binding':'BOUND',
            'endpoint':{'n':n,'pA':a,'pB':b,'wA':wa[0],'wB':wb[0],'field':{}},
            'recovered_initial_carries':[wa[0],wb[0]],
            'input_carry_anchors':0,'reads_verified':T*2,
            'route':{'operation':'internal_pair_steps','ticks':T,
                     'symmetry':'H_h','h':hA},
            'way_home':{'operation':'transport','h':hA,
                        'historical_inverse':{'operation':'inverse_pair_steps','ticks':T}},
            'full_return_via_involution':T*2 if hA!=0 else T,
            **basics}


def pair_switch_holonomy(pair, *, max_ticks=20000):
    """Read-only first-vacuum event, then bind its switch holonomy.

    This method still EXECUTES until the event. It does not derive T
    from static boundary data. It derives WHERE from the switch checksum
    before comparing the actually observed endpoint.
    """
    if type(max_ticks) is not int or max_ticks<1 or max_ticks>200000:
        raise ValueError('max_ticks must be in 1..200000')
    initial=pair.snapshot(); head=pair.last_hash
    base={'source':initial,'ledger_head':head,'live_machine_unchanged':True,
          'method':'event-driven first vacuum followed by switch-checksum binding'}
    if initial['field']:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'explicit empty field needed at initial boundary',**base}
    working=pair.copy()
    WA=[];WB=[];tick=None
    for t in range(1,max_ticks+1):
        WA.append(working.wA);WB.append(working.wB)
        working.forward()
        if not working.V:
            tick=t
            break
    if tick is None:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':f'first vacuum is not linked within {max_ticks} executed ticks; no absence claim',
                'ticks_executed':max_ticks,**base}
    endpoint=working.snapshot()
    if endpoint['wA']!=initial['wA'] or endpoint['wB']!=initial['wB']:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'first vacuum does not close labeled carry tapes; exchanged-label contract needs separate binding',
                'first_vacuum_tick':tick,'endpoint':endpoint,**base}
    qa=[WA[i]^WA[(i+1)%tick]^WA[(i+2)%tick] for i in range(tick)]
    qb=[WB[i]^WB[(i+1)%tick]^WB[(i+2)%tick] for i in range(tick)]
    cert=pair_switch_boundary(qa,qb,n=pair.n,pA=pair.pA,pB=pair.pB)
    assert pair.snapshot()==initial and pair.last_hash==head
    if cert.get('binding')!='BOUND':
        return {**cert,'first_vacuum_tick':tick,'actual_endpoint':endpoint,**base}
    if cert['endpoint']!=endpoint:
        raise AssertionError('switch-only reconstructed endpoint disagrees with executed endpoint')
    h=cert['route']['h']
    expected=pair.copy(); expected.transport(h)
    assert expected.snapshot()==endpoint
    return {**cert,'status':'RUNS','binding':'BOUND',
            'first_vacuum_tick':tick,
            'switch_sha256_A':_switch_digest(qa,pair.n),
            'switch_sha256_B':_switch_digest(qb,pair.n),
            'switch_tape_lengths':[len(qa),len(qb)],
            'observed_h':initial['pA']^endpoint['pA'],
            'holonomy_equals_observed':True,
            **base}

# Optional LM Studio tool registration, safe to rerun this cell.
if 'PAIR_TOOLS' in globals() and 'TOOLS' in globals() and 'pair_model_dispatch' in globals():
    if '_PAIR_DISPATCH_BEFORE_V52' not in globals():
        _PAIR_DISPATCH_BEFORE_V52 = pair_model_dispatch
    def pair_model_dispatch(name,args):
        if name=='nexus_pair_switch_holonomy':
            if not isinstance(args,dict) or args:
                return {'status':'OPEN','binding':'UNLINKED',
                        'missing_link':'nexus_pair_switch_holonomy accepts no arguments'}
            return pair_switch_holonomy(pair_machine)
        return _PAIR_DISPATCH_BEFORE_V52(name,args)
    _V52_TOOL={'type':'function','function':{
        'name':'nexus_pair_switch_holonomy',
        'description':'READ-ONLY: execute to first field-vacuum event without supplied depth, then derive the SAME-LABEL translated WHERE from the cyclic switch checksum. Independently recover carries, verify all joint reads and writes, and attach exact return. Does not derive clock before execution.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}}}
    if not any(t.get('function',{}).get('name')=='nexus_pair_switch_holonomy' for t in PAIR_TOOLS):
        PAIR_TOOLS.append(_V52_TOOL)
    if not any(t.get('function',{}).get('name')=='nexus_pair_switch_holonomy' for t in TOOLS):
        TOOLS.append(_V52_TOOL)
    if 'SYSTEM' in globals():
        _V52_PROMPT=' For switch-derived WHERE, use nexus_pair_switch_holonomy with empty arguments; do not supply ticks. Treat this as event-driven depth discovery, not a static period calculation.'
        if _V52_PROMPT not in SYSTEM:
            SYSTEM += _V52_PROMPT

print('NEXUS v5.2 switch-boundary compiler installed; pre-readout seal:',V52_SEAL_SHA256)


"""NEXUS AI v5.3 — compiler-side relative pair and boundary-goal constraints.
Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828.

RUN AFTER the v5.2 engine in a single Jupyter cell, or use the v5.3
complete notebook. Standard library only. No LM call or ledger writes.

This is a STATIC goal compiler. It does not guess ticks or enumerate
candidate cycles. A viable target with no independent monodromy map
returns OPEN/UNLINKED; a mathematically impossible target returns
RUNS/UNBOUND. BOUND in spatial_binding means only that the spatial
XOR offset has been derived, NOT that an executable orbit is known.
"""

import hashlib as _v53_hashlib
import json as _v53_json
import random as _v53_random

V53_SEAL_TEXT = ("NEXUS v5.3 STATIC PREDICTION; Dean A. Kulik; 2026-11-03.\n"
 "WAY HOME: the relative paired native map has an exact inverse from "
 "new carries and new relative field.\n"
 "For an even-width affine carrier K=span(u,eR,eL), every initial carry, "
 "relative reader separation, active field coordinate and field value "
 "inside K stays there under the simultaneous pair update. Therefore "
 "no internal step can relocate this closed-carrier seed into another "
 "K coset.\n"
 "The charge C=XOR(D values) XOR A(delta) is invariant; any vacuum "
 "endpoint must have delta'=A(C), independently of its clock.\n"
 "The source seed (n=4, A=(0,2), B=(15,11), D={}) is carrier-closed. "
 "Its goal outside the carrier is UNBOUND; target A=9 is spatially "
 "BOUND to h=9, but its switch tapes and vacuum return time stay UNLINKED "
 "until a separate monodromy or executed certificate supplies them.\n")
V53_SEAL_SHA256 = _v53_hashlib.sha256(V53_SEAL_TEXT.encode()).hexdigest()


def _v53_clean_field(d):
    result = {int(k): int(v) for k,v in d.items() if int(v)}
    return result


def _v53_rel_from_pair(pair):
    snap = pair.snapshot()
    pA = snap['pA']
    return {'n': snap['n'], 'delta': pA ^ snap['pB'],
            'wA': snap['wA'], 'wB': snap['wB'],
            'E': {int(k)^pA: int(v) for k,v in snap['field'].items() if int(v)}}


def _v53_rel_forward(R):
    """Exactly one simultaneous native step, in A-relative coordinates.

    E'[r] = E[r xor xA] xor ([r=xA] yA) xor ([r=xA xor delta] yB)
    delta' = delta xor xA xor xB ; pA' = pA xor xA.
    """
    n, d, e = R['n'], R['delta'], R['E']
    xa, xb = e.get(0, 0)^R['wA'], e.get(d, 0)^R['wB']
    ya,yb = partner(xa,n),partner(xb,n)
    out = {r^xa:v for r,v in e.items()}
    for addr,val in ((xa,ya),(xa^d,yb)):
        v=out.get(addr,0)^val
        if v: out[addr]=v
        else: out.pop(addr,None)
    return {'n':n, 'delta':d^xa^xb, 'wA':ya, 'wB':yb, 'E':out}, (xa,xb)


def _v53_rel_backward(R):
    """Exact inverse of _v53_rel_forward (no anchor supplied)."""
    n, dn = R['n'], R['delta']
    xa,xb = partner(R['wA'],n),partner(R['wB'],n)
    d=dn^xa^xb
    # Undo the two simultaneous writes in the *old* relative frame.
    pre={r^xa:v for r,v in R['E'].items()}
    for addr,val in ((0,R['wA']),(d,R['wB'])):
        v=pre.get(addr,0)^val
        if v: pre[addr]=v
        else: pre.pop(addr,None)
    return {'n':n,'delta':d,'wA':xa^pre.get(0,0),
            'wB':xb^pre.get(d,0),'E':pre}, (xa,xb)


def _v53_charge(R):
    n=R['n']
    u,K,eR,eL = _v52_carrier(n)
    if (R['delta'] not in K or R['wA'] not in K or R['wB'] not in K
          or any(k not in K or v not in K for k,v in R['E'].items())):
        return None
    value=_v52_A(R['delta'],n,K,u)^_v52_xor(R['E'].values())
    return {'charge':value,'vacuum_delta':_v52_A(value,n,K,u),
            'delta':R['delta'],'field_xor':_v52_xor(R['E'].values())}


def pair_compile_boundary_goal(pair, *, placement_class='same_carrier',
        reader_ownership='same_labels', field_condition='vacuum',
        target_where_A=None, target_delta=None, require_nonzero_h=True):
    """Resolve what is implied by a pair-vacuum goal WITHOUT executing F.

    RETURN CONTRACT: spatial_binding can be BOUND while route binding
    is UNLINKED. No clock or switch data is generated by convention.

    reader_ownership: 'same_labels' or 'swapped_labels' (interpretation
      of endpoint as a move of the original pair)
    placement_class: 'same_carrier', 'outside_carrier', or 'any'
    field_condition: 'vacuum' (required for this projection)
    target_where_A: optional absolute endpoint for *physical reader A*
    target_delta: optional required endpoint XOR separation.
    """
    snap=pair.snapshot()
    head=pair.last_hash
    base={'source':snap, 'ledger_head':head, 'live_machine_unchanged':True,
          'ledger_head_unchanged':True, 'method':'static constraint projection',
          'native_steps_executed':0, 'return_depth':None,
          'clock_binding':'UNLINKED', 'switch_tapes_generated':False}
    def fail(binding,reason,**extras):
        return {'status':'RUNS' if binding=='UNBOUND' else 'OPEN',
                'binding':binding,'missing_link':reason,**base,**extras}
    if placement_class not in ('same_carrier','outside_carrier','any'):
        return fail('UNLINKED','placement_class must be same_carrier, outside_carrier, or any')
    if reader_ownership not in ('same_labels','swapped_labels'):
        return fail('UNLINKED','reader_ownership must be same_labels or swapped_labels')
    if field_condition!='vacuum':
        return fail('UNLINKED','only explicit vacuum endpoint boundary has been compiled')
    if type(require_nonzero_h) is not bool:
        return fail('UNLINKED','require_nonzero_h must be a boolean')
    n=snap['n']; mask=(1<<n)-1
    if target_where_A is not None and (type(target_where_A) is not int or not 0<=target_where_A<=mask):
        return fail('UNLINKED','target_where_A must be a valid n-bit address')
    if target_delta is not None and (type(target_delta) is not int or not 0<=target_delta<=mask):
        return fail('UNLINKED','target_delta must be a valid n-bit word')
    if n<4 or n%2:
        return fail('UNLINKED','static affine carrier proof applies to even widths 4..16 only')
    u,K,eR,eL=_v52_carrier(n)
    R=_v53_rel_from_pair(pair)
    delta=R['delta']
    closed=(delta in K and R['wA'] in K and R['wB'] in K
            and all(k in K and v in K for k,v in R['E'].items()))
    carrier_info={'carrier_basis':{'u':u,'eR':eR,'eL':eL},
        'carrier_words':sorted(K), 'initial_carrier_coset_representative':snap['pA'],
        'carrier_closed_under_native_step':closed,
        'relative_reader_separation':delta}
    if closed:
        charge=_v53_charge(R)
        implied_delta=charge['vacuum_delta']
        carrier_info.update(charge)
    else:
        charge=None; implied_delta=None
    if closed and target_delta is not None and target_delta!=implied_delta:
        return fail('UNBOUND','conserved charge requires a different vacuum reader separation',
                    required_target_delta=target_delta, implied_vacuum_delta=implied_delta,**carrier_info)
    # A carrier-closed system cannot leave its original K-coset by internal evolution.
    if closed and placement_class=='outside_carrier':
        return fail('UNBOUND','carrier subspace is invariant; no internal trajectory can leave this coset',
                    proof='P(K) subset K; reads, writes and XOR position updates preserve K',
                    **carrier_info)
    # Physical reader A's desired endpoint fixes h differently under swapped ownership.
    provider_origin=snap['pA'] if reader_ownership=='same_labels' else snap['pB']
    h=None if target_where_A is None else provider_origin^target_where_A
    if h is not None and require_nonzero_h and h==0:
        return fail('UNBOUND','nonzero translated return requested but derived h is zero',
                    implied_h=h,**carrier_info)
    if closed and h is not None and h not in K:
        return fail('UNBOUND','target displacement lies outside the invariant carrier',
                    implied_h=h,**carrier_info)
    if closed and placement_class=='same_carrier' and h is not None and h not in K:
        return fail('UNBOUND','target endpoint is outside the requested carrier',
                    implied_h=h,**carrier_info)
    if h is not None and placement_class=='outside_carrier' and h in K:
        return fail('UNBOUND','target displacement does not change carrier coset',
                    implied_h=h,**carrier_info)
    projected_endpoint=None
    if h is not None:
        # These fields are the required TRANSLATED return, not a simulated endpoint.
        if reader_ownership=='same_labels':
            projected_endpoint={'pA':snap['pA']^h,'pB':snap['pB']^h,
                                'wA':snap['wA'],'wB':snap['wB'],'field':{}}
        else:
            projected_endpoint={'pA':snap['pB']^h,'pB':snap['pA']^h,
                                'wA':snap['wB'],'wB':snap['wA'],'field':{}}
    return {'status':'OPEN','binding':'UNLINKED',
            'spatial_binding':'BOUND' if h is not None else 'UNLINKED',
            'spatial_route':None if h is None else {'operation':'transport' if reader_ownership=='same_labels'
                        else 'swap_then_transport','h':h,
                        'way_home':{'operation':'transport' if reader_ownership=='same_labels'
                                             else 'swap_then_transport','h':h}},
            'required_endpoint':projected_endpoint,
            'required_vacuum_delta':implied_delta,
            'return_ownership':reader_ownership,
            'field_condition':field_condition,'placement_class':placement_class,
            'missing_link':'vacuum monodromy: derive return depth T and admissible coupled switches from initial relative state',
            **carrier_info,**base}


def verify_v53_relative_and_goals():
    """Separate deterministic checks against v5 paired native execution."""
    rng=_v53_random.Random(0x53B1A)
    count=0
    for n in range(2,17):
        mask=(1<<n)-1
        for _ in range(120):
            d={}
            for __ in range(rng.randrange(8)):
                k=rng.randrange(mask+1);v=rng.randrange(mask+1)
                if v:d[k]=v
            pair=PairMachine(n,rng.randrange(mask+1),rng.randrange(mask+1),
                             rng.randrange(mask+1),rng.randrange(mask+1),d)
            R=_v53_rel_from_pair(pair)
            rr,(xa,xb)=_v53_rel_forward(R)
            original=pair.snapshot(); before_head=pair.last_hash
            temp=pair.copy(); step=temp.forward()
            assert (xa,xb)==(step['xA'],step['xB'])
            assert rr==_v53_rel_from_pair(temp)
            back,(bxa,bxb)=_v53_rel_backward(rr)
            assert back==R and (bxa,bxb)==(xa,xb)
            assert pair.snapshot()==original and pair.last_hash==before_head
            count+=5
    for n in range(4,17,2):
        u,K,eR,eL=_v52_carrier(n)
        # All carrier combinations including coincident readers and populated fields.
        for _ in range(160):
            p0=rng.randrange(1<<n)
            pB=p0 ^ rng.choice(tuple(K))
            field={p0^k:rng.choice(tuple(K)) for k in rng.sample(sorted(K),rng.randrange(9))}
            pair=PairMachine(n,p0,rng.choice(tuple(K)),pB,rng.choice(tuple(K)),field)
            R=_v53_rel_from_pair(pair)
            c0=_v53_charge(R)
            assert c0 is not None
            new=pair.copy(); new.forward()
            assert _v53_charge(_v53_rel_from_pair(new))['charge']==c0['charge']
            assert _v53_charge(_v53_rel_from_pair(new))['vacuum_delta']==c0['vacuum_delta']
            assert new.pA^p0 in K and new.pB^p0 in K
            assert all(v in K and (int(k)^p0) in K for k,v in new.snapshot()['field'].items())
            count+=4
    seed=seed_pair()
    before=seed.snapshot(); head=seed.last_hash
    cross=pair_compile_boundary_goal(seed,placement_class='outside_carrier')
    assert cross['binding']=='UNBOUND' and cross['carrier_closed_under_native_step']
    valid=pair_compile_boundary_goal(seed,target_where_A=9)
    assert valid['binding']=='UNLINKED' and valid['spatial_binding']=='BOUND'
    assert valid['spatial_route']['h']==9
    assert valid['required_endpoint']=={'pA':9,'pB':6,'wA':2,'wB':11,'field':{}}
    wrong_sep=pair_compile_boundary_goal(seed,target_delta=0)
    assert wrong_sep['binding']=='UNBOUND' and wrong_sep['implied_vacuum_delta']==15
    no_placement=pair_compile_boundary_goal(seed)
    assert no_placement['binding']=='UNLINKED' and no_placement['spatial_binding']=='UNLINKED'
    assert seed.snapshot()==before and seed.last_hash==head
    count+=9
    print(f'RUNS: {count:,} independent relative/inverse/charge/goal assertions; pre-readout seal {V53_SEAL_SHA256}')
    return {'status':'RUNS','checks':count,'sealed_prediction_sha256':V53_SEAL_SHA256,
            'same_carrier_goal':valid,'cross_carrier_goal':cross}


# Register an optional read-only model tool. Re-running this cell adds no duplicates.
if 'PAIR_TOOLS' in globals() and 'TOOLS' in globals() and 'pair_model_dispatch' in globals():
    if '_PAIR_DISPATCH_BEFORE_V53' not in globals():
        _PAIR_DISPATCH_BEFORE_V53=pair_model_dispatch
    def pair_model_dispatch(name,args):
        if name=='nexus_pair_compile_boundary_goal':
            allowed={'placement_class','reader_ownership','field_condition',
                     'target_where_A'}
            if not isinstance(args,dict) or set(args)-allowed:
                return {'status':'OPEN','binding':'UNLINKED',
                        'missing_link':'invalid arguments for static pair boundary goal'}
            return pair_compile_boundary_goal(pair_machine,**args)
        return _PAIR_DISPATCH_BEFORE_V53(name,args)
    _V53_TOOL={'type':'function','function':{
        'name':'nexus_pair_compile_boundary_goal',
        'description':('READ-ONLY STATIC pair boundary goal, no ticks/no scan. '
          'Derives affine-carrier exclusion, conserved charge and required '
          'relative vacuum separation. The model MUST NOT invent target_delta; it is '
          'derived from conserved charge. Optionally binds target_where_A to '
          'the spatial XOR offset. Returns UNBOUND for impossible goals or '
          'OPEN/UNLINKED for missing switch/clock monodromy. No physical return is executed.'),
        'parameters':{'type':'object','properties':{
            'placement_class':{'type':'string','enum':['same_carrier','outside_carrier','any']},
            'reader_ownership':{'type':'string','enum':['same_labels','swapped_labels']},
            'field_condition':{'type':'string','enum':['vacuum']},
            'target_where_A':{'type':'integer','minimum':0,'maximum':65535},
            },'required':['placement_class'],'additionalProperties':False}}}
    if not any(t.get('function',{}).get('name')=='nexus_pair_compile_boundary_goal' for t in PAIR_TOOLS):
        PAIR_TOOLS.append(_V53_TOOL)
    if not any(t.get('function',{}).get('name')=='nexus_pair_compile_boundary_goal' for t in TOOLS):
        TOOLS.append(_V53_TOOL)
    if 'SYSTEM' in globals():
        _V53_PROMPT=(' Static pair-boundary requests use nexus_pair_compile_boundary_goal; '
           'it does no stepping and never derives a period from missing data. '
           'UNBOUND proves impossible under the declared carrier invariants; '
           'UNLINKED names the missing return-time and switch-tape law. ''The general model tool cannot supply target_delta or require_nonzero_h. ''native_steps_executed=0 means static analysis, not T=0.')
        if _V53_PROMPT not in SYSTEM:
            SYSTEM+=_V53_PROMPT

print('NEXUS v5.3.1 boundary goal compiler (tool-contract correction) installed; original v5.3 seal:',V53_SEAL_SHA256)

# v5.3 deterministic checks (read-only; no model needed)
_v53_result = verify_v53_relative_and_goals()
print(_v53_json.dumps({
    'same_carrier_placement': _v53_result['same_carrier_goal']['spatial_route'],
    'pair_return_route_binding': _v53_result['same_carrier_goal']['binding'],
    'cross_carrier_binding': _v53_result['cross_carrier_goal']['binding'],
    'clock_dependency': _v53_result['same_carrier_goal']['missing_link'],
}, indent=2))


# v5.3.1 regression — invoke the same dispatcher exposed to Gemma.
def verify_v531_tool_contract():
    old_state=pair_machine.snapshot()
    old_hash=pair_machine.last_hash
    schema=next(t for t in PAIR_TOOLS if t['function']['name']=='nexus_pair_compile_boundary_goal')['function']['parameters']
    assert 'target_delta' not in schema['properties']
    assert 'require_nonzero_h' not in schema['properties']
    outside=pair_model_dispatch('nexus_pair_compile_boundary_goal',
                                {'placement_class':'outside_carrier','field_condition':'vacuum'})
    inside=pair_model_dispatch('nexus_pair_compile_boundary_goal',
                               {'placement_class':'same_carrier','field_condition':'vacuum',
                                'reader_ownership':'same_labels','target_where_A':9})
    fabricated=pair_model_dispatch('nexus_pair_compile_boundary_goal',
                                   {'placement_class':'same_carrier', 'target_where_A':9, 'target_delta':0})
    assert outside['binding']=='UNBOUND', outside
    assert 'carrier subspace is invariant' in outside['missing_link'], outside
    assert inside['binding']=='UNLINKED', inside
    assert inside['spatial_binding']=='BOUND' and inside['spatial_route']['h']==9, inside
    assert inside['required_vacuum_delta']==15 and inside['return_depth'] is None, inside
    assert fabricated['binding']=='UNLINKED' and 'invalid arguments' in fabricated['missing_link'], fabricated
    assert pair_machine.snapshot()==old_state and pair_machine.last_hash==old_hash
    print('RUNS: v5.3.1 tool-contract regression; no invented delta; no native steps; live pair unchanged')
    print('Outside-carrier:',outside['binding'],outside['missing_link'])
    print('Same-carrier WHERE 9:',inside['binding'], 'spatial',inside['spatial_binding'],
          'h',inside['spatial_route']['h'],'vacuum delta',inside['required_vacuum_delta'],
          'return depth',inside['return_depth'])
    print('Fabricated target_delta:',fabricated['binding'])
    return {'outside':outside,'inside':inside,'fabricated':fabricated}

_V531_REGRESSION=verify_v531_tool_contract()


"""NEXUS v5.3.2 — proof-carrying pair-goal BIND.
Dean A. Kulik, QuHarmonics Research Group. ORCID 0009-0003-3128-8828.

Append to v5.3.1 after the original compiler definitions. Standard library only.
A model supplies a GOAL, never a depth. Pre-existing, explicitly registered
execution witnesses are revalidated by exact forward and inverse execution.
Unknown clocks remain OPEN/UNLINKED; a bounded certificate registry is not an
exhaustive classification of all possible routes.
"""
import hashlib as _v532_hashlib
import json as _v532_json
from pathlib import Path as _v532_Path

V532_MAX_VERIFY_TICKS = 100_000
V532_SCHEMA = 1

def _v532_digest(data):
    return _v532_hashlib.sha256(canonical(data).encode('utf-8')).hexdigest()

def _v532_from_snapshot(s):
    required={'n','pA','wA','pB','wB','field'}
    if type(s) is not dict or set(s)!=required or type(s['field']) is not dict:
        raise ValueError('certificate must carry a COMPLETE pair snapshot')
    if any(type(s[k]) is not int for k in ('n','pA','wA','pB','wB')):
        raise ValueError('noninteger reader state')
    if any(type(v) is not int or str(int(k))!=str(k)
           for k,v in s['field'].items()):
        raise ValueError('noncanonical field in certificate')
    p=PairMachine(s['n'],s['pA'],s['wA'],s['pB'],s['wB'],
                  {int(k):v for k,v in s['field'].items()})
    if p.snapshot()!=s:
        raise ValueError('noncanonical or incomplete snapshot')
    return p

def _v532_reverse_spatial(source, endpoint, h, reader_labels):
    """Validate an O(1) alternate way home for an EXACT complete state."""
    end=_v532_from_snapshot(endpoint)
    end.transport(h)
    if reader_labels=='swapped_labels':
        end.swap()
    return end.snapshot()==source

class PairCertificateRegistry:
    """Immutable-content records, checked at registration AND after disk reload.

    Records are evidence for one exact source and one witnessed depth.
    A record's SHA-256 detects accidental changes; it is not an authenticity
    signature, nor a pre-readout prediction seal.
    """
    def __init__(self):
        self.records={}

    def register(self, source, ticks, *, origin='independently supplied execution witness'):
        if type(ticks) is not int or not 1<=ticks<=V532_MAX_VERIFY_TICKS:
            raise ValueError('ticks outside bounded witness verifier; not a clock impossibility')
        if type(origin) is not str or not origin.strip():
            raise ValueError('explicit witness origin required')
        initial=source.snapshot()
        # Verify on a COPY. Does not alter the source or its ledger.
        res=pair_return_certificate(source,ticks,max_steps=V532_MAX_VERIFY_TICKS)
        if res.get('binding') not in ('BOUND','AMBIGUOUS') or not res.get('return_verified'):
            raise ValueError('supplied witness did not reach a nonzero translated return')
        if res['source']!=initial or not res.get('live_machine_unchanged'):
            raise AssertionError('witness verifier changed source')
        candidates=([res['route']] if res['binding']=='BOUND' else res['providers'])
        inserted=[]
        for route in candidates:
            h=route['h']; owner=route['reader_labels']
            if not _v532_reverse_spatial(initial,res['endpoint'],h,owner):
                raise AssertionError('spatial way home not exact')
            body={
                'schema':V532_SCHEMA,
                'source':initial,
                'source_sha256':_v532_digest(initial),
                'ticks':ticks,
                'endpoint':res['endpoint'],
                'h':h,
                'reader_labels':owner,
                'field_vacuum_at_endpoint':not bool(res['endpoint']['field']),
                'historical_inverse':{'operation':'inverse_pair_steps','ticks':ticks},
                'spatial_inverse':{'operation':'transport' if owner=='same_labels'
                    else 'swap_then_transport','h':h},
                'verification':'native forward, exact backward, and spatial inverse; no depth discovery',
                'origin':origin,
            }
            cert={'id':_v532_digest(body), **body}
            self.records[cert['id']]=cert
            inserted.append(cert)
        return inserted

    def verify_record(self, item):
        if type(item) is not dict or 'id' not in item:
            raise ValueError('certificate record missing id')
        body={k:v for k,v in item.items() if k!='id'}
        if _v532_digest(body)!=item['id']:
            raise ValueError('certificate content hash mismatch')
        if body.get('schema')!=V532_SCHEMA:
            raise ValueError('unsupported certificate schema')
        source=_v532_from_snapshot(body['source'])
        if _v532_digest(body['source'])!=body['source_sha256']:
            raise ValueError('certificate source fingerprint mismatch')
        ticks=body['ticks']
        if type(ticks) is not int or not 1<=ticks<=V532_MAX_VERIFY_TICKS:
            raise ValueError('out-of-bounds witness depth')
        if body.get('reader_labels') not in ('same_labels','swapped_labels'):
            raise ValueError('invalid reader ownership')
        h=body['h']
        if type(h) is not int or not 1<=h<(1<<source.n):
            raise ValueError('invalid nonzero displacement')
        r=pair_return_certificate(source,ticks,expected_h=h,max_steps=V532_MAX_VERIFY_TICKS)
        options=[r['route']] if r.get('binding')=='BOUND' else r.get('providers',[])
        if (r.get('binding') not in ('BOUND','AMBIGUOUS') or
            not r.get('return_verified') or r.get('endpoint')!=body['endpoint'] or
            not any(o['h']==h and o['reader_labels']==body['reader_labels'] for o in options) or
            not _v532_reverse_spatial(body['source'],body['endpoint'],h,body['reader_labels'])):
            raise ValueError('certificate failed native re-verification')
        if body.get('field_vacuum_at_endpoint')!= (not bool(body['endpoint']['field'])):
            raise ValueError('incorrect field-vacuum flag')
        if body.get('historical_inverse')!={'operation':'inverse_pair_steps','ticks':ticks}:
            raise ValueError('incorrect historical inverse')
        expected_spatial={'operation':'transport' if body['reader_labels']=='same_labels'
                          else 'swap_then_transport','h':h}
        if body.get('spatial_inverse')!=expected_spatial:
            raise ValueError('incorrect spatial inverse')
        return True

    def save(self, file_path):
        path=_v532_Path(file_path)
        payload={'schema':V532_SCHEMA,'certificates':list(self.records.values())}
        tmp=path.with_suffix(path.suffix+'.tmp')
        tmp.write_text(_v532_json.dumps(payload,indent=2,sort_keys=True)+'\n',encoding='utf-8')
        tmp.replace(path)
        return str(path)

    def load(self, file_path):
        doc=_v532_json.loads(_v532_Path(file_path).read_text(encoding='utf-8'))
        if set(doc)!={'schema','certificates'} or doc['schema']!=V532_SCHEMA or type(doc['certificates']) is not list:
            raise ValueError('invalid certificate repository')
        verified=[]
        for item in doc['certificates']:
            self.verify_record(item)
            verified.append(item)
        for item in verified:
            self.records[item['id']]=item
        return len(verified)

    def match(self, source):
        digest=_v532_digest(source.snapshot())
        # Matching on the COMPLETE source, not just h, n, reader delta, or carries.
        return [r for r in self.records.values()
                if r['source_sha256']==digest and r['source']==source.snapshot()]


def pair_bind_goal_with_certificates(pair, registry, *, placement_class='same_carrier',
       reader_ownership='same_labels', field_condition='vacuum', target_where_A=None):
    """Statically compile the goal, then DI-bind matching verified providers.

    No forward steps, no candidate depths, and no user-supplied ticks in this
    operation. Match is among registered verified witnesses only; this does
    not assert global uniqueness of possible trajectories.
    """
    initial=pair.snapshot(); head=pair.last_hash
    static=pair_compile_boundary_goal(pair,placement_class=placement_class,
                   reader_ownership=reader_ownership,field_condition=field_condition,
                   target_where_A=target_where_A)
    if static['source']!=initial or pair.snapshot()!=initial or pair.last_hash!=head:
        raise AssertionError('static binder mutated source')
    if static['binding']=='UNBOUND':
        return {**static,'proof_provider':'static_invariant',
                'certificate_candidates':0,'native_steps_executed':0}
    if static['binding'] not in ('UNLINKED',):
        return static
    possible=[]
    for rec in registry.match(pair):
        if rec['reader_labels']!=reader_ownership: continue
        if field_condition=='vacuum' and not rec['field_vacuum_at_endpoint']: continue
        if target_where_A is not None and rec['endpoint']['pA']!=target_where_A: continue
        if rec['h']==0: continue
        if static.get('required_endpoint') is not None:
            req=static['required_endpoint']
            if any(rec['endpoint'].get(k)!=v for k,v in req.items()): continue
        if static.get('spatial_route') is not None and rec['h']!=static['spatial_route']['h']:
            continue
        if placement_class=='same_carrier' and static.get('carrier_closed_under_native_step'):
            if rec['h'] not in set(static['carrier_words']): continue
        if placement_class=='outside_carrier' and static.get('carrier_closed_under_native_step'):
            continue
        possible.append(rec)
    common={'source':initial,'ledger_head':head,'ledger_head_unchanged':True,
            'live_machine_unchanged':pair.snapshot()==initial,
            'native_steps_executed':0,'proof_provider':'verified_certificate_registry',
            'registry_matching_source_count':len(registry.match(pair)),
            'certificate_candidates':len(possible),
            'clock_derivation':'not derived from static constraints; execution depth supplied by a pre-existing verified witness',
            'global_route_uniqueness_claimed':False}
    if not possible:
        return {**static,**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':'no registered verified execution witness for this exact source and required endpoint; vacuum clock law remains OPEN'}
    if len(possible)>1:
        return {**static,**common,'status':'OPEN','binding':'AMBIGUOUS',
                'providers':[{'certificate_id':c['id'],'ticks':c['ticks'],'h':c['h'],
                              'reader_labels':c['reader_labels'],'endpoint':c['endpoint']}
                             for c in possible],
                'missing_link':'multiple verified registered execution providers; goal does not uniquely select a route'}
    c=possible[0]
    return {**static,**common,'status':'RUNS','binding':'BOUND',
            'spatial_binding':'BOUND','clock_binding':'BOUND_BY_EXISTING_CERTIFICATE',
            'return_depth':c['ticks'],'endpoint':c['endpoint'],
            'certificate_id':c['id'],'certificate_origin':c['origin'],
            'route':{'operation':'native_pair_steps','ticks':c['ticks'],
                     'generated_displacement':c['h'],'reader_labels':c['reader_labels']},
            'way_home_alternatives':[c['spatial_inverse'],c['historical_inverse']],
            'return_by_twice_depth':2*c['ticks'],
            'twice_depth_minimality_claimed':False,
            'missing_link':None}

# Seed the registry from the exact already-documented v5 witness. This is
# independently REVERIFIED on notebook startup; it is not a clock calculation.
pair_certificate_registry=PairCertificateRegistry()
_v532_seed_registration=pair_certificate_registry.register(
    seed_pair(),806,origin='v5 documented exact pair self-move: supplied-depth witness; reverified on registration')

# A distinct tool contract: Gemma is NOT allowed to supply ticks, h, or delta.
if 'PAIR_TOOLS' in globals() and 'TOOLS' in globals() and 'pair_model_dispatch' in globals():
    if '_PAIR_DISPATCH_BEFORE_V532' not in globals():
        _PAIR_DISPATCH_BEFORE_V532=pair_model_dispatch
    def pair_model_dispatch(name,args):
        if name=='nexus_pair_bind_goal_from_certificate':
            allowed={'placement_class','reader_ownership','field_condition','target_where_A'}
            if type(args) is not dict or set(args)-allowed:
                return {'status':'OPEN','binding':'UNLINKED',
                        'missing_link':'invalid pair-goal args: no ticks, h, or target_delta permitted'}
            if 'placement_class' not in args:
                return {'status':'OPEN','binding':'UNLINKED','missing_link':'placement_class required'}
            return pair_bind_goal_with_certificates(pair_machine,pair_certificate_registry,**args)
        return _PAIR_DISPATCH_BEFORE_V532(name,args)
    _V532_TOOL={'type':'function','function':{
        'name':'nexus_pair_bind_goal_from_certificate',
        'description':('READ-ONLY certificate-aware pair GOAL binder. Supply placement class '
          'and optionally physical reader-A target WHERE, ownership and vacuum condition. '
          'NO ticks, displacement h, or reader delta. Statically projects endpoint and '
          'resolves exact source against REVERIFIED previously established execution '
          'witnesses; does not derive unknown clocks. Returns UNBOUND on invariant '
          'contradiction, BOUND on one registered provider, AMBIGUOUS on multiple, '
          'UNLINKED when none owns the execution route. The 806-tick witness is '
          'available only for its exact original seed state.'),
        'parameters':{'type':'object','properties':{
          'placement_class':{'type':'string','enum':['same_carrier','outside_carrier','any']},
          'reader_ownership':{'type':'string','enum':['same_labels','swapped_labels']},
          'field_condition':{'type':'string','enum':['vacuum']},
          'target_where_A':{'type':'integer','minimum':0,'maximum':65535},
        },'required':['placement_class'],'additionalProperties':False}}}
    for _tools in (PAIR_TOOLS,TOOLS):
        if not any(z.get('function',{}).get('name')=='nexus_pair_bind_goal_from_certificate' for z in _tools):
            _tools.append(_V532_TOOL)
    if 'SYSTEM' in globals():
        _V532_GUIDE=(' For a pair goal that might have an existing verified execution witness, '
          'call nexus_pair_bind_goal_from_certificate without ticks or h. BOUND means '
          'a single registered exact-source certificate, NOT unique over all possible '
          'histories, and NOT a derivation of return depth from boundary constraints. '
          'way_home_alternatives are independent inverse choices, not consecutive steps. '
          'UNLINKED requires a missing certified route or clock law. ') 
        if _V532_GUIDE not in SYSTEM:
            SYSTEM+=_V532_GUIDE


def verify_v532():
    import tempfile as _tempfile
    seed=seed_pair(); before=seed.snapshot(); head=seed.last_hash
    bind=lambda **kw:pair_bind_goal_with_certificates(seed,pair_certificate_registry,**kw)
    yes=bind(placement_class='same_carrier',target_where_A=9)
    assert yes['binding']=='BOUND' and yes['return_depth']==806 and yes['route']['generated_displacement']==9,yes
    assert yes['endpoint']=={'n':4,'pA':9,'pB':6,'wA':2,'wB':11,'field':{}},yes
    assert len(yes['way_home_alternatives'])==2 and yes['global_route_uniqueness_claimed'] is False
    assert yes['native_steps_executed']==0 and yes['clock_binding']=='BOUND_BY_EXISTING_CERTIFICATE'
    no_where=bind(placement_class='same_carrier')
    assert no_where['binding']=='BOUND' and no_where['endpoint']['pA']==9
    cross=bind(placement_class='outside_carrier')
    assert cross['binding']=='UNBOUND' and cross['proof_provider']=='static_invariant'
    missing=bind(placement_class='same_carrier',target_where_A=6)
    assert missing['binding']=='UNLINKED' and missing['return_depth'] is None
    mismatch=PairMachine(4,0,2,15,11,{0:1})
    other=pair_bind_goal_with_certificates(mismatch,pair_certificate_registry,placement_class='same_carrier',target_where_A=9)
    assert other['binding']=='UNLINKED' and other['registry_matching_source_count']==0
    assert seed.snapshot()==before and seed.last_hash==head
    # Two VALID depths for the same endpoint => registered-provider ambiguity,
    # not false assertion of universal shortest/unique return.
    reg=PairCertificateRegistry()
    reg.register(seed,806,origin='documented translated half return')
    reg.register(seed,2418,origin='same endpoint after a further witnessed 1612-step cycle')
    amb=pair_bind_goal_with_certificates(seed,reg,placement_class='same_carrier',target_where_A=9)
    assert amb['binding']=='AMBIGUOUS' and amb['certificate_candidates']==2,amb
    # Roundtrip persisted certificate records, reverify all steps on reload.
    with _tempfile.TemporaryDirectory() as td:
        target=_v532_Path(td)/'registry.json'
        reg.save(target)
        recovered=PairCertificateRegistry()
        assert recovered.load(target)==2
        assert pair_bind_goal_with_certificates(seed,recovered,placement_class='same_carrier',target_where_A=9)['binding']=='AMBIGUOUS'
        doc=_v532_json.loads(target.read_text());doc['certificates'][0]['ticks']=807
        target.write_text(_v532_json.dumps(doc))
        try:
            PairCertificateRegistry().load(target)
        except ValueError:
            pass
        else:
            raise AssertionError('modified certificate incorrectly accepted')
    # Same tool dispatcher available to Gemma.
    req={'placement_class':'same_carrier','target_where_A':9}
    output=pair_model_dispatch('nexus_pair_bind_goal_from_certificate',req)
    expected_match=(pair_machine.snapshot()==seed.snapshot())
    assert output['binding']==('BOUND' if expected_match else 'UNLINKED'),output
    rejected=pair_model_dispatch('nexus_pair_bind_goal_from_certificate',{'placement_class':'same_carrier','ticks':806})
    assert rejected['binding']=='UNLINKED'
    print('RUNS: v5.3.2 certificate-aware BIND checks (source fingerprint, inverse, outside carrier, missing provider, ambiguous depths, disk reload/tamper rejection, model tool).')
    print('Goal WHERE 9:',yes['binding'],'depth',yes['return_depth'],'h',yes['route']['generated_displacement'])
    print('Outside carrier:',cross['binding'],'different source:',other['binding'],'unknown endpoint:',missing['binding'])
    print('Two valid witnesses to one endpoint:',amb['binding'],'providers',amb['certificate_candidates'])
    print('Current live pair exact seed:',expected_match,'tool result:',output['binding'])
    return {'where_9':yes,'outside_carrier':cross,'source_mismatch':other,
            'unknown_endpoint':missing,'ambiguous_depths':amb}

_V532_RESULT=verify_v532()
print('NEXUS v5.3.2 certificate-aware pair-goal BIND ready; no LM Studio call was made.')


"""NEXUS AI v5.3.3 -- exact symmetry lift of verified pair-return certificates.

Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828
Append this to v5.3.2. Standard-library only. This file does not walk
or search any native orbit in the goal-binder path.

The only providers are already-verified certificates. Algebraic theorem:
F^t(S)=H_h J^e(S) and [F,H_a]=[F,J]=[H_a,J]=0 imply
F^t(H_a J^b S)=H_h J^e(H_a J^b S).
"""

def _v533_machine_from_snapshot(snapshot):
    return _v532_from_snapshot(snapshot)


def _v533_apply_symmetry(snapshot, offset=0, swap=False):
    """Transform a COMPLETE snapshot via H_offset J^swap, without mutation."""
    obj = _v533_machine_from_snapshot(snapshot)
    if swap:
        obj.swap()
    obj.transport(offset)
    return obj.snapshot()


def _v533_source_provider(record, target_source):
    """At most two algebraic address solutions, never address enumeration.

    A candidate offset is fixed by pA alone for each of the two label modes.
    Full source equality is then required (includes both carries and all cells).
    """
    base = record['source']
    if base['n'] != target_source['n']:
        return []
    matches = []
    for swap in (False, True):
        anchor = base['pB'] if swap else base['pA']
        offset = anchor ^ target_source['pA']
        if _v533_apply_symmetry(base, offset, swap) == target_source:
            matches.append({'source_translation': offset,
                            'source_reader_swap': swap})
    return matches


def _v533_lifted_candidates(source, registry):
    """Lift every VERIFIED registry certificate to the query's exact source.

    No forward steps here; checked provider is a certificate reverified when
    registered or loaded. Preserve all genuinely distinct route witnesses.
    """
    snap = source.snapshot()
    out = []
    seen = set()
    for c in registry.records.values():
        for sym in _v533_source_provider(c, snap):
            end = _v533_apply_symmetry(c['endpoint'],
                      sym['source_translation'], sym['source_reader_swap'])
            # Recheck the claimed spatial inverse directly on complete state.
            if not _v532_reverse_spatial(snap, end, c['h'], c['reader_labels']):
                raise AssertionError('symmetry-lifted certificate has no exact way home')
            key = (c['id'], c['ticks'], c['reader_labels'], c['h'],
                   canonical(end))
            if key in seen:
                continue  # stabilizer ambiguity is not a second execution route
            seen.add(key)
            out.append({'certificate': c, 'symmetry': sym, 'endpoint': end})
    return out


def pair_bind_goal_by_symmetry(pair, registry, *,
        placement_class='same_carrier', reader_ownership='same_labels',
        field_condition='vacuum', target_where_A=None):
    """Bind a goal by invariants and symmetry-lifted execution certificates.

    BOUND denotes one registered route, not universal route uniqueness.
    No ticks, transport offset, or relative separation is supplied to binder.
    """
    before = pair.snapshot()
    head = pair.last_hash
    static = pair_compile_boundary_goal(
        pair, placement_class=placement_class,
        reader_ownership=reader_ownership, field_condition=field_condition,
        target_where_A=target_where_A)
    if pair.snapshot() != before or pair.last_hash != head:
        raise AssertionError('static goal compilation mutated the live machine')
    if static['binding'] == 'UNBOUND':
        return {**static, 'proof_provider': 'static_invariant',
                'certificate_candidates': 0,
                'native_steps_executed': 0}
    if static['binding'] != 'UNLINKED':
        return static

    lifted = _v533_lifted_candidates(pair, registry)
    valid = []
    for candidate in lifted:
        c = candidate['certificate']
        end = candidate['endpoint']
        if c['reader_labels'] != reader_ownership:
            continue
        if field_condition == 'vacuum' and end['field']:
            continue
        if target_where_A is not None and end['pA'] != target_where_A:
            continue
        if static.get('required_endpoint') is not None:
            if any(end.get(k) != v
                   for k, v in static['required_endpoint'].items()):
                continue
        if static.get('spatial_route') is not None:
            if c['h'] != static['spatial_route']['h']:
                continue
        if placement_class == 'same_carrier' and static.get('carrier_closed_under_native_step'):
            if c['h'] not in static['carrier_words']:
                continue
        if placement_class == 'outside_carrier' and static.get('carrier_closed_under_native_step'):
            continue
        valid.append(candidate)

    common = {'source': before, 'ledger_head': head,
              'ledger_head_unchanged': pair.last_hash == head,
              'live_machine_unchanged': pair.snapshot() == before,
              'native_steps_executed': 0,
              'proof_provider': 'symmetry_lifted_verified_certificate',
              'registry_liftable_sources': len(lifted),
              'certificate_candidates': len(valid),
              'clock_derivation': 'not derived from static initial constraints; '
                       'inherited from a separately verified source certificate',
              'global_route_uniqueness_claimed': False,
              'cross_carrier_release_claimed': False}

    if not valid:
        return {**static, **common, 'status': 'OPEN', 'binding': 'UNLINKED',
                'missing_link': 'no verified registry certificate is symmetry-equivalent '
                                'to this COMPLETE source and requested endpoint; '
                                'unknown vacuum clock law remains OPEN'}
    if len(valid) > 1:
        return {**static, **common, 'status': 'OPEN', 'binding': 'AMBIGUOUS',
                'providers': [{'certificate_id': x['certificate']['id'],
                               'ticks': x['certificate']['ticks'],
                               'h': x['certificate']['h'],
                               'reader_labels': x['certificate']['reader_labels'],
                               'endpoint': x['endpoint'],
                               'source_symmetry': x['symmetry']}
                              for x in valid],
                'missing_link': 'more than one registered execution depth/provider '
                                'satisfies the goal; owner needs qualification'}
    x = valid[0]
    c, end, sym = x['certificate'], x['endpoint'], x['symmetry']
    return {**static, **common,
            'status': 'RUNS', 'binding': 'BOUND',
            'spatial_binding': 'BOUND',
            'clock_binding': 'BOUND_BY_SYMMETRY_LIFTED_CERTIFICATE',
            'return_depth': c['ticks'],
            'endpoint': end,
            'certificate_id': c['id'],
            'certificate_origin': c['origin'],
            'source_symmetry': sym,
            'route': {'operation': 'native_pair_steps',
                      'ticks': c['ticks'],
                      'generated_displacement': c['h'],
                      'reader_labels': c['reader_labels']},
            'way_home_alternatives': [c['spatial_inverse'], c['historical_inverse']],
            'return_by_twice_depth': 2*c['ticks'],
            'twice_depth_minimality_claimed': False,
            'missing_link': None}


# Public model tool: same goal contract as v5.3.2 but allows symmetry-equivalent
# live sources. No arbitrary source snapshots or candidate offsets from Gemma.
if 'pair_model_dispatch' in globals() and 'PAIR_TOOLS' in globals() and 'TOOLS' in globals():
    if '_PAIR_DISPATCH_BEFORE_V533' not in globals():
        _PAIR_DISPATCH_BEFORE_V533 = pair_model_dispatch
    def pair_model_dispatch(name, args):
        if name == 'nexus_pair_bind_goal_by_symmetry':
            allowed = {'placement_class','reader_ownership','field_condition',
                       'target_where_A'}
            if type(args) is not dict or set(args) - allowed:
                return {'status': 'OPEN', 'binding': 'UNLINKED',
                        'missing_link': 'invalid pair-goal input; ticks, h, '
                                        'target_delta and source snapshots are forbidden'}
            if 'placement_class' not in args:
                return {'status': 'OPEN', 'binding': 'UNLINKED',
                        'missing_link': 'placement_class is required'}
            return pair_bind_goal_by_symmetry(
                pair_machine, pair_certificate_registry, **args)
        return _PAIR_DISPATCH_BEFORE_V533(name, args)
    _V533_TOOL = {'type': 'function', 'function': {
        'name': 'nexus_pair_bind_goal_by_symmetry',
        'description': 'READ-ONLY paired goal BIND using static constraints and '
          'verified certificate symmetry under absolute XOR transport and '
          'reader-label exchange. Does not walk an unknown orbit, or derive '
          'a new clock. Supply only the goal, never depth, h, reader delta, '
          'or source snapshot. BOUND means a single REGISTERED proof provider; '
          'UNBOUND means invariant contradiction; UNLINKED missing provider; '
          'AMBIGUOUS multiple registered route providers.',
        'parameters': {'type': 'object', 'properties': {
            'placement_class': {'type':'string',
                'enum':['same_carrier','outside_carrier','any']},
            'reader_ownership': {'type':'string',
                'enum':['same_labels','swapped_labels']},
            'field_condition': {'type':'string', 'enum':['vacuum']},
            'target_where_A': {'type':'integer','minimum':0,'maximum':65535}
        }, 'required':['placement_class'], 'additionalProperties':False}}}
    for collection in (PAIR_TOOLS, TOOLS):
        if not any(z.get('function',{}).get('name') ==
                   'nexus_pair_bind_goal_by_symmetry' for z in collection):
            collection.append(_V533_TOOL)
    if 'SYSTEM' in globals():
        _V533_GUIDE = (' For a paired vacuum-return goal with existing '
          'symmetry-equivalent evidence, call nexus_pair_bind_goal_by_symmetry. '
          'No ticks, h, delta or candidate source required. The route depth '
          'comes from an existing reverified certificate and is not a new '
          'clock derivation. The same carrier invariant still prohibits '
          'internal cross-coset release. Alternative ways home are alternatives.')
        if _V533_GUIDE not in SYSTEM:
            SYSTEM += _V533_GUIDE


def verify_v533():
    """Executable independent checks. No test populates the registry by searching."""
    import random as _r
    rng = _r.Random(5332026)
    start = seed_pair()
    before = start.snapshot()
    checks = 0
    # Verify commuting conjugacy using the actual native engine.
    for n in range(2, 17):
        mask = (1 << n) - 1
        for _ in range(80):
            field = {}
            for __ in range(rng.randrange(6)):
                k, v = rng.randrange(mask+1), rng.randrange(mask+1)
                if v: field[k] = v
            p = PairMachine(n, rng.randrange(mask+1), rng.randrange(mask+1),
                            rng.randrange(mask+1), rng.randrange(mask+1), field)
            snap = p.snapshot()
            offset, sw = rng.randrange(mask+1), bool(rng.getrandbits(1))
            a = p.copy()
            if sw: a.swap()
            a.transport(offset)
            a.forward()
            b = p.copy()
            b.forward()
            if sw: b.swap()
            b.transport(offset)
            assert a.snapshot() == b.snapshot()
            assert p.snapshot() == snap
            checks += 2

    # Lift ONE original certificate into all 32 symmetry images at width 4,
    # including a distinct carrier coset. The lift itself runs NO steps.
    for offset in range(16):
        for sw in (False,True):
            q = seed_pair()
            if sw: q.swap()
            q.transport(offset)
            snap, head = q.snapshot(), q.last_hash
            target = q.pA ^ 9
            r = pair_bind_goal_by_symmetry(q, pair_certificate_registry,
                       placement_class='same_carrier', target_where_A=target)
            assert r['binding'] == 'BOUND', (offset,sw,r)
            assert r['return_depth']==806 and r['route']['generated_displacement']==9
            assert r['endpoint']['pA'] == target
            assert r['source_symmetry'] in (
              {'source_translation':offset,'source_reader_swap':sw},
              # Source stabilizers can provide an equivalent address description.
              {'source_translation':offset ^ 15,'source_reader_swap':not sw})
            assert r['native_steps_executed'] == 0
            assert r['global_route_uniqueness_claimed'] is False
            assert r['cross_carrier_release_claimed'] is False
            assert q.snapshot()==snap and q.last_hash==head
            # Independent native endpoint check (NOT part of binding).
            if offset in (0,1,7,15):
                witness = q.copy()
                for _ in range(806): witness.forward()
                assert witness.snapshot() == r['endpoint']
                checks += 1
            checks += 8
    shifted = seed_pair(); shifted.transport(1); shifted.swap()
    assert shifted.pA == 14 and shifted.pB == 1
    lifted = pair_bind_goal_by_symmetry(shifted,pair_certificate_registry,
                placement_class='same_carrier',target_where_A=7)
    assert lifted['binding']=='BOUND' and lifted['endpoint']['pA']==7
    assert lifted['endpoint']['pB']==8
    checks += 3
    cross = pair_bind_goal_by_symmetry(shifted,pair_certificate_registry,
                                     placement_class='outside_carrier')
    assert cross['binding']=='UNBOUND'
    unknown = pair_bind_goal_by_symmetry(shifted,pair_certificate_registry,
                         placement_class='same_carrier',target_where_A=5)
    assert unknown['binding']=='UNLINKED'
    modified = seed_pair(); modified.wA ^= 1
    incorrect = pair_bind_goal_by_symmetry(modified,pair_certificate_registry,
                         placement_class='same_carrier',target_where_A=9)
    assert incorrect['binding']=='UNLINKED'
    # Extra bogus source field also must not match original certificate.
    modified = PairMachine(4,0,2,15,11,{0:1})
    incorrect_field = pair_bind_goal_by_symmetry(modified,pair_certificate_registry,
                         placement_class='same_carrier',target_where_A=9)
    assert incorrect_field['binding']=='UNLINKED'
    checks += 4
    # Multiple depths remain distinct providers even after symmetry lifting.
    multi = PairCertificateRegistry()
    multi.register(seed_pair(),806,origin='first documented return')
    multi.register(seed_pair(),2418,origin='later repeated return')
    q = seed_pair(); q.transport(1); q.swap()
    ambiguous = pair_bind_goal_by_symmetry(q,multi,
                         placement_class='same_carrier',target_where_A=7)
    assert ambiguous['binding']=='AMBIGUOUS' and ambiguous['certificate_candidates']==2
    checks += 2
    assert before==start.snapshot()
    assert pair_model_dispatch('nexus_pair_bind_goal_by_symmetry',
      {'placement_class':'same_carrier','ticks':806})['binding']=='UNLINKED'
    assert pair_model_dispatch('nexus_pair_bind_goal_by_symmetry',
      {'placement_class':'same_carrier','target_where_A':9})['binding'] == (
       'BOUND' if pair_machine.snapshot()==before else 'UNLINKED')
    checks += 3
    print(f'RUNS: {checks:,} v5.3.3 symmetry, complete-source, inverse, and tool-contract checks')
    print('Original seed WHERE 9: BOUND via existing certificate, depth 806')
    print('Translated and swapped source, WHERE 7: BOUND via H1 + swap certificate lift')
    print('New source outside original carrier coset: yes; internal cross-carrier release: no')
    print('Absent endpoint: UNLINKED; forbidden release: UNBOUND; two witnesses: AMBIGUOUS')
    return {'checks':checks,'lifted':lifted,'cross':cross,
            'missing':unknown,'ambiguous':ambiguous}

_V533_RESULTS = verify_v533()
print('NEXUS v5.3.3 symmetry-lifted certificate compiler ready. No LM Studio call made.')

# ============================================================================
# NEXUS AI v5.4 UNIFIED — public goal gateway and capability/ownership audit
# Maintains the complete v2 → v5.3.3 source and execution contracts above.
# The new gateway only binds existing verified providers: no period scan.
# ============================================================================

V54_VERSION = '5.4-unified'
V54_STATUS = {
    'native_step_inverse': 'RUNS',
    'single_reader_lazy_frame': 'RUNS',
    'single_reader_goal_bind': 'RUNS',
    'paired_forward_inverse': 'RUNS',
    'first_vacuum_event_execution': 'RUNS',
    'switch_tape_reconstruction': 'RUNS_on_admissible_tapes',
    'pair_relative_state_and_carrier_invariant': 'RUNS',
    'registered_certificate_reverification': 'RUNS',
    'symmetry_lifted_certificate_bind': 'RUNS',
    'unknown_clock_from_boundary_without_executing': 'OPEN',
    'internal_cross_carrier_release_of_closed_seed': 'UNBOUND_by_carrier_invariant',
}

V54_LEGACY_TOOL_NAMES = (
    'nexus_state', 'nexus_step', 'nexus_reverse', 'nexus_transport',
    'nexus_commute', 'nexus_bind', 'nexus_bind_predecessor',
    'nexus_bind_predecessor_of', 'nexus_bind_endpoint',
    'nexus_pair_state', 'nexus_pair_step', 'nexus_pair_reverse',
    'nexus_pair_bind_return', 'nexus_pair_bind_first_vacuum',
    'nexus_pair_switch_holonomy', 'nexus_pair_compile_boundary_goal',
    'nexus_pair_bind_goal_from_certificate',
    'nexus_pair_bind_goal_by_symmetry',
)


def nexus_pair_capabilities():
    """Read-only grounded inventory; nothing is inferred from model output."""
    return {
        'status': 'RUNS',
        'version': V54_VERSION,
        'single_reader': machine.snapshot(),
        'paired_readers': pair_machine.snapshot(),
        'single_ledger_head': machine.last_hash,
        'pair_ledger_head': pair_machine.last_hash,
        'registered_certificate_count': len(pair_certificate_registry.records),
        'registered_providers': [
            {'id': c['id'], 'ticks': c['ticks'], 'h': c['h'],
             'reader_labels': c['reader_labels'], 'origin': c['origin'],
             'source_sha256': c['source_sha256']}
            for c in sorted(pair_certificate_registry.records.values(),
                            key=lambda c: c['id'])
        ],
        'capability_ledger': dict(V54_STATUS),
        'available_tools': sorted({x['function']['name'] for x in TOOLS}),
        'clock_derived_from_initial_constraints': False,
        'live_machine_unchanged': True,
        'ledger_head_unchanged': True,
        'missing_link': 'unseen vacuum monodromy / cyclic switches and their depth'
    }


def nexus_pair_goal(*, placement_class, reader_ownership='same_labels',
                    field_condition='vacuum', target_where_A=None):
    """Pure, single public gateway: static constraints then certificate provider.

    Supply a requirement, never a period. BOUND means exactly one registered
    applicable execution provider, not globally unique possible histories.
    No candidate t, offset h, target_delta, or source-field default accepted.
    """
    kwargs=dict(placement_class=placement_class,
                reader_ownership=reader_ownership,
                field_condition=field_condition,
                target_where_A=target_where_A)
    start=pair_machine.snapshot()
    head=pair_machine.last_hash
    result=pair_bind_goal_by_symmetry(pair_machine,
                          pair_certificate_registry, **kwargs)
    if pair_machine.snapshot()!=start or pair_machine.last_hash!=head:
        raise AssertionError('read-only unified goal modified the live pair')
    return {
        **result,
        'compiler_gateway': 'v5.4 goal → static invariant → symmetry-lifted proof provider',
        'unknown_clock_derivation_claimed': False,
        'way_home_semantics': 'alternatives, not sequential operations',
    }

_V54_PREVIOUS_PAIR_DISPATCH = pair_model_dispatch
_V54_ALLOWED_GOAL_ARGS = {'placement_class','reader_ownership',
                          'field_condition','target_where_A'}

def pair_model_dispatch(name, args):
    """Preserve all v2–v5.3.3 dispatchers without changing their contracts."""
    if name == 'nexus_pair_capabilities':
        if type(args) is not dict or args:
            return {'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'nexus_pair_capabilities takes exactly {}'}
        return nexus_pair_capabilities()
    if name == 'nexus_pair_goal':
        if type(args) is not dict:
            return {'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'goal arguments must be a JSON object'}
        missing={'placement_class'}-set(args)
        unexpected=set(args)-_V54_ALLOWED_GOAL_ARGS
        if missing or unexpected:
            return {'status':'OPEN','binding':'UNLINKED',
                    'missing_fields':sorted(missing),
                    'unexpected_fields':sorted(unexpected),
                    'missing_link':'only a typed goal is accepted; ticks, h, '
                                   'target_delta, and candidate source are forbidden'}
        if args['placement_class'] not in ('same_carrier','outside_carrier','any'):
            return {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid placement_class'}
        if args.get('reader_ownership','same_labels') not in ('same_labels','swapped_labels'):
            return {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid reader_ownership'}
        if args.get('field_condition','vacuum')!='vacuum':
            return {'status':'OPEN','binding':'UNLINKED','missing_link':'only vacuum boundary is implemented'}
        t=args.get('target_where_A')
        if t is not None and (type(t) is not int or not 0<=t<(1<<pair_machine.n)):
            return {'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'target_where_A must be a valid n-bit address'}
        return nexus_pair_goal(**args)
    return _V54_PREVIOUS_PAIR_DISPATCH(name,args)

_V54_TOOLS = [
    {'type':'function', 'function':{
        'name':'nexus_pair_goal',
        'description': 'Preferred read-only PAIR GOAL BIND. Supply an exact placement '
          'goal, NOT ticks, h, target_delta, or a source snapshot. Compiler first '
          'applies conserved/carrier constraints then binds symmetry-lifted '
          'REGISTERED execution certificates. No unknown orbit scan or clock '
          'derivation. BOUND means a single matching registered provider; '
          'AMBIGUOUS multiple providers; UNLINKED a missing route; UNBOUND '
          'an invariant contradiction. Return inverse alternatives, not '
          'operations to execute sequentially.',
        'parameters':{'type':'object', 'properties':{
            'placement_class':{'type':'string','enum':['same_carrier','outside_carrier','any']},
            'reader_ownership':{'type':'string','enum':['same_labels','swapped_labels']},
            'field_condition':{'type':'string','enum':['vacuum']},
            'target_where_A':{'type':'integer','minimum':0,'maximum':65535},
        }, 'required':['placement_class'], 'additionalProperties':False}
    }},
    {'type':'function','function':{
        'name':'nexus_pair_capabilities',
        'description':'READ ONLY inventory of exact single/pair states, ledgers, '
          'registered certified routes, model tools, and OPEN versus RUNS links. '
          'No arguments and no model-assumed defaults.',
        'parameters':{'type':'object','properties':{},'additionalProperties':False}
    }}
]
for catalog in (PAIR_TOOLS, TOOLS):
    for tool in _V54_TOOLS:
        name=tool['function']['name']
        if not any(z['function']['name']==name for z in catalog):
            catalog.append(tool)

V54_GUIDE=(' The preferred general pair-goal interface is nexus_pair_goal; '
   'supply only the goal, never ticks, transport offset h, relative delta, '
   'or source snapshot. Use nexus_pair_capabilities to see the compiler ledger. '
   'A bound route has a verified provider, not a newly derived clock. '
   'Two ways home are alternatives and never chained. OPEN means missing '
   'route law; no missing readout may be defaulted.')
if V54_GUIDE not in SYSTEM:
    SYSTEM += V54_GUIDE


def verify_v54_unified():
    """New interface checks; all predecessor version checks run above."""
    pair_before=pair_machine.snapshot()
    pair_head=pair_machine.last_hash
    single_before=machine.snapshot()
    single_head=machine.last_hash
    names=[z['function']['name'] for z in TOOLS]
    assert len(names)==len(set(names)), 'duplicated model tool names'
    assert set(V54_LEGACY_TOOL_NAMES).issubset(names)
    assert {'nexus_pair_goal','nexus_pair_capabilities'}.issubset(names)
    checks=3
    caps=dispatch_model_tool(machine,'nexus_pair_capabilities',{})
    assert caps['version']==V54_VERSION and caps['status']=='RUNS'
    assert not caps['clock_derived_from_initial_constraints']
    assert caps['registered_certificate_count']>=1
    assert caps['live_machine_unchanged'] and caps['ledger_head_unchanged']
    checks+=4
    assert dispatch_model_tool(machine,'nexus_pair_capabilities',{'ticks':806})['binding']=='UNLINKED'
    assert dispatch_model_tool(machine,'nexus_pair_goal',{'ticks':806})['binding']=='UNLINKED'
    assert dispatch_model_tool(machine,'nexus_pair_goal',{
        'placement_class':'same_carrier','target_delta':0})['binding']=='UNLINKED'
    assert dispatch_model_tool(machine,'nexus_pair_goal',{
        'placement_class':'same_carrier','target_where_A':True})['binding']=='UNLINKED'
    assert dispatch_model_tool(machine,'nexus_pair_goal',{
        'placement_class':'same_carrier','target_where_A':1<<pair_machine.n})['binding']=='UNLINKED'
    checks+=5
    before=pair_machine.copy()
    # Validate model dispatch against whatever live paired state is replayed.
    res=dispatch_model_tool(machine,'nexus_pair_goal',{
        'placement_class':'outside_carrier'})
    # Carrier-invariant contradiction holds only for carrier-closed sources.
    if pair_compile_boundary_goal(before,placement_class='outside_carrier')['binding']=='UNBOUND':
        assert res['binding']=='UNBOUND'
    else:
        assert res['binding'] in ('UNLINKED','BOUND','AMBIGUOUS')
    checks+=1
    # Direct goal-to-certificate verification on original seed is separate
    # from active live pair, so no precondition about user ledger is needed.
    original_live=pair_machine
    try:
        globals()['pair_machine']=seed_pair()
        r=dispatch_model_tool(machine,'nexus_pair_goal',{
            'placement_class':'same_carrier','target_where_A':9})
        assert r['binding']=='BOUND' and r['return_depth']==806
        assert r['route']['generated_displacement']==9
        assert r['way_home_semantics']=='alternatives, not sequential operations'
        missing=dispatch_model_tool(machine,'nexus_pair_goal',{
            'placement_class':'same_carrier','target_where_A':6})
        assert missing['binding']=='UNLINKED'
        other=dispatch_model_tool(machine,'nexus_pair_goal',{
            'placement_class':'outside_carrier'})
        assert other['binding']=='UNBOUND'
        assert pair_machine.snapshot()==seed_pair().snapshot()
        checks+=6
        lift=seed_pair();lift.transport(1);lift.swap()
        globals()['pair_machine']=lift
        r2=dispatch_model_tool(machine,'nexus_pair_goal',{
            'placement_class':'same_carrier','target_where_A':7})
        assert r2['binding']=='BOUND' and r2['return_depth']==806
        assert r2['native_steps_executed']==0
        checks+=2
    finally:
        globals()['pair_machine']=original_live
    assert pair_machine.snapshot()==pair_before and pair_machine.last_hash==pair_head
    assert machine.snapshot()==single_before and machine.last_hash==single_head
    checks+=2
    print(f'RUNS: {checks:,} v5.4 unified tool-contract and read-only checks')
    print('Tool catalog:',len(TOOLS),'unique tools;',
          len(pair_certificate_registry.records),'registered certificate(s)')
    print('Goal WHERE 9: existing 806-step proof provider (not a new clock calculation)')
    print('OPEN: unknown vacuum-monodromy depth directly from initial boundary')
    return {'status':'RUNS','checks':checks,'tools':len(TOOLS),
            'source_protected':True,'clock_derivation':'OPEN'}

V54_VERIFY=verify_v54_unified()
print('NEXUS AI v5.4 UNIFIED READY — no LM Studio call made.')


# ============================================================================
# NEXUS AI v5.5 — PROOF COMPOSITION FROM A SYMMETRY-LIFTED CERTIFICATE
# Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828
#
# Exact theorem, for this complete source S:
#    F^t(S) = H_h J^e(S),   FH_h=H_hF,   FJ=JF,  H_h J=J H_h
# implies for every integer k>=0:
#    F^(kt)(S) = (H_h J^e)^k(S)
# and for k even this is S, since H_h^2=J^2=I.
# No general clock derivation or global periodicity is claimed.
# Registry witnesses were verified at registration or load; composition never
# advances an unknown orbit. Proof selection is a DI provider lookup.
# ============================================================================

V55_VERSION = '5.5-proof-composition'
V55_OPEN = ('UNKNOWN CLOCK: derive first vacuum monodromy and cyclic switches '
            'from the initial coupled boundary without executing an orbit or '
            'using a registered depth')


def v55_proof_power(pair, lifted, count):
    """Apply a verified relative-return certificate symbolically (O(1) in count).

    For a supplied exact complete state and an already qualified registry
    provider, this is a route theorem, not a fresh execution certificate.
    """
    if type(count) is not int or count < 0:
        raise ValueError('composition count must be a nonnegative integer')
    source = pair.snapshot()
    record = lifted['certificate']
    # The registry is trusted only after content-hash and source checks.
    if _v532_digest({k:v for k,v in record.items() if k!='id'}) != record['id']:
        raise ValueError('provider content hash mismatch')
    if record.get('verification') != ('native forward, exact backward, '
                                     'and spatial inverse; no depth discovery'):
        raise ValueError('provider does not carry expected verification contract')
    if record['source_sha256'] != _v532_digest(record['source']):
        raise ValueError('provider source fingerprint mismatch')
    sym = lifted['symmetry']
    offset, swap = sym['source_translation'], sym['source_reader_swap']
    if _v533_apply_symmetry(record['source'],offset,swap) != source:
        raise ValueError('provider not equivalent to COMPLETE current source')
    g_h = record['h']
    g_swap = record['reader_labels'] == 'swapped_labels'
    expected_first = _v533_apply_symmetry(source,g_h,g_swap)
    if expected_first != lifted['endpoint']:
        raise ValueError('lifted endpoint is not the certified symmetry image')
    endpoint = (_v533_apply_symmetry(source, g_h, g_swap)
                if count % 2 else source)
    # Exact instance-level return: no universal shortest-period claim.
    fixed_at_first = (expected_first == source)
    return {
        'status':'RUNS', 'binding':'BOUND',
        'proof_kind':'symmetry_group_composition_on_complete_source',
        'source':source,'endpoint':endpoint,
        'base_certificate_id':record['id'],
        'base_depth':record['ticks'],
        'composition_count':count,
        'composed_depth':count*record['ticks'],
        'base_symmetry':{'operation':'transport_and_optional_swap',
                         'h':g_h, 'reader_swap':g_swap},
        'composed_symmetry':{'operation':'transport_and_optional_swap',
                             'h':g_h if count%2 else 0,
                             'reader_swap':g_swap if count%2 else False},
        'first_witness_already_fixed_source':fixed_at_first,
        'full_state_return':endpoint==source,
        'way_home':{
            'historical':{'operation':'inverse_pair_steps',
                          'ticks':count*record['ticks']},
            'endpoint_spatial':{'operation':'transport_and_optional_swap',
                                'h':g_h if count%2 else 0,
                                'reader_swap':g_swap if count%2 else False},
            'semantics':'ALTERNATIVES, not successive operations',
        },
        'native_steps_executed':0,
        'return_depth_derived_from_unknown_initial_boundary':False,
        'global_minimality_claimed':False,
        'missing_link':V55_OPEN,
    }


def v55_compile_exact_return(pair, registry, *, placement_class='same_carrier',
                             reader_ownership='same_labels',
                             field_condition='vacuum',target_where_A=None):
    """A full-state return goal resolved by certificate algebra; NO tick input."""
    source = pair.snapshot()
    head = pair.last_hash
    common={'source':source, 'native_steps_executed':0,
            'ledger_head':head,'ledger_head_unchanged':True,
            'live_machine_unchanged':True,
            'clock_derived_from_initial_constraints':False,
            'minimal_period_claimed':False}
    if placement_class == 'outside_carrier':
        return {**common,'status':'RUNS','binding':'UNBOUND',
                'reason':'exact source-state return cannot simultaneously end outside its initial carrier'}
    if placement_class not in ('same_carrier','any'):
        return {**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':'invalid placement class'}
    if reader_ownership != 'same_labels':
        return {**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':'exact-state goal requires same final labeled state; '
                               'a distinct swapped-label goal needs its own qualifier'}
    if field_condition!='vacuum':
        return {**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':'non-vacuum return goal interface not implemented'}
    if source['field']:
        return {**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':'current exact-state goal requires a vacuum source; '
                               'a populated-field return is not disproven'}
    if target_where_A is not None and target_where_A != source['pA']:
        return {**common,'status':'RUNS','binding':'UNBOUND',
                'reason':'exact state return requires original reader A WHERE'}
    lifted=_v533_lifted_candidates(pair, registry)
    usable=[]
    for option in lifted:
        rec=option['certificate']
        # Certified vacuum at both ends required to bind the vacuum goal.
        if option['endpoint']['field']:
            continue
        if rec['ticks']<=0:
            continue
        first=v55_proof_power(pair, option,1)
        # A symmetry fixed point may already be a full-state return at t.
        repeats=1 if first['full_state_return'] else 2
        full=v55_proof_power(pair, option,repeats)
        if full['endpoint'] != source:
            raise AssertionError('twofold symmetry composition did not close')
        usable.append(full)
    if pair.snapshot()!=source or pair.last_hash!=head:
        raise AssertionError('proof resolver modified live state')
    if not usable:
        return {**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':'no certified complete-source return provider; '+V55_OPEN,
                'registry_lifted_candidates':len(lifted)}
    # Do not select an apparently shorter candidate: multiple providers
    # require ownership qualification, even if they land on the same state.
    if len(usable)>1:
        return {**common,'status':'OPEN','binding':'AMBIGUOUS',
                'provider_count':len(usable),
                'providers':[{'certificate_id':x['base_certificate_id'],
                              'composed_depth':x['composed_depth'],
                              'composition_count':x['composition_count']}
                             for x in usable],
                'missing_link':'multiple registered proof providers; no global '
                               'minimal-depth or unique-clock claim'}
    p=usable[0]
    return {**common,**p,
            'status':'RUNS','binding':'BOUND',
            'route':{'operation':'compose_verified_pair_return',
                     'certificate_id':p['base_certificate_id'],
                     'repeat':p['composition_count'],
                     'ticks':p['composed_depth']},
            'clock_owner':'existing verified execution certificate, lifted and composed',
            'depth_not_obtained_by_orbit_scan_in_binder':True}


# Upgrade the single preferred v5.4 tool, not another public LLM tool.
_V55_PREVIOUS_GATEWAY = nexus_pair_goal
_V55_PREVIOUS_DISPATCH = pair_model_dispatch


def nexus_pair_goal(*, placement_class='same_carrier',
                    reader_ownership='same_labels', field_condition='vacuum',
                    target_where_A=None, return_kind='translated_vacuum'):
    if return_kind == 'exact_state':
        return v55_compile_exact_return(
            pair_machine, pair_certificate_registry,
            placement_class=placement_class,
            reader_ownership=reader_ownership,
            field_condition=field_condition,
            target_where_A=target_where_A)
    if return_kind == 'translated_vacuum':
        return _V55_PREVIOUS_GATEWAY(
            placement_class=placement_class,
            reader_ownership=reader_ownership,
            field_condition=field_condition,
            target_where_A=target_where_A)
    return {'status':'OPEN','binding':'UNLINKED',
            'missing_link':'return_kind must be translated_vacuum or exact_state'}


def pair_model_dispatch(name,args):
    if name != 'nexus_pair_goal':
        return _V55_PREVIOUS_DISPATCH(name,args)
    allowed={'placement_class','reader_ownership','field_condition',
             'target_where_A','return_kind'}
    if type(args) is not dict or set(args)-allowed:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'pair goal accepts only typed goal constraints, '
                               'not ticks, h, target_delta or source snapshots'}
    if args.get('placement_class','same_carrier') not in ('same_carrier','outside_carrier','any'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid placement_class'}
    if args.get('reader_ownership','same_labels') not in ('same_labels','swapped_labels'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid reader_ownership'}
    if args.get('field_condition','vacuum')!='vacuum':
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'only vacuum field goals supported'}
    t=args.get('target_where_A')
    if t is not None and (type(t) is not int or not 0<=t<(1<<pair_machine.n)):
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'target_where_A must be a valid n-bit address'}
    if args.get('return_kind','translated_vacuum') not in ('translated_vacuum','exact_state'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid return_kind'}
    return nexus_pair_goal(**args)


for _v55_tools in (TOOLS,PAIR_TOOLS):
    _pair_goal_tool=next(t for t in _v55_tools
                         if t['function']['name']=='nexus_pair_goal')
    _pair_goal_tool['function']['parameters']['properties']['return_kind']={
        'type':'string','enum':['translated_vacuum','exact_state']}
    _pair_goal_tool['function']['parameters']['required']=[]
    _pair_goal_tool['function']['description']=(
        'PREFERRED read-only NEXUS pair goal. return_kind=translated_vacuum '
        'binds a symmetry-lifted existing witness for a requested WHERE. '
        'return_kind=exact_state derives a full source-state return by '
        'COMPOSING the symmetry action of an existing certified relative '
        'return, without executing an orbit. Omit ticks, h, target_delta, '
        'and source snapshots. No unseen first-vacuum clock derivation; '
        'BOUND means one applicable verified proof provider, UNLINKED '
        'missing owner, AMBIGUOUS multiple owners, UNBOUND contradictory goal.')

_V55_GUIDE=(' In v5.5 nexus_pair_goal accepts return_kind=translated_vacuum '
    'or exact_state. For exact complete-source return set return_kind=exact_state; '
    'do not supply ticks. The compiler composes certified spatial symmetries '
    'and returns a derived exact-state depth, not an unknown minimal period. '
    'Do not conflate proof composition with vacuum clock derivation.')
if _V55_GUIDE not in SYSTEM:
    SYSTEM += _V55_GUIDE

# Wrap the inventory without modifying previous capability ledger.
_V55_PREVIOUS_CAPABILITIES = nexus_pair_capabilities

def nexus_pair_capabilities():
    old = _V55_PREVIOUS_CAPABILITIES()
    old['version']=V55_VERSION
    old['capability_ledger']['certified_route_group_composition']='RUNS'
    old['capability_ledger']['unknown_clock_from_initial_boundary']='OPEN'
    old['compiler_gateway']='nexus_pair_goal(return_kind=exact_state|translated_vacuum)'
    return old


def verify_v55_proof_composition():
    """Independent native cross-checks occur ONLY in tests, never in BIND."""
    count=0
    seed=seed_pair()
    source=seed.snapshot()
    source_head=seed.last_hash
    reg=pair_certificate_registry
    options=_v533_lifted_candidates(seed,reg)
    assert len(options)==1,options
    first=v55_proof_power(seed,options[0],1)
    assert first['composed_depth']==806 and first['endpoint']=={
        'n':4,'pA':9,'wA':2,'pB':6,'wB':11,'field':{}}
    exact=v55_compile_exact_return(seed,reg)
    assert exact['binding']=='BOUND' and exact['composed_depth']==1612,exact
    assert exact['composition_count']==2 and exact['endpoint']==source
    assert exact['native_steps_executed']==0 and exact['clock_owner'].startswith('existing')
    count+=10
    # Run the actual machine only in the independent check, with no use in binding.
    for width in (4,6,8):
        k=(1<<width)-1
        eR=1<<(width//2-1)
        eL=1<<(width//2)
        start=PairMachine(width,0,eR,k,k^eL)  # width-native carrier seed
        cert=PairCertificateRegistry()
        cert.register(start,806,origin=f'v5.5 independent witness width {width}')
        for shift,sw in ((0,False),(1,False),(3,True)):
            q=start.copy(); q.transport(shift)
            if sw: q.swap()
            lifted=_v533_lifted_candidates(q,cert)
            assert len(lifted)==1
            for copies in (0,1,2,3,4):
                predicted=v55_proof_power(q,lifted[0],copies)
                actual=q.copy()
                for _ in range(806*copies):
                    actual.forward()
                assert predicted['endpoint']==actual.snapshot(),(width,shift,sw,copies)
                assert predicted['native_steps_executed']==0
                count+=2
    # Symbolic power is constant-work in the repetition count.
    huge=v55_proof_power(seed, options[0], 10**30+1)
    assert huge['composed_depth']==806*(10**30+1)
    assert huge['endpoint']==first['endpoint']
    count+=2
    # Provider miss: no implicit default/field substitution.
    bad=PairMachine(4,0,2,15,10,{0:1})
    assert v55_compile_exact_return(bad,reg)['binding']=='UNLINKED'
    assert pair_model_dispatch('nexus_pair_goal',
                              {'return_kind':'exact_state'})['binding']==(
         'BOUND' if pair_machine.snapshot()==source else 'UNLINKED')
    assert pair_model_dispatch('nexus_pair_goal',
                              {'return_kind':'exact_state','ticks':806})['binding']=='UNLINKED'
    assert pair_model_dispatch('nexus_pair_goal',
                              {'return_kind':'exact_state','target_where_A':9})['binding']=='UNBOUND'
    assert pair_model_dispatch('nexus_pair_goal',
                              {'placement_class':'same_carrier','target_where_A':9})['binding']=='BOUND'
    assert pair_model_dispatch('nexus_pair_goal',
                              {'return_kind':'exact_state','placement_class':'outside_carrier'})['binding']=='UNBOUND'
    count+=5
    # Two separately verified route providers cannot silently be collapsed.
    two=PairCertificateRegistry()
    two.register(seed,806,origin='independent translated return provider')
    two.register(seed,2418,origin='independent repeated translated return provider')
    assert v55_compile_exact_return(seed,two)['binding']=='AMBIGUOUS'
    count+=1
    assert seed.snapshot()==source and seed.last_hash==source_head
    print(f'RUNS: {count} v5.5 symbolic-composition and independent native checks')
    print('Exact-state return derived from registered 806-step witness: '
          '1612 steps, endpoint original source, native steps in binder: 0')
    print('OPEN: unknown first-vacuum depth law (not a stored-certificate route)')
    return {'checks':count,'exact_return':exact,'huge_repeat_depth':huge['composed_depth']}

_V55_RESULT=verify_v55_proof_composition()
print('NEXUS AI v5.5 PROOF COMPOSITION READY — no LM Studio call made.')

# NEXUS AI v5.6 — finite 64-seed carrier index; exact-provider first.
# Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828
# This extension is designed to be appended AFTER the full v5.5 compiler.
# It never enumerates a new candidate depth. Reported depths are provenance
# only until the native v5.5 witness verifier establishes the complete route.
from collections import Counter as _v56_Counter

V56_VERSION = '5.6-carrier-certificate-index'
V56_LABELS = ('0','eR','eL','s','u','u+eR','u+eL','u+s')
V56_SEED_READOUTS = (
    ('0', '0', 40057082),
    ('0', 'eR', 2),
    ('0', 's', 46536554),
    ('eR', 'eR', 11375),
    ('eR', 'eL', 155413184),
    ('eL', 'eR', 6733),
    ('eL', 'eL', 155413184),
    ('eL', 's', 155413184),
    ('eL', 'u', 155413184),
    ('eL', 'u+eL', 155413184),
    ('eL', 'u+s', 155413184),
    ('s', '0', 3069701),
    ('s', 's', 87391602),
    ('s', 'u', 87391602),
    ('s', 'u+eL', 3123),
    ('s', 'u+s', 87391602),
    ('u', '0', 23246730),
    ('u', 's', 145769810),
    ('u', 'u', 23246730),
    ('u', 'u+eL', 806),
    ('u', 'u+s', 145769810),
    ('u+eR', '0', 155413184),
    ('u+eR', 'eR', 11375),
    ('u+eR', 'u', 155413184),
    ('u+eR', 'u+eL', 155413184),
    ('u+eR', 'u+s', 155413184),
    ('u+eL', 'eR', 11375),
    ('u+s', '0', 48051878),
    ('u+s', 's', 48051878),
    ('u+s', 'u', 91135750),
    ('u+s', 'u+eR', 975),
    ('u+s', 'u+s', 91135750),
)
V56_BOUND = 100_000 # Existing native witness verifier's actual limit.


def v56_carrier_word(label, width=4):
    if label not in V56_LABELS:
        raise ValueError('carrier label must be one of '+str(V56_LABELS))
    if type(width) is not int or width < 4 or width > 16 or width % 2:
        raise ValueError('canonical carrier needs even bit width 4..16')
    code=V56_LABELS.index(label)
    eR=1<<(width//2-1)
    eL=1<<(width//2)
    u=(1<<width)-1
    return ((eR if code&1 else 0) ^
            (eL if code&2 else 0) ^
            (u if code&4 else 0))


def v56_seed(where_B, carry_B, width=4):
    # The canonical source C pairing has reader A=(WHERE=0,carry=eR)
    return PairMachine(width,0,v56_carrier_word('eR',width),
                       v56_carrier_word(where_B,width),
                       v56_carrier_word(carry_B,width))


def v56_build_carrier_index(width=4):
    if width != 4:
        raise ValueError('the 32 supplied C readouts were obtained at canonical width 4; reverify other widths separately')
    '''Populate all 64 canonical *index slots*. Verify only 8 already-reported
    short witnesses. Long readouts are UNLINKED, not fabricated certificates.
    Never mutate the original global v5.5 registry or either live machine.
    '''
    records={(b,c):t for b,c,t in V56_SEED_READOUTS}
    assert len(records)==len(V56_SEED_READOUTS)==32
    registry=PairCertificateRegistry()
    table={}
    for b in V56_LABELS:
        for c in V56_LABELS:
            pair=v56_seed(b,c,width)
            state=pair.snapshot()
            charge=_v53_charge(_v53_rel_from_pair(pair))
            assert charge is not None
            reported=records.get((b,c))
            state_hash=_v532_digest(state)
            item={
                'where_B':b,'carry_B':c,'source_sha256':state_hash,
                'source':state,
                'relative_charge':charge['charge'],
                'vacuum_delta':charge['vacuum_delta'],
                'reported_self_move_tick':reported,
                'readout_provenance':('NEXUS_Self_Move_output.txt; '
                     'prior executable-C output, not a pre-readout seal') if reported else None,
                'certificate_ids':[],
                'binding':'UNLINKED',
                'status':'OPEN',
                'missing_link':'No verified return provider for this complete source',
            }
            if reported is not None and reported <= V56_BOUND:
                # This is verification of an ALREADY-SUPPLIED depth, not a
                # clock derived before data or a search for candidate depths.
                bound=registry.register(pair,reported,
                   origin=('v5.6 independent native recheck of previously '
                           'reported C readout; depth was supplied'))
                assert bound
                item['certificate_ids']=[x['id'] for x in bound]
                item['binding']='BOUND' if len(bound)==1 else 'AMBIGUOUS'
                item['status']='RUNS'
                item['missing_link']=None
            elif reported is not None:
                item['missing_link']=(
                    'Previously reported self-move at depth '+str(reported)+
                    '; native registry verifier is bounded to '+str(V56_BOUND)+
                    '. Readout UNLINKED as a certificate; no default filled.')
            table[(b,c)]=item
    assert len(table)==64
    return registry,table


V56_REGISTRY,V56_CARRIER_INDEX=v56_build_carrier_index()


def v56_catalog(include_rows=False):
    states=list(V56_CARRIER_INDEX.values())
    out={
       'status':'RUNS','index':'complete_canonical_64_seed_manifest',
       'width':4,'indexed_seeds':len(states),
       'reported_self_moves':sum(x['reported_self_move_tick'] is not None for x in states),
       'native_verified_seeds':sum(bool(x['certificate_ids']) for x in states),
       'reported_but_unverified':sum(x['reported_self_move_tick'] is not None and not x['certificate_ids'] for x in states),
       'no_supplied_self_move_readout':sum(x['reported_self_move_tick'] is None for x in states),
       'registered_certificate_records':len(V56_REGISTRY.records),
       'clock_from_initial_boundary':'OPEN',
       'unresolved_seed_binding':'UNLINKED',
       'cross_carrier_release_of_canonical_closed_seed':'UNBOUND',
       'all_64_proved':False,
       'lookups':'constant-size hash-index lookup; does not derive previously unknown clocks',
       'live_machine_unchanged':True,
    }
    if include_rows:
        out['rows']=[{'where_B':r['where_B'],'carry_B':r['carry_B'],
                      'binding':r['binding'],
                      'reported_tick':r['reported_self_move_tick'],
                      'verified_provider_count':len(r['certificate_ids'])}
                     for r in states]
    return out


def v56_carrier_goal(where_B,carry_B,*,target_where_A=None,
                     placement_class='same_carrier',return_kind='translated_vacuum',
                     reader_ownership='any'):
    '''Resolve exactly one of the 64 canonical seed goals from existing certificates.
    No simulation, candidate-depth scan, or insertion of missing readouts.
    Either returns a complete exact provider or names missing ownership.
    '''
    base=V56_CARRIER_INDEX.get((where_B,carry_B))
    if base is None:
        return {'status':'OPEN','binding':'UNLINKED',
                'missing_link':'canonical B WHERE/CARRY label missing or invalid'}
    if placement_class not in ('same_carrier','outside_carrier'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'unsupported placement class'}
    if return_kind not in ('translated_vacuum','exact_state'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'unsupported return_kind'}
    if reader_ownership not in ('any','same_labels','swapped_labels'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'unsupported reader ownership'}
    source=base['source']
    before_id=base['source_sha256']
    common={'source':source,'source_sha256':before_id,
            'native_steps_executed':0,'live_machine_unchanged':True,
            'ledger_head_unchanged':True,
            'witness_policy':'only SHA-256-checked native-reverified readouts',
            'depth_derived_from_new_initial_boundary':False}
    pair=v56_seed(where_B,carry_B)
    assert pair.snapshot()==source
    K={v56_carrier_word(v) for v in V56_LABELS}
    if placement_class=='outside_carrier':
        return {**common,'status':'RUNS','binding':'UNBOUND',
                'reason':'canonical empty-field seed is closed in the invariant 8-word carrier'}
    if target_where_A is not None:
        if type(target_where_A) is not int or not 0<=target_where_A<16:
            return {**common,'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'target_where_A must be a 4-bit address'}
        if target_where_A not in K:
            return {**common,'status':'RUNS','binding':'UNBOUND',
                    'reason':'requested WHERE is outside the invariant source carrier'}
    if return_kind=='exact_state':
        if reader_ownership=='swapped_labels':
            return {**common,'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'exact labeled-state goal incompatible with required swapped ownership'}
        return v55_compile_exact_return(
            pair,V56_REGISTRY,placement_class='same_carrier',
            reader_ownership='same_labels',field_condition='vacuum',
            target_where_A=target_where_A)
    provider_options=[]
    for item in _v533_lifted_candidates(pair,V56_REGISTRY):
        c=item['certificate']
        end=item['endpoint']
        if end['field']:
            continue
        if target_where_A is not None and end['pA']!=target_where_A:
            continue
        if reader_ownership!='any' and c['reader_labels']!=reader_ownership:
            continue
        if not _v532_reverse_spatial(source,end,c['h'],c['reader_labels']):
            raise AssertionError('registered candidate missing exact spatial inverse')
        provider_options.append({
            'certificate_id':c['id'],'depth':c['ticks'],
            'h':c['h'],'reader_labels':c['reader_labels'],
            'endpoint':end,
            'way_home':{'spatial':c['spatial_inverse'],
                        'historical':c['historical_inverse'],
                        'note':'independent ALTERNATIVES, never successive actions'},
            'proof':'registered native forward/backward and complete-state spatial return',
        })
    if not provider_options:
        return {**common,'status':'OPEN','binding':'UNLINKED',
                'reported_tick_not_proof':base['reported_self_move_tick'],
                'missing_link':base['missing_link'] or
                    'no registered provider satisfies the requested endpoint or owner'}
    result={**common,'status':'RUNS' if len(provider_options)==1 else 'OPEN',
            'binding':'BOUND' if len(provider_options)==1 else 'AMBIGUOUS',
            'provider_count':len(provider_options),
            'providers':provider_options}
    if len(provider_options)==1:
        result['route']=provider_options[0]
    else:
        result['missing_link']='multiple legitimate providers; owner qualifier required'
    return result


# New access tools only; preserve v5.5 existing entry point and tool contracts.
_V56_PREVIOUS_PAIR_DISPATCH=pair_model_dispatch

def pair_model_dispatch(name,args):
    if name=='nexus_pair_carrier_catalog':
        if type(args) is not dict or args:
            return {'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'catalog tool takes no arguments'}
        return v56_catalog()
    if name=='nexus_pair_carrier_goal':
        if type(args) is not dict:
            return {'status':'OPEN','binding':'UNLINKED','missing_link':'expected object'}
        allowed={'where_B','carry_B','target_where_A','placement_class',
                 'return_kind','reader_ownership'}
        if (set(args)-allowed or not {'where_B','carry_B'}.issubset(args) or
            type(args.get('where_B')) is not str or
            type(args.get('carry_B')) is not str):
            return {'status':'OPEN','binding':'UNLINKED',
                    'missing_link':'where_B and carry_B required; no depth or h accepted'}
        return v56_carrier_goal(**args)
    return _V56_PREVIOUS_PAIR_DISPATCH(name,args)

V56_TOOLS=[
   {'type':'function','function':{
      'name':'nexus_pair_carrier_catalog',
      'description':('Read-only inventory of all 64 canonical coupled pair seeds. '
         'Only short supplied readouts native-reverified are certificates. '
         'UNLINKED is not UNBOUND; no unknown first-vacuum clock derived.'),
      'parameters':{'type':'object','properties':{},'additionalProperties':False}}},
   {'type':'function','function':{
      'name':'nexus_pair_carrier_goal',
      'description':('Read-only BIND for canonical 8-word carrier seed where '
         'reader A=(0,eR) and B=(where_B,carry_B). The compiler finds '
         'previously verified routes via the 64-slot certificate index. '
         'Do not supply ticks, h, target_delta, or fabricated readouts.'),
      'parameters':{'type':'object','properties':{
         'where_B':{'type':'string','enum':list(V56_LABELS)},
         'carry_B':{'type':'string','enum':list(V56_LABELS)},
         'target_where_A':{'type':'integer','minimum':0,'maximum':15},
         'return_kind':{'type':'string','enum':['translated_vacuum','exact_state']},
         'placement_class':{'type':'string','enum':['same_carrier','outside_carrier']},
         'reader_ownership':{'type':'string','enum':['any','same_labels','swapped_labels']}},
      'required':['where_B','carry_B'],'additionalProperties':False}}}
]
for _v56_schema in V56_TOOLS:
    TOOLS.append(_v56_schema)
    PAIR_TOOLS.append(_v56_schema)
SYSTEM += (' v5.6 offers nexus_pair_carrier_catalog for the 64 canonical '
   'seed index and nexus_pair_carrier_goal for a specific canonical B source. '
   'A complete index is not 64 certified routes. For unresolved seeds report '
   'UNLINKED, and never infer a clock from absent readouts.')


def verify_v56_carrier_index():
    # No orbit search. Exactly the eight pre-existing short depths are supplied
    # once to the native verifier. This test only checks registry and interface.
    assert len(V56_CARRIER_INDEX)==64
    assert len(V56_SEED_READOUTS)==32
    counts=v56_catalog()
    assert counts['native_verified_seeds']==8,counts
    assert counts['reported_but_unverified']==24,counts
    assert counts['no_supplied_self_move_readout']==32,counts
    assert all(x['source_sha256']==_v532_digest(x['source']) for x in V56_CARRIER_INDEX.values())
    assert V56_CARRIER_INDEX[('u','u+eL')]['reported_self_move_tick']==806
    q=v56_carrier_goal('u','u+eL',target_where_A=9,
                       reader_ownership='same_labels')
    assert q['binding']=='BOUND' and q['route']['depth']==806 and q['route']['h']==9,q
    full=v56_carrier_goal('u','u+eL',return_kind='exact_state')
    assert full['binding']=='BOUND' and full['composed_depth']==1612,full
    unknown=v56_carrier_goal('eR','eL')
    assert unknown['binding']=='UNLINKED' and unknown['status']=='OPEN',unknown
    long=v56_carrier_goal('eR','eL')
    assert long['binding']=='UNLINKED',long
    contradiction=v56_carrier_goal('u','u+eL',target_where_A=1)
    assert contradiction['binding']=='UNBOUND',contradiction
    cross=v56_carrier_goal('u','u+eL',placement_class='outside_carrier')
    assert cross['binding']=='UNBOUND',cross
    via_tool=pair_model_dispatch('nexus_pair_carrier_goal',{
        'where_B':'u','carry_B':'u+eL','target_where_A':9,
        'reader_ownership':'same_labels'})
    assert via_tool['binding']=='BOUND',via_tool
    assert dispatch_model_tool(machine,'nexus_pair_carrier_goal',{'where_B':'u',
        'carry_B':'u+eL','target_where_A':9,'reader_ownership':'same_labels'})['binding']=='BOUND'
    rejected=pair_model_dispatch('nexus_pair_carrier_goal',{
        'where_B':'u','carry_B':'u+eL','ticks':806})
    assert rejected['binding']=='UNLINKED',rejected
    # The canonical 64-index itself cannot produce a new physics deconfinement rule.
    assert pair_machine.snapshot()==seed_pair().snapshot()
    print('RUNS: v5.6 canonical 64 seed index; eight prior short readouts native-reverified.')
    print('RUNS: original seed WHERE 9 BOUND at known 806 depth; full 1612 by composition.')
    print('OPEN: 24 long-depth reports and 32 other seeds have no installed verified provider.')
    print('OPEN: first-vacuum time law from initial boundary and cross-shell port mechanism.')
    return counts

_V56_PREVIOUS_CAPABILITIES=nexus_pair_capabilities

def nexus_pair_capabilities():
    old=_V56_PREVIOUS_CAPABILITIES()
    old['version']=V56_VERSION
    old['carrier_seed_catalog']=v56_catalog()
    old['capability_ledger']['64_canonical_seed_slots_indexed']='RUNS'
    old['capability_ledger']['64_native_verified_return_clocks']='OPEN'
    return old

V56_REPORT=verify_v56_carrier_index()
print('NEXUS AI v5.6 CARRIER INDEX READY; LM Studio not contacted.')


"""NEXUS AI v5.7 — carrier Klein-four symmetry and generalized vacuum certificates.
Dean A. Kulik · QuHarmonics Research Group · ORCID 0009-0003-3128-8828

DROP-IN: execute after the COMPLETE v5.6 initialization cell.
Python standard library only; no LM Studio calls. Original event ledgers untouched.

Scope: the eight-word K carrier and its translated cosets, with BOTH carries
and every stored field value in K and all reader positions/field addresses in
ONE carrier coset. General populated unrestricted n-bit states are NOT covered.
Witness times are linked from supplied data and reverified; never newly derived.
"""
import hashlib as _v57_hashlib
import random as _v57_random

V57_VERSION = '5.7-carrier-klein-four'
V57_PHI = ('I', 'J', 'A', 'A0')
V57_OPEN = ('Unknown first-vacuum depth from an unseen initial boundary; '
            'unregistered orbit has no clock provider. Cross-coset internal release '
            'remains forbidden for an isolated closed K carrier.')


def v57_basis(n):
    if type(n) is not int or not (4 <= n <= 16 and n % 2 == 0):
        raise ValueError('native carrier construction needs even width 4..16')
    eR, eL = 1 << (n // 2 - 1), 1 << (n // 2)
    u = (1 << n) - 1
    return eR, eL, u, eR ^ eL


def v57_K(n):
    er, el, u, _ = v57_basis(n)
    return frozenset((er if i & 1 else 0) ^
                     (el if i & 2 else 0) ^
                     (u if i & 4 else 0) for i in range(8))


def v57_phi_word(word, n, phi='I'):
    """Linear automorphism on actual native K words (not v5.6 label indices)."""
    if phi not in V57_PHI or type(word) is not int or word not in v57_K(n):
        raise ValueError('phi is only defined on native eight-word carrier K')
    er, el, u, s = v57_basis(n)
    l0 = bool(word & er) ^ bool(word & el)
    toggle = {'I': 0, 'J': s, 'A': u ^ s, 'A0': u}[phi]
    return word ^ (toggle if l0 else 0)


def v57_K_projection(word,n):
    """XOR-linear projection onto K: complement has bits 0,eR,eL cleared.

    Coordinates use eR,eL,u as K basis and the bit-0 as u's pivot.
    Unlike a minimum-coset representative, this projection is linear and
    makes Phi(H_g S) = H_{Phi(g)} Phi(S) exact for every n-bit translation.
    """
    er,el,u,_=v57_basis(n)
    if type(word) is not int or not 0<=word<(1<<n):
        raise ValueError('word outside native width')
    c=word&1
    rest=word^(u if c else 0)
    return (u if c else 0)^(er if rest&er else 0)^(el if rest&el else 0)


def v57_phi_address(word,n,phi='I'):
    """Canonical linear extension of phi to all n-bit WHERE addresses."""
    k=v57_K_projection(word,n)
    return word^k^v57_phi_word(k,n,phi)


def v57_qualify(snapshot):
    """Phi is valid on all K-valued carries/field values; addresses arbitrary.

    Different reader cosets remain distinct under native K-valued execution.
    The symmetry acts on complete native addresses through a fixed linear
    complement rather than a per-coset arbitrary representative.
    """
    try:
        n=snapshot['n'];K=v57_K(n)
        if snapshot['wA'] not in K or snapshot['wB'] not in K:
            return None
        if any(v not in K for v in snapshot['field'].values()):
            return None
        if not all(type(snapshot[q]) is int and 0<=snapshot[q]<(1<<n)
                   for q in ('pA','pB')):
            return None
        return True
    except (ValueError, KeyError, TypeError):
        return None


def v57_phi_snapshot(snapshot,phi='I'):
    """Phi on native addresses and K-valued carried/kept words."""
    if phi not in V57_PHI:
        raise ValueError('unknown carrier automorphism')
    if phi=='I':
        return _v532_from_snapshot(snapshot).snapshot()
    if v57_qualify(snapshot) is None:
        raise ValueError('UNLINKED: K-valued carry/field prerequisite absent')
    n=snapshot['n']
    p=lambda z:v57_phi_address(z,n,phi)
    D={p(int(k)):v57_phi_word(v,n,phi) for k,v in snapshot['field'].items()}
    return PairMachine(n,p(snapshot['pA']),v57_phi_word(snapshot['wA'],n,phi),
                       p(snapshot['pB']),v57_phi_word(snapshot['wB'],n,phi),D).snapshot()


_V57_PREV_APPLY = _v533_apply_symmetry

def _v533_apply_symmetry(snapshot, offset=0, swap=False, phi='I'):
    """Backward compatible H_offset J^swap, extended by scoped Phi_phi."""
    if phi == 'I':
        return _V57_PREV_APPLY(snapshot, offset, swap)
    base = v57_phi_snapshot(snapshot, phi)
    return _V57_PREV_APPLY(base, offset, swap)


def _v533_source_provider(record, target_source):
    """Derive source translation from pA; no scanning offset candidates."""
    base = record['source']
    if base['n'] != target_source['n']:
        return []
    options = []
    for phi in V57_PHI:
        if phi != 'I' and v57_qualify(base) is None:
            continue
        for swap in (False, True):
            mapped = _v533_apply_symmetry(base, 0, swap, phi)
            offset = mapped['pA'] ^ target_source['pA']
            if _v533_apply_symmetry(base, offset, swap, phi) == target_source:
                options.append({'source_translation':offset,
                                'source_reader_swap':swap,
                                'source_phi':phi})
    return options


def _v533_lifted_candidates(source, registry):
    """Phi-conjugated h; distinct algebraic descriptions aren't extra clocks."""
    snap = source.snapshot()
    out = {}
    for rec in registry.records.values():
        for sym in _v533_source_provider(rec, snap):
            phi = sym['source_phi']
            h = rec['h']
            if phi != 'I':
                if h not in v57_K(rec['source']['n']):
                    continue  # no unsupported extrapolation of carrier Phi
                h = v57_phi_word(h, rec['source']['n'], phi)
            end = _v533_apply_symmetry(rec['endpoint'],
                sym['source_translation'], sym['source_reader_swap'], phi)
            if not _v532_reverse_spatial(snap, end, h, rec['reader_labels']):
                raise AssertionError('lifted complete-state spatial inverse failed')
            key=(rec['id'], rec['ticks'], canonical(end), h, rec['reader_labels'])
            if key in out:
                out[key]['symmetry_owners'].append(sym)
            else:
                out[key]={'certificate':rec,'symmetry':sym,
                          'symmetry_owners':[sym], 'endpoint':end,
                          'h_lifted':h}
    return list(out.values())


def v55_proof_power(pair, lifted, count):
    """Existing proof composition corrected for the conjugated offset phi(h)."""
    if type(count) is not int or count < 0:
        raise ValueError('composition count must be a nonnegative integer')
    source = pair.snapshot()
    record = lifted['certificate']
    if _v532_digest({k:v for k,v in record.items() if k!='id'}) != record['id']:
        raise ValueError('certificate hash mismatch')
    if record.get('verification') != ('native forward, exact backward, '
                                     'and spatial inverse; no depth discovery'):
        raise ValueError('unknown verification contract')
    if record['source_sha256'] != _v532_digest(record['source']):
        raise ValueError('source fingerprint mismatch')
    sym = lifted['symmetry']
    if _v533_apply_symmetry(record['source'],sym['source_translation'],
                           sym['source_reader_swap'],sym.get('source_phi','I')) != source:
        raise ValueError('certificate not equivalent to complete source')
    h = lifted.get('h_lifted',record['h'])
    exchange = record['reader_labels']=='swapped_labels'
    first = _v533_apply_symmetry(source,h,exchange)
    if first!=lifted['endpoint']:
        raise ValueError('conjugated endpoint failed the certificate relation')
    target = first if count % 2 else source
    return {'status':'RUNS','binding':'BOUND','source':source,'endpoint':target,
            'base_certificate_id':record['id'],'base_depth':record['ticks'],
            'composition_count':count,'composed_depth':count*record['ticks'],
            'base_symmetry':{'operation':'transport_and_optional_swap',
                             'h':h,'reader_swap':exchange,
                             'conjugating_phi':sym.get('source_phi','I')},
            'composed_symmetry':{'operation':'transport_and_optional_swap',
                                 'h':h if count%2 else 0,
                                 'reader_swap':exchange if count%2 else False},
            'full_state_return':target==source,
            'way_home':{'historical':{'operation':'inverse_pair_steps',
                                      'ticks':count*record['ticks']},
                        'endpoint_spatial':{'operation':'transport_and_optional_swap',
                                            'h':h if count%2 else 0,
                                            'reader_swap':exchange if count%2 else False},
                        'semantics':'ALTERNATIVES; never successive actions'},
            'native_steps_executed':0,
            'return_depth_derived_from_unknown_initial_boundary':False,
            'global_minimality_claimed':False,'missing_link':V57_OPEN}


# The v5.4 goal binder must also use the CONJUGATED h, not the stored base h.
def pair_bind_goal_by_symmetry(pair, registry, *,
        placement_class='same_carrier', reader_ownership='same_labels',
        field_condition='vacuum', target_where_A=None):
    before=pair.snapshot(); head=pair.last_hash
    static=pair_compile_boundary_goal(
        pair,placement_class=placement_class,reader_ownership=reader_ownership,
        field_condition=field_condition,target_where_A=target_where_A)
    assert pair.snapshot()==before and pair.last_hash==head
    if static['binding']=='UNBOUND':
        return {**static,'proof_provider':'static_invariant',
                'native_steps_executed':0,'certificate_candidates':0}
    if static['binding']!='UNLINKED':
        return static
    valid=[]
    lifted=_v533_lifted_candidates(pair,registry)
    for item in lifted:
        rec=item['certificate']; end=item['endpoint']; h=item['h_lifted']
        if rec['reader_labels']!=reader_ownership:
            continue
        if field_condition=='vacuum' and end['field']:
            continue
        if target_where_A is not None and end['pA']!=target_where_A:
            continue
        if static.get('required_endpoint') is not None:
            if any(end.get(k)!=v for k,v in static['required_endpoint'].items()):
                continue
        if static.get('spatial_route') is not None and h!=static['spatial_route']['h']:
            continue
        if (placement_class=='same_carrier' and
            static.get('carrier_closed_under_native_step') and
            h not in static['carrier_words']):
            continue
        valid.append(item)
    common={'source':before,'ledger_head':head,
            'ledger_head_unchanged':pair.last_hash==head,
            'live_machine_unchanged':pair.snapshot()==before,
            'native_steps_executed':0,
            'proof_provider':'Klein_symmetry_lifted_verified_certificate',
            'registry_liftable_sources':len(lifted),
            'certificate_candidates':len(valid),
            'clock_derivation':'inherited from prior native-reverified witness',
            'global_route_uniqueness_claimed':False}
    if not valid:
        return {**static,**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':V57_OPEN}
    if len(valid)>1:
        return {**static,**common,'status':'OPEN','binding':'AMBIGUOUS',
                'providers':[{'certificate_id':x['certificate']['id'],
                              'ticks':x['certificate']['ticks'],
                              'h':x['h_lifted'],
                              'reader_labels':x['certificate']['reader_labels'],
                              'endpoint':x['endpoint'],
                              'source_symmetry_owners':x['symmetry_owners']} for x in valid],
                'missing_link':'multiple distinct registered route witnesses'}
    c=valid[0]['certificate']; choice=valid[0]; h=choice['h_lifted']
    return {**static,**common,'status':'RUNS','binding':'BOUND',
            'spatial_binding':'BOUND','clock_binding':'BOUND_BY_SYMMETRY_LIFTED_CERTIFICATE',
            'return_depth':c['ticks'],'endpoint':choice['endpoint'],
            'certificate_id':c['id'], 'certificate_origin':c['origin'],
            'source_symmetry':choice['symmetry'],
            'source_symmetry_owners':choice['symmetry_owners'],
            'route':{'operation':'native_pair_steps','ticks':c['ticks'],
                     'generated_displacement':h,'reader_labels':c['reader_labels']},
            'way_home_alternatives':[
              {'operation':'transport' if c['reader_labels']=='same_labels'
               else 'swap_then_transport','h':h},
              {'operation':'inverse_pair_steps','ticks':c['ticks']}],
            'return_by_twice_depth':2*c['ticks'],
            'twice_depth_minimality_claimed':False,'missing_link':None}


def v57_initial_stabilizers(source):
    """Exact reader-exchanging point stabilizers of canonical vacuum seeds."""
    n=source['n']
    if (source['field'] or source['pA']!=0 or
        source['wA']!=v56_carrier_word('eR',n) or
        source['pB'] not in v57_K(n) or source['wB'] not in v57_K(n)):
        return []
    result=[]
    for phi in V57_PHI:
        if (source['wB']==v57_phi_word(source['wA'],n,phi)
                and v57_phi_word(source['pB'],n,phi)==source['pB']):
            h=source['pB']
            assert _v533_apply_symmetry(source,h,True,phi)==source
            result.append({'operation':'H_h Phi_phi J_swap',
                           'h':h,'phi':phi,'swap':True,
                           'fixed_under_all_future_pair_steps':True})
    return result


def v57_group_apply(source,h=0,phi='I',swap=False):
    return _v533_apply_symmetry(source,h,swap,phi)


def v57_macro_power(source, ticks, h, phi='I', swap=False, repetitions=1):
    """Symbolic powers of G=H_h Phi_phi J^swap. G^4=I on its scope.

    Composition is H_h Phi_phi H_h Phi_phi = H_{h XOR phi(h)}.
    Derives endpoint and way home without running native F during binding.
    """
    if type(repetitions) is not int or repetitions<0 or type(ticks) is not int or ticks<=0:
        raise ValueError('valid nonnegative repetition count and supplied witness depth required')
    n=source['n']
    if h not in v57_K(n):
        raise ValueError('macro spatial displacement must stay in the native K carrier')
    if phi!='I' and v57_qualify(source) is None:
        raise ValueError('carrier field/carry prerequisite unlinked')
    ph=v57_phi_word(h,n,phi)
    if repetitions%2==0:
        displacement=((repetitions//2)&1)*(h^ph)
        apply_phi=False; apply_swap=False
    else:
        displacement=(((repetitions-1)//2)&1)*(h^ph)^h
        apply_phi=(phi!='I'); apply_swap=bool(swap)
    endpoint=v57_group_apply(source,displacement,phi if apply_phi else 'I',apply_swap)
    order=(2 if ph==h else 4) if (h or phi!='I' or swap) else 1
    return {'status':'RUNS','binding':'BOUND','source':source,'endpoint':endpoint,
            'macro':{'phi':phi,'h':h,'swap':bool(swap),'order_divides':order},
            'witness_depth':ticks,'composition_count':repetitions,
            'composed_depth':ticks*repetitions,
            'composed_action':{'phi':phi if apply_phi else 'I',
                               'h':displacement,'swap':apply_swap},
            'exact_state_return':endpoint==source,
            'way_home':{
                'spatial':{'operation':'inverse_group_action',
                           'apply_in_order':[
                             {'operation':'transport','h':displacement},
                             {'operation':'reader_swap','enabled':apply_swap},
                             {'operation':'carrier_phi','phi':phi if apply_phi else 'I'}]},
                'historical':{'operation':'inverse_pair_steps','ticks':ticks*repetitions},
                'semantics':'ALTERNATIVES, never successive methods'},
            'native_steps_executed_during_composition':0,
            'minimum_period_claimed':False,
            'clock_derived_from_initial_boundary':False}


def v57_verify_macro_witness(where_B, carry_B, ticks, *, origin):
    """Recheck a linked depth and ALL eligible symmetry owners, no guessing."""
    if type(ticks) is not int or not (1<=ticks<=100_000):
        raise ValueError('supplied native witness depth 1..100000 required')
    pair=v56_seed(where_B,carry_B)
    source=pair.snapshot()
    current=pair.copy()
    prior_vacuum=False
    for i in range(1,ticks+1):
        current.forward()
        if i<ticks and not current.snapshot()['field']:
            prior_vacuum=True
    endpoint=current.snapshot()
    if endpoint['field']:
        raise ValueError('supplied depth did not land at vacuum')
    inverse=current.copy()
    for _ in range(ticks):
        inverse.backward()
    if inverse.snapshot()!=source:
        raise AssertionError('failed exact joint inverse')
    providers=[]
    for phi in V57_PHI:
        for swap in (False,True):
            at_zero=_v533_apply_symmetry(source,0,swap,phi)
            h=at_zero['pA'] ^ endpoint['pA']
            if h not in v57_K(source['n']):
                continue
            if _v533_apply_symmetry(source,h,swap,phi)==endpoint:
                # Verify algebraic one-step inverse, not just historical reversal.
                home=_v533_apply_symmetry(endpoint,h,swap,'I')
                home=v57_phi_snapshot(home,phi)
                if home!=source:
                    raise AssertionError('generalized symmetry way home is false')
                providers.append({'phi':phi,'h':h,'swap':swap,
                                  'macro_order_divides':v57_macro_power(source,ticks,h,phi,swap,4)['macro']['order_divides']})
    if not providers:
        raise ValueError('supplied first-vacuum readout has no generalized symmetry owner')
    body={'schema':'nexus-v57-generalized-macro-1','source':source,'endpoint':endpoint,
          'source_sha256':_v532_digest(source), 'ticks':ticks,
          'providers':providers,'first_vacuum_verified':not prior_vacuum,
          'exact_forward_backward_verified':True,'origin':origin}
    return {'id':_v532_digest(body),**body}


# SUPPLIED first-vacuum depths from the user's carrier run, not computed clocks.
# The final seed is deliberately order FOUR (a nontrivial check of macro algebra).
V57_LINKED_FIRST_VACUUM=(
    ('s','eR',144),        # Phi_A0
    ('u+s','eR',132),      # Phi_A
    ('s','u+eR',4875),    # Phi_J
    ('u','eL',2478),      # Phi_A
    ('0','eR',1),         # G=H_eR Phi_J, order 4; full period 4
)
V57_SYMMETRY_CERTS={}
for _v57_b,_v57_w,_v57_depth in V57_LINKED_FIRST_VACUUM:
    _v57_cert=v57_verify_macro_witness(
        _v57_b,_v57_w,_v57_depth,
        origin='user-supplied 2026-11-03 first-vacuum carrier trace; native reverified')
    V57_SYMMETRY_CERTS[(_v57_b,_v57_w)]=_v57_cert


def v57_lift_generalized(source, registry=None):
    """DI-bind generalized witnesses to exactly symmetry-equivalent sources."""
    if registry is None:
        registry=V57_SYMMETRY_CERTS
    snap=source.snapshot() if isinstance(source,PairMachine) else source
    results={}
    for cert in registry.values():
        for sym in _v533_source_provider(cert,snap):
            psi=sym['source_phi']
            mapped_end=_v533_apply_symmetry(cert['endpoint'],
                          sym['source_translation'],sym['source_reader_swap'],psi)
            for provider in cert['providers']:
                hh=(v57_phi_address(provider['h'],snap['n'],psi) ^
                    sym['source_translation'] ^
                    v57_phi_address(sym['source_translation'],snap['n'],provider['phi']))
                if _v533_apply_symmetry(snap,hh,provider['swap'],provider['phi'])!=mapped_end:
                    raise AssertionError('generalized conjugation check failed')
                key=(cert['id'],canonical(mapped_end),hh,provider['phi'],provider['swap'])
                entry=results.get(key)
                if entry:
                    entry['source_owners'].append(sym)
                else:
                    results[key]={
                        'certificate_id':cert['id'],'source':snap,'endpoint':mapped_end,
                        'ticks':cert['ticks'],'first_vacuum_verified':cert['first_vacuum_verified'],
                        'source_owners':[sym],
                        'macro':{'phi':provider['phi'],'h':hh,'swap':provider['swap']},
                        'way_home':{'spatial':{'operation':'inverse_generalized_symmetry',
                            'h':hh,'phi':provider['phi'],'swap':provider['swap'],
                            'apply_in_order':['H_h','J_swap_if_needed','Phi_phi']},
                            'historical':{'operation':'inverse_pair_steps','ticks':cert['ticks']},
                            'semantics':'ALTERNATIVES, not sequential steps'}}
    return list(results.values())


def v57_carrier_goal(where_B,carry_B,*,target_where_A=None,
                      placement_class='same_carrier',return_kind='symmetry_vacuum',
                      reader_ownership='any'):
    """Canonical 64-seed static goal. New mode permits Phi-transformed carries."""
    if return_kind not in ('symmetry_vacuum','translated_vacuum','exact_state'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'unsupported return_kind'}
    if return_kind!='symmetry_vacuum':
        # Old contract remains available for pure H/J returns.
        return _V57_PREV_CARRIER_GOAL(where_B,carry_B,
            target_where_A=target_where_A,placement_class=placement_class,
            return_kind=return_kind,reader_ownership=reader_ownership)
    base=V56_CARRIER_INDEX.get((where_B,carry_B))
    if base is None:
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'canonical seed missing'}
    source=base['source']; n=source['n']
    if placement_class=='outside_carrier':
        return {'status':'RUNS','binding':'UNBOUND','source':source,
                'reason':'closed isolated K carrier invariant; phi does not cause deconfinement',
                'native_steps_executed':0}
    if placement_class!='same_carrier':
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'unsupported placement class'}
    if reader_ownership not in ('any','same_labels','swapped_labels'):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'unsupported reader ownership'}
    if target_where_A is not None and (type(target_where_A) is not int or
        not (0<=target_where_A<(1<<n))):
        return {'status':'OPEN','binding':'UNLINKED','missing_link':'invalid WHERE'}
    if target_where_A is not None and target_where_A not in v57_K(n):
        return {'status':'RUNS','binding':'UNBOUND','source':source,
                'reason':'requested endpoint WHERE outside invariant K carrier'}
    options=[]
    for x in v57_lift_generalized(source):
        if target_where_A is not None and x['endpoint']['pA']!=target_where_A:
            continue
        if reader_ownership!='any' and (
            ('swapped_labels' if x['macro']['swap'] else 'same_labels')!=reader_ownership):
            continue
        options.append(x)
    # Group distinct symmetry descriptions that share one native history.
    hist={}
    for x in options:
        key=(x['certificate_id'],x['ticks'],canonical(x['endpoint']))
        hist.setdefault(key,[]).append(x)
    common={'source':source,'source_sha256':base['source_sha256'],
            'native_steps_executed':0,'ledger_head_unchanged':True,
            'live_machine_unchanged':True,
            'clock_owner':'previously supplied and native-reverified first-vacuum witness',
            'clock_derived_from_initial_boundary':False,
            'stabilizers':v57_initial_stabilizers(source),
            'symmetry_group':'H_h ⋊ (Klein_four × J_swap) scoped to K-valued cosets'}
    if not hist:
        return {**common,'status':'OPEN','binding':'UNLINKED',
                'missing_link':'No matching generalized vacuum certificate; '+V57_OPEN}
    if len(hist)>1:
        return {**common,'status':'OPEN','binding':'AMBIGUOUS',
                'history_provider_count':len(hist),
                'histories':[{'ticks':items[0]['ticks'],
                              'endpoint':items[0]['endpoint'],
                              'certificate_id':items[0]['certificate_id']}
                             for items in hist.values()],
                'missing_link':'multiple distinct execution histories satisfy goal'}
    owners=next(iter(hist.values()))
    sample=owners[0]
    macro=sample['macro']
    power=v57_macro_power(source,sample['ticks'],macro['h'],
                           macro['phi'],macro['swap'],2)
    return {**common,'status':'RUNS','binding':'BOUND',
            'ticks':sample['ticks'],'endpoint':sample['endpoint'],
            'history_provider_count':1,
            'symmetry_owners':[x['macro'] for x in owners],
            'certificate_id':sample['certificate_id'],
            'macro_order_divides':power['macro']['order_divides'],
            'twice_depth_endpoint':power['endpoint'],
            'way_home_alternatives':sample['way_home'],
            'unknown_clock_law':'OPEN', 'minimum_period_claimed':False}


_V57_PREV_CARRIER_GOAL=v56_carrier_goal
v56_carrier_goal=v57_carrier_goal

# Existing v5.6 tool references v56_carrier_goal dynamically; extend its schema.
for _v57_schema in TOOLS:
    if _v57_schema.get('function',{}).get('name')=='nexus_pair_carrier_goal':
        _v57_enum=_v57_schema['function']['parameters']['properties']['return_kind']['enum']
        if 'symmetry_vacuum' not in _v57_enum:
            _v57_enum.append('symmetry_vacuum')
        _v57_schema['function']['description']+=(
            ' return_kind=symmetry_vacuum accepts a Phi-transformed vacuum endpoint '
            'and returns all valid symmetry owners. Depths come only from reverified supplied readouts.')
        break
SYSTEM += (' v5.7: carrier K supports four scoped commuting Phi maps I,J,A,A0. '
    'Use nexus_pair_carrier_goal with return_kind=symmetry_vacuum for a generalized '
    'half-return; do not call it a pure transport unless phi=I. '
    'For 0/eR the first-vacuum macro has order four. Unknown clock remains OPEN.')

_V57_PREV_CAPABILITIES=nexus_pair_capabilities

def nexus_pair_capabilities():
    data=_V57_PREV_CAPABILITIES()
    data['version']=V57_VERSION
    data['carrier_klein_symmetry']={
        'phi_names':list(V57_PHI),
        'scope':'native K-valued states confined to one carrier coset',
        'transformation_bound_at_n4':4*2*(1<<4),
        'note':'128 transformations, not necessarily 128 distinct images of a seed',
        'canonical_seed_stabilizer_count':sum(bool(v57_initial_stabilizers(x['source']))
            for x in V56_CARRIER_INDEX.values()),
        'native_reverified_generalized_vacuum_witnesses':len(V57_SYMMETRY_CERTS),
        'unknown_clock':'OPEN',
        'outside_carrier_release':'UNBOUND for isolated K carrier',
    }
    data['default_pair_initial_stabilizers']=v57_initial_stabilizers(pair_machine.snapshot())
    data['capability_ledger']['carrier_Klein_four_commutes_with_pair_step']='RUNS'
    data['capability_ledger']['generalized_first_vacuum_witnesses']='RUNS_on_supplied_depths'
    data['capability_ledger']['unknown_clock_from_initial_boundary']='OPEN'
    return data


def verify_v57():
    rng=_v57_random.Random(570257)
    count=0
    # Native P_n on K; automorphism group and involutions.
    for n in (4,6,8,10,12,14,16):
        K=v57_K(n)
        assert len(K)==8
        for phi in V57_PHI:
            assert {v57_phi_word(x,n,phi) for x in K}==K
            for k in K:
                assert partner(v57_phi_word(k,n,phi),n)==v57_phi_word(partner(k,n),n,phi)
                assert v57_phi_word(v57_phi_word(k,n,phi),n,phi)==k
                count+=2
    # Commutation on genuine populated K fields, including translated cosets.
    for n in (4,6,8,16):
        K=list(v57_K(n))
        for _ in range(120):
            g=rng.randrange(1<<n)
            pA,pB=(g^rng.choice(K),rng.randrange(1<<n))
            wA,wB=rng.choice(K),rng.choice(K)
            field={rng.randrange(1<<n):rng.choice(K) for __ in range(rng.randrange(8))}
            field={k:v for k,v in field.items() if v}
            S=PairMachine(n,pA,wA,pB,wB,field)
            before=S.snapshot()
            for phi in V57_PHI:
                a=_v532_from_snapshot(v57_phi_snapshot(before,phi))
                a.forward()
                b=S.copy(); b.forward()
                after=v57_phi_snapshot(b.snapshot(),phi)
                assert a.snapshot()==after
                assert v57_phi_snapshot(v57_phi_snapshot(before,phi),phi)==before
                count+=2
    # Verify the inverse of the COMPOSED action at every k, not the base macro.
    for b,w,t in V57_LINKED_FIRST_VACUUM:
        cert=V57_SYMMETRY_CERTS[(b,w)]
        for owner in cert['providers']:
            for k in range(5):
                power=v57_macro_power(cert['source'],t,owner['h'],
                     owner['phi'],owner['swap'],k)
                inv=power['way_home']['spatial']['apply_in_order']
                back=_v532_from_snapshot(power['endpoint'])
                back.transport(inv[0]['h'])
                if inv[1]['enabled']:back.swap()
                assert v57_phi_snapshot(back.snapshot(),inv[2]['phi'])==cert['source']
                count+=1
    # Point stabilizers computed directly from initial source (no orbit reading).
    owners=[(b,w) for b in V56_LABELS for w in V56_LABELS
            if v57_initial_stabilizers(v56_seed(b,w).snapshot())]
    assert len(owners)==20,owners
    for (b,w),c in V57_SYMMETRY_CERTS.items():
        assert c['first_vacuum_verified'] and c['exact_forward_backward_verified']
        assert c['ticks']==dict(((x,y),t) for x,y,t in V57_LINKED_FIRST_VACUUM)[(b,w)]
        x=v57_carrier_goal(b,w,return_kind='symmetry_vacuum')
        assert x['binding']=='BOUND' and x['ticks']==c['ticks'],x
        count+=3
    # Four Phi-only half returns are twofold, the identical pair's is fourfold.
    for b,w,t in V57_LINKED_FIRST_VACUUM[:4]:
        c=V57_SYMMETRY_CERTS[(b,w)]
        own=next(x for x in c['providers'] if x['phi']!='I' and not x['swap'] and x['h']==0)
        power=v57_macro_power(c['source'],t,own['h'],own['phi'],own['swap'],2)
        assert power['exact_state_return']
        native=_v532_from_snapshot(c['source'])
        for _ in range(2*t): native.forward()
        assert native.snapshot()==c['source']
        count+=2
    c=V57_SYMMETRY_CERTS[('0','eR')]
    p=next(x for x in c['providers'] if x['phi']=='J' and x['h']==v56_carrier_word('eR') and not x['swap'])
    assert not v57_macro_power(c['source'],1,p['h'],p['phi'],p['swap'],2)['exact_state_return']
    assert v57_macro_power(c['source'],1,p['h'],p['phi'],p['swap'],4)['exact_state_return']
    actual=_v532_from_snapshot(c['source'])
    for _ in range(4):actual.forward()
    assert actual.snapshot()==c['source']
    count+=3
    # Two distinct widths and several symmetries of the existing 806 certificate.
    original=v56_seed('u','u+eL')
    candidates=_v533_lifted_candidates(original,V56_REGISTRY)
    assert any(c['certificate']['ticks']==806 for c in candidates)
    assert v55_compile_exact_return(original,V56_REGISTRY)['binding']=='BOUND'
    for n in (4,6,8):
        base=v56_seed('u','u+eL',width=n)
        for phi in V57_PHI:
            for shift in (0, ((1<<n)-1)^v56_carrier_word('eR',n)):
                source=v57_group_apply(base.snapshot(),shift,phi,False)
                a=_v532_from_snapshot(source)
                for _ in range(806):a.forward()
                expected=v57_group_apply(base.snapshot(),shift,phi,False)
                h=v57_phi_word(v56_carrier_word('u+s',n),n,phi)
                expected=_v533_apply_symmetry(expected,h,False)
                assert a.snapshot()==expected
                count+=1
    # Exact conjugation of generalized Phi witness under translations out of K.
    for pair_key,cert in V57_SYMMETRY_CERTS.items():
        for shift in (0, 1, 8, 13):
            moved=_v533_apply_symmetry(cert['source'],shift,False)
            lifted=v57_lift_generalized(moved)
            owned=[x for x in lifted if x['certificate_id']==cert['id']]
            assert owned, (pair_key,shift)
            native=_v532_from_snapshot(moved)
            for _ in range(cert['ticks']):native.forward()
            assert all(x['endpoint']==native.snapshot() for x in owned)
            count+=1
    for n in (4,6,8,16):
        for _ in range(48):
            a,b=rng.randrange(1<<n),rng.randrange(1<<n)
            for phi in V57_PHI:
                assert v57_phi_address(a^b,n,phi)==(v57_phi_address(a,n,phi)^v57_phi_address(b,n,phi))
                count+=1
    # No K-lifting when the value constraints are absent.
    invalid=PairMachine(4,0,1,15,2,{0:3}).snapshot()
    assert v57_qualify(invalid) is None
    try:v57_phi_snapshot(invalid,'A')
    except ValueError:pass
    else:raise AssertionError('invalid unrestricted field admitted')
    # Static goal: no mutation or new depth inference.
    before=pair_machine.snapshot();head=pair_machine.last_hash
    z=pair_model_dispatch('nexus_pair_carrier_goal',{'where_B':'s','carry_B':'eR',
            'return_kind':'symmetry_vacuum'})
    assert z['binding']=='BOUND' and z['ticks']==144,z
    assert pair_machine.snapshot()==before and pair_machine.last_hash==head
    print('RUNS: v5.7',count,'native carrier symmetry / commutation / inverse assertions.')
    print('RUNS: 20/64 initial reader-exchange stabilizers from algebra.')
    print('RUNS: five supplied first-vacuum depths native reverified, including order-four macro.')
    print('OPEN: vacuum depth from initial boundary; no cross-carrier release under isolated K rule.')
    return {'assertions':count,'stabilized_seeds':20,
            'new_generalized_certificates':len(V57_SYMMETRY_CERTS)}

V57_REPORT=verify_v57()
print('NEXUS AI v5.7 KLEIN-FOUR EXTENSION READY; LM Studio not contacted.')



# NEXUS v5.7.1: compiler-gated Klein-four goal tool.
# The v5.7 session demonstrated a schema gap: Gemma supplied placement_class="any"
# to a carrier-only goal. This new schema has *exactly* two inputs. Their
# provenance is explicit; the tool never infers a placement class from model text.
_V571_PREVIOUS_PAIR_DISPATCH = pair_model_dispatch

def pair_model_dispatch(name, args):
    if name != 'nexus_pair_carrier_symmetry_goal':
        return _V571_PREVIOUS_PAIR_DISPATCH(name, args)
    required = {'where_B', 'carry_B'}
    if type(args) is not dict or set(args) != required:
        received = set(args) if type(args) is dict else set()
        return {
            'status': 'OPEN', 'binding': 'UNLINKED',
            'missing_fields': sorted(required - received),
            'unexpected_fields': sorted(received - required),
            'missing_link': 'carrier symmetry goal accepts exactly where_B and carry_B; no implicit target WHERE, placement class, depth, or offset'
        }
    if any(type(args[k]) is not str or args[k] not in V56_LABELS for k in required):
        return {'status': 'OPEN', 'binding': 'UNLINKED',
                'missing_link': 'where_B and carry_B must be native named carrier words'}
    # Carrier class and generalized vacuum closure are fixed by this tool's
    # declared *type*. They are NOT missing parameters being populated by default.
    return v57_carrier_goal(args['where_B'], args['carry_B'],
        placement_class='same_carrier', return_kind='symmetry_vacuum',
        reader_ownership='any', target_where_A=None)

_V571_SCHEMA = {'type':'function', 'function':{
    'name':'nexus_pair_carrier_symmetry_goal',
    'description':(
        'READ-ONLY compiler-side generalized Klein-four vacuum return for a '
        'canonical seed. Exactly two inputs: the second reader WHERE and carry. '
        'The first reader is fixed at (0,eR); initial field vacuum; the carrier '
        'is invariant. Find a previously native-reverified vacuum-return witness '
        'and its I/J/A/A0 owners. Do not supply placement_class, target_where_A, '
        'ticks, h, target_delta, reader_ownership, or return_kind. Unavailable '
        'witnesses remain UNLINKED. This tool does NOT derive new clock laws.'),
    'parameters':{'type':'object','properties':{
        'where_B':{'type':'string','enum':list(V56_LABELS),
                   'description':'Second reader starting WHERE, native carrier label'},
        'carry_B':{'type':'string','enum':list(V56_LABELS),
                   'description':'Second reader starting carry, native carrier label'}},
        'required':['where_B','carry_B'], 'additionalProperties':False}}}
for _v571_catalog in (TOOLS, PAIR_TOOLS):
    if not any(x.get('function',{}).get('name')=='nexus_pair_carrier_symmetry_goal'
               for x in _v571_catalog):
        _v571_catalog.append(_V571_SCHEMA)
SYSTEM += (' For a canonical generalized Klein-four first-vacuum query, prefer '
    'nexus_pair_carrier_symmetry_goal and supply exactly where_B and carry_B. '
    'Do not invent placement_class, ticks or target WHERE. If no verified '
    'certificate exists, report UNLINKED; the unknown clock law is OPEN.')

_V571_PREVIOUS_CAPABILITIES = nexus_pair_capabilities

def nexus_pair_capabilities():
    data = _V571_PREVIOUS_CAPABILITIES()
    data['version'] = '5.7.1'
    data['carrier_klein_symmetry']['preferred_tool'] = 'nexus_pair_carrier_symmetry_goal'
    data['carrier_klein_symmetry']['tool_arguments'] = ['where_B', 'carry_B']
    data['carrier_klein_symmetry']['prior_model_call_failure'] = (
        'v5.7 Gemma sent placement_class=any to a carrier-only tool; '
        'v5.7.1 uses a two-argument goal contract')
    return data

def verify_v571_tool_contract():
    seed_before = pair_machine.snapshot()
    pair_head_before = pair_machine.last_hash
    single_before = machine.snapshot()
    single_head_before = machine.last_hash
    schema = next(t for t in TOOLS
        if t['function']['name']=='nexus_pair_carrier_symmetry_goal')['function']['parameters']
    assert set(schema['properties']) == {'where_B','carry_B'}
    assert set(schema['required']) == {'where_B','carry_B'}
    assert schema['additionalProperties'] is False
    # All calls go through the exact same dispatch used by LM Studio.
    c = dispatch_model_tool(machine,'nexus_pair_carrier_symmetry_goal',
                            {'where_B':'s','carry_B':'eR'})
    assert c['binding']=='BOUND' and c['ticks']==144 and c['native_steps_executed']==0,c
    assert any(x['phi']=='A0' for x in c['symmetry_owners']),c
    id_pair = dispatch_model_tool(machine,'nexus_pair_carrier_symmetry_goal',
                                 {'where_B':'0','carry_B':'eR'})
    assert id_pair['binding']=='BOUND' and id_pair['ticks']==1 and id_pair['macro_order_divides']==4,id_pair
    x = dispatch_model_tool(machine,'nexus_pair_carrier_symmetry_goal',
                            {'where_B':'eR','carry_B':'eL'})
    assert x['binding']=='UNLINKED',x
    wrong = dispatch_model_tool(machine,'nexus_pair_carrier_symmetry_goal',
            {'where_B':'s','carry_B':'eR','placement_class':'any',
             'target_where_A':0,'return_kind':'symmetry_vacuum'})
    assert wrong['binding']=='UNLINKED' and 'placement_class' in wrong['unexpected_fields'],wrong
    missing = dispatch_model_tool(machine,'nexus_pair_carrier_symmetry_goal',
                                  {'where_B':'s'})
    assert missing['binding']=='UNLINKED' and missing['missing_fields']==['carry_B'],missing
    assert pair_machine.snapshot()==seed_before and pair_machine.last_hash==pair_head_before
    assert machine.snapshot()==single_before and machine.last_hash==single_head_before
    assert 'nexus_pair_carrier_symmetry_goal' in nexus_pair_capabilities()['available_tools']
    print('RUNS: v5.7.1 schema-locked carrier symmetry goal; BOUND 144, order-4 BOUND 1, absent UNLINKED, malformed UNLINKED, machines/ledgers unchanged.')
    return {'status':'RUNS','goal_example':c,'order_four_example':id_pair,
            'negative_contract_test':wrong}

V571_REPORT = verify_v571_tool_contract()
print('NEXUS AI v5.7.1 READY — LM Studio not contacted.')


RUNS: 30,844 deterministic conditions; P, arbitrary-field roundtrip, transport involution/group law, F H = H F, inverse commutation, and 1000-step return at n=4,8,16.
RUNS: 10,803 BIND checks (widths 2..16, populated fields, exact inverse, ambiguous/unlinked cases).
RUNS: 25,204 pair compiler checks across widths 2..16; two independent source-C supplied-depth certificates.
Replayed 19 event(s), including 4 legacy event(s). Last chain digest: d3c28760a59dc2aa6f4de437de31d217e0e9e976af823b3c183a047cb1513077

NEXUS PAIR v5 READY — two isolated compiler states
Single reader: {"field":{"4":1,"5":8},"n":4,"p":12,"w":1}
Paired readers: {"field":{},"n":4,"pA":0,"pB":15,"wA":2,"wB":11}
NEXUS pair ledger: D:\@User Data\Downloads\NEXUS_local_AI_pair_events.jsonl
LM Studio: http://localhost:1235
To verify the supplied 806-step pair return, use: pair_bind_return(pair_machine,806)
NEXUS v5.2 switch-boundary compiler installed; pre-readout seal: 261d35de94944a60bf1fbffd6019513734f567c50f54945cb60cca9

## Read-only goal-contract demonstrations
Runs against the exact tool dispatcher exposed to Gemma.


In [2]:
from pprint import pprint
print('Klein-four Phi_A0 goal:')
pprint(dispatch_model_tool(machine,'nexus_pair_carrier_symmetry_goal',{'where_B':'s','carry_B':'eR'}))
print('\nOrder-four identity-pair goal:')
pprint(dispatch_model_tool(machine,'nexus_pair_carrier_symmetry_goal',{'where_B':'0','carry_B':'eR'}))
print('\nContract status:')
pprint({'tool':'nexus_pair_carrier_symmetry_goal','inputs':['where_B','carry_B'], 'clock':'OPEN for unknown seeds'})


Klein-four Phi_A0 goal:
{'binding': 'BOUND',
 'certificate_id': 'e43a0d426cbb29800e762affcd083c4278674653dd3cc51439c559b4da9e1e0a',
 'clock_derived_from_initial_boundary': False,
 'clock_owner': 'previously supplied and native-reverified first-vacuum '
                'witness',
 'endpoint': {'field': {}, 'n': 4, 'pA': 0, 'pB': 6, 'wA': 13, 'wB': 13},
 'history_provider_count': 1,
 'ledger_head_unchanged': True,
 'live_machine_unchanged': True,
 'macro_order_divides': 2,
 'minimum_period_claimed': False,
 'native_steps_executed': 0,
 'source': {'field': {}, 'n': 4, 'pA': 0, 'pB': 6, 'wA': 2, 'wB': 2},
 'source_sha256': '7955c0365031104db88abb3833bdfd8b369479b2ec6f5b540be03a0d505215fe',
 'stabilizers': [{'fixed_under_all_future_pair_steps': True,
                  'h': 6,
                  'operation': 'H_h Phi_phi J_swap',
                  'phi': 'I',
                  'swap': True}],
 'status': 'RUNS',
 'symmetry_group': 'H_h ⋊ (Klein_four × J_swap) scoped to K-valued cosets',
 'symm

## Optional live Gemma test
Requires LM Studio with Gemma loaded at the configured endpoint. This cell is not run automatically.


In [3]:
model_id = find_model()
if model_id:
    ask_model(model_id, machine,
        'Use nexus_pair_capabilities to check the new goal interface. '
        'Then call nexus_pair_carrier_symmetry_goal with exactly '
        'where_B=s and carry_B=eR. Report the Phi owner, exact inverse, '
        'and the still OPEN unknown-clock dependency. Do not invent additional inputs.')


  1. Gemma 4 E4B [google/gemma-4-e4b]
Using first loaded model: Gemma 4 E4B
  CALL nexus_pair_capabilities raw arguments: {}
  TOOL nexus_pair_capabilities: {"available_tools":["nexus_bind","nexus_bind_endpoint","nexus_bind_predecessor","nexus_bind_predecessor_of","nexus_commute","nexus_pair_bind_first_vacuum","nexus_pair_bind_goal_by_symmetry","nexus_pair_bind_goal_from_certificate","nexus_pair_bind_return","nexus_pair_capabilities","nexus_pair_carrier_catalog","nexus_pair_carrier_goal","nexus_pair_carrier_symmetry_goal","nexus_pair_compile_boundary_goal","nexus_pair_goal","nexus_pair_reverse","nexus_pair_state","nexus_pair_step","nexus_pair_switch_holonomy","nexus_reverse","nexus_state","nexus_step","nexus_transport"],"capability_ledger":{"64_canonical_seed_slots_indexed":"RUNS","64_native_verified_return_clocks":"OPEN","carrier_Klein_four_commutes_with_pair_step":"RUNS","certified_route_group_composition":"RUNS","first_vacuum_event_execution":"RUNS","generalized_first_vacuum_witness